# Reinforcement Learning in Stock Trading
### Advanced Multi-Stock DQN Trading Agent (with Technical Indicators)

This notebook is **fully self-contained** — real historical stock data is embedded
directly inside it, so it runs top-to-bottom with no external files needed.

**What this project does:**
- Trains a **Double DQN** Reinforcement Learning agent to trade stocks (Buy / Sell / Hold)
- Uses **4 real stocks**: AAPL (Apple), MSFT (Microsoft), IBM, SBUX (Starbucks)
- Uses **technical indicators** as features (moving averages, RSI, volatility, momentum) — not just raw price
- Compares the trained agent against **Buy & Hold** and **Random Trading** baselines
- Produces graphs and a results table for your report

**How to run:** Click `Runtime > Run all` (Colab) or `Kernel > Restart & Run All` (Jupyter).
Training all 4 stocks takes roughly 10–15 minutes on CPU.


## 1. Setup: Import Libraries

In [ ]:
import random
import io
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
except ImportError as e:
    raise ImportError(
        "PyTorch is required. Install it with `pip install torch` "
        "and then restart the notebook kernel."
    ) from e

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Libraries loaded. Using device:", DEVICE)


## 2. Data: Real Historical Stock Prices (Embedded)

The data below is **real daily closing price history** for 4 stocks from Jan 2007 to Feb 2016
(2,305 trading days each). It's embedded directly as text so no internet/file access is needed.


In [ ]:
AAPL_CSV = """Date,Close
2007-01-03,11.086612
2007-01-04,11.332687
2007-01-05,11.251984
2007-01-08,11.30755
2007-01-09,12.24687
2007-01-10,12.832951
2007-01-11,12.674194
2007-01-12,12.518082
2007-01-16,12.846181
2007-01-17,12.561739
2007-01-18,11.783825
2007-01-19,11.708415
2007-01-22,11.482184
2007-01-23,11.337979
2007-01-24,11.470278
2007-01-25,11.410743
2007-01-26,11.295643
2007-01-29,11.36973
2007-01-30,11.318134
2007-01-31,11.341948
2007-02-01,11.210972
2007-02-02,11.212295
2007-02-05,11.105133
2007-02-06,11.132916
2007-02-07,11.397513
2007-02-08,11.401483
2007-02-09,11.016494
2007-02-12,11.229494
2007-02-13,11.205681
2007-02-14,11.28506
2007-02-15,11.273153
2007-02-16,11.222879
2007-02-20,11.364438
2007-02-21,11.801023
2007-02-22,11.842036
2007-02-23,11.783825
2007-02-26,11.709738
2007-02-27,11.10381
2007-02-28,11.193774
2007-03-01,11.517905
2007-03-02,11.299612
2007-03-05,11.420004
2007-03-06,11.667402
2007-03-07,11.605222
2007-03-08,11.642265
2007-03-09,11.638296
2007-03-12,11.889664
2007-03-13,11.695185
2007-03-14,11.906862
2007-03-15,11.849974
2007-03-16,11.85262
2007-03-19,12.05636
2007-03-20,12.102664
2007-03-21,12.418857
2007-03-22,12.430764
2007-03-23,12.372553
2007-03-26,12.680808
2007-03-27,12.629212
2007-03-28,12.335509
2007-03-29,12.402982
2007-03-30,12.29185
2007-04-02,12.389752
2007-04-03,12.502205
2007-04-04,12.471777
2007-04-05,12.526019
2007-04-09,12.389752
2007-04-10,12.469131
2007-04-11,12.249516
2007-04-12,12.196596
2007-04-13,11.938614
2007-04-16,12.096049
2007-04-17,11.953167
2007-04-18,11.959782
2007-04-19,11.942583
2007-04-20,12.035192
2007-04-23,12.37123
2007-04-24,12.335509
2007-04-25,12.614659
2007-04-26,13.076381
2007-04-27,13.219263
2007-04-30,13.203387
2007-05-01,13.159729
2007-05-02,13.281444
2007-05-03,13.282767
2007-05-04,13.337008
2007-05-07,13.748457
2007-05-08,13.899277
2007-05-09,14.140061
2007-05-10,14.200918
2007-05-11,14.386135
2007-05-14,14.468161
2007-05-15,14.224731
2007-05-16,14.200918
2007-05-17,14.478745
2007-05-18,14.555478
2007-05-21,14.814783
2007-05-22,15.021168
2007-05-23,14.935173
2007-05-24,14.644117
2007-05-25,15.031752
2007-05-29,15.12833
2007-05-30,15.713089
2007-05-31,16.033251
2007-06-01,15.664139
2007-06-04,16.051773
2007-06-05,16.229054
2007-06-06,16.357383
2007-06-07,16.414271
2007-06-08,16.469837
2007-06-11,15.900953
2007-06-12,15.92609
2007-06-13,15.545071
2007-06-14,15.710443
2007-06-15,15.941965
2007-06-18,16.549216
2007-06-19,16.360029
2007-06-20,16.08088
2007-06-21,16.391781
2007-06-22,16.272712
2007-06-25,16.185394
2007-06-26,15.829512
2007-06-27,16.12586
2007-06-28,15.949903
2007-06-29,16.145706
2007-07-02,16.042512
2007-07-03,16.824396
2007-07-05,17.562621
2007-07-06,17.503087
2007-07-09,17.24246
2007-07-10,17.509702
2007-07-11,17.514994
2007-07-12,17.737256
2007-07-13,18.221468
2007-07-16,18.270418
2007-07-17,18.377581
2007-07-18,18.273064
2007-07-19,18.521786
2007-07-20,19.017906
2007-07-23,19.011291
2007-07-24,17.845741
2007-07-25,18.159288
2007-07-26,19.315577
2007-07-27,19.031134
2007-07-30,18.710973
2007-07-31,17.431646
2007-08-01,17.860294
2007-08-02,18.057418
2007-08-03,17.443553
2007-08-06,17.893368
2007-08-07,17.864263
2007-08-08,17.729317
2007-08-09,16.721203
2007-08-10,16.537309
2007-08-13,16.906422
2007-08-14,16.408979
2007-08-15,15.862586
2007-08-16,15.485535
2007-08-17,16.148352
2007-08-20,16.169518
2007-08-21,16.877315
2007-08-22,17.53087
2007-08-23,17.34036
2007-08-24,17.899983
2007-08-27,17.496472
2007-08-28,16.778093
2007-08-29,17.738579
2007-08-30,18.025666
2007-08-31,18.320693
2007-09-04,19.072148
2007-09-05,18.093139
2007-09-06,17.861617
2007-09-07,17.432969
2007-09-10,18.086524
2007-09-11,17.92512
2007-09-12,18.105045
2007-09-13,18.15135
2007-09-14,18.36435
2007-09-17,18.311432
2007-09-18,18.6435
2007-09-19,18.623656
2007-09-20,18.562797
2007-09-21,19.070825
2007-09-24,19.617217
2007-09-25,20.265479
2007-09-26,20.211237
2007-09-27,20.440113
2007-09-28,20.303846
2007-10-01,20.683543
2007-10-02,20.962692
2007-10-03,20.892574
2007-10-04,20.670313
2007-10-05,21.359588
2007-10-08,22.214236
2007-10-09,22.207621
2007-10-10,22.066062
2007-10-11,21.462781
2007-10-12,22.126919
2007-10-15,22.091199
2007-10-16,22.435174
2007-10-17,22.854561
2007-10-18,22.953785
2007-10-19,22.546305
2007-10-22,23.067561
2007-10-23,24.628683
2007-10-24,24.598254
2007-10-25,24.181515
2007-10-26,24.435527
2007-10-29,24.487124
2007-10-30,24.739813
2007-10-31,25.130094
2007-11-01,24.798026
2007-11-02,24.854914
2007-11-05,24.631329
2007-11-06,25.373523
2007-11-07,24.647206
2007-11-08,23.214412
2007-11-09,21.878199
2007-11-12,20.342212
2007-11-13,22.485449
2007-11-14,21.976098
2007-11-15,21.736638
2007-11-16,22.013143
2007-11-19,21.690334
2007-11-20,22.338597
2007-11-21,22.287
2007-11-23,22.69448
2007-11-26,22.826778
2007-11-27,23.127095
2007-11-28,23.84283
2007-11-29,24.381285
2007-11-30,24.107428
2007-12-03,23.662904
2007-12-04,23.788588
2007-12-05,24.541366
2007-12-06,25.130094
2007-12-07,25.705593
2007-12-10,25.693686
2007-12-11,24.943554
2007-12-12,25.250486
2007-12-13,25.378815
2007-12-14,25.188306
2007-12-17,24.395838
2007-12-18,24.207973
2007-12-19,24.226496
2007-12-20,24.767596
2007-12-21,25.653997
2007-12-24,26.300935
2007-12-26,26.320781
2007-12-27,26.270508
2007-12-28,26.437202
2007-12-31,26.20568
2008-01-02,25.777034
2008-01-03,25.788941
2008-01-04,23.820339
2008-01-07,23.501499
2008-01-08,22.656112
2008-01-09,23.734345
2008-01-10,23.551774
2008-01-11,22.846623
2008-01-14,23.652321
2008-01-15,22.363733
2008-01-16,21.120127
2008-01-17,21.2855
2008-01-18,21.34768
2008-01-22,20.590933
2008-01-23,18.398749
2008-01-24,17.939673
2008-01-25,17.200124
2008-01-28,17.200124
2008-01-29,17.402542
2008-01-30,17.487211
2008-01-31,17.907921
2008-02-01,17.694921
2008-02-04,17.417093
2008-02-05,17.11413
2008-02-06,16.140414
2008-02-07,16.039866
2008-02-08,16.600812
2008-02-11,17.126037
2008-02-12,16.518787
2008-02-13,17.119422
2008-02-14,16.862762
2008-02-15,16.488358
2008-02-19,16.164228
2008-02-20,16.381197
2008-02-21,16.079557
2008-02-22,15.804375
2008-02-25,15.841419
2008-02-26,15.763363
2008-02-27,16.267419
2008-02-28,17.186894
2008-02-29,16.539955
2008-03-03,16.104692
2008-03-04,16.487035
2008-03-05,16.469837
2008-03-06,15.998854
2008-03-07,16.173487
2008-03-10,15.834804
2008-03-11,16.848211
2008-03-12,16.673576
2008-03-13,16.926267
2008-03-14,16.750309
2008-03-17,16.766185
2008-03-18,17.571882
2008-03-19,17.155143
2008-03-20,17.631418
2008-03-24,18.459606
2008-03-25,18.651438
2008-03-26,19.191215
2008-03-27,18.554861
2008-03-28,18.920004
2008-03-31,18.98483
2008-04-01,19.782591
2008-04-02,19.512701
2008-04-03,20.057771
2008-04-04,20.25225
2008-04-07,20.624009
2008-04-08,20.220498
2008-04-09,20.03528
2008-04-10,20.446728
2008-04-11,19.466397
2008-04-14,19.551068
2008-04-15,19.630448
2008-04-16,20.334276
2008-04-17,20.43879
2008-04-18,21.305346
2008-04-21,22.247311
2008-04-22,21.194214
2008-04-23,21.550098
2008-04-24,22.350503
2008-04-25,22.455019
2008-04-28,22.787089
2008-04-29,23.158847
2008-04-30,23.013319
2008-05-01,23.813724
2008-05-02,23.938085
2008-05-05,24.439496
2008-05-06,24.694832
2008-05-07,24.156378
2008-05-08,24.483155
2008-05-09,24.270155
2008-05-12,24.893279
2008-05-13,25.131417
2008-05-14,24.641913
2008-05-15,25.100989
2008-05-16,24.821838
2008-05-19,24.289999
2008-05-20,24.594285
2008-05-21,23.574265
2008-05-22,23.423443
2008-05-23,23.968513
2008-05-27,24.664403
2008-05-28,24.741136
2008-05-29,24.698801
2008-05-30,24.971335
2008-06-02,24.620745
2008-06-03,24.524167
2008-06-04,24.500354
2008-06-05,25.061299
2008-06-06,24.559888
2008-06-09,24.026726
2008-06-10,24.559888
2008-06-11,23.920886
2008-06-12,22.922033
2008-06-13,22.804288
2008-06-16,23.395662
2008-06-17,24.002911
2008-06-18,23.648352
2008-06-19,23.932793
2008-06-20,23.187954
2008-06-23,22.908803
2008-06-24,22.92071
2008-06-25,23.468426
2008-06-26,22.260541
2008-06-27,22.502646
2008-06-30,22.152056
2008-07-01,23.109897
2008-07-02,22.249957
2008-07-03,22.506615
2008-07-07,23.1734
2008-07-08,23.75419
2008-07-09,23.053008
2008-07-10,23.367878
2008-07-11,22.83207
2008-07-14,23.004058
2008-07-15,22.443112
2008-07-16,22.862499
2008-07-17,22.730199
2008-07-18,21.849092
2008-07-21,21.999913
2008-07-22,21.434998
2008-07-23,21.995944
2008-07-24,21.039425
2008-07-25,21.448228
2008-07-28,20.426883
2008-07-29,20.781444
2008-07-30,21.151879
2008-07-31,21.028843
2008-08-01,20.725877
2008-08-04,20.272094
2008-08-05,21.252427
2008-08-06,21.722087
2008-08-07,21.640061
2008-08-08,22.431205
2008-08-11,22.961722
2008-08-12,23.381108
2008-08-13,23.721116
2008-08-14,23.723762
2008-08-15,23.250134
2008-08-18,23.203828
2008-08-19,22.957754
2008-08-20,23.263364
2008-08-21,23.0583
2008-08-22,23.389047
2008-08-25,22.828101
2008-08-26,22.972306
2008-08-27,23.108574
2008-08-28,22.985536
2008-08-29,22.428559
2008-09-02,21.986682
2008-09-03,22.088553
2008-09-04,21.32916
2008-09-05,21.191568
2008-09-08,20.892574
2008-09-09,20.067032
2008-09-10,20.057771
2008-09-11,20.195361
2008-09-12,19.704535
2008-09-15,18.569413
2008-09-16,18.50591
2008-09-17,16.911714
2008-09-18,17.739902
2008-09-19,18.642177
2008-09-22,17.337714
2008-09-23,16.780739
2008-09-24,17.028135
2008-09-25,17.454137
2008-09-26,16.965955
2008-09-29,13.925737
2008-09-30,15.037044
2008-10-01,14.436409
2008-10-02,13.243077
2008-10-03,12.842212
2008-10-06,12.983772
2008-10-07,11.795731
2008-10-08,11.879079
2008-10-09,11.740166
2008-10-10,12.806492
2008-10-13,14.587229
2008-10-14,13.769625
2008-10-15,12.958635
2008-10-16,13.479891
2008-10-17,12.885871
2008-10-20,13.023461
2008-10-21,12.103987
2008-10-22,12.815753
2008-10-23,12.995678
2008-10-24,12.750927
2008-10-27,12.183366
2008-10-28,13.21794
2008-10-29,13.831805
2008-10-30,14.690422
2008-10-31,14.233992
2008-11-03,14.150644
2008-11-04,14.683807
2008-11-05,13.666432
2008-11-06,13.110778
2008-11-07,12.997001
2008-11-10,12.684777
2008-11-11,12.537926
2008-11-12,11.922738
2008-11-13,12.758864
2008-11-14,11.938614
2008-11-17,11.660787
2008-11-18,11.894956
2008-11-19,11.416035
2008-11-20,10.648704
2008-11-21,10.925208
2008-11-24,12.297142
2008-11-25,12.012701
2008-11-26,12.568354
2008-11-28,12.260099
2008-12-01,11.765303
2008-12-02,12.23364
2008-12-03,12.687423
2008-12-04,12.093403
2008-12-05,12.436056
2008-12-08,13.192803
2008-12-09,13.237785
2008-12-10,12.993032
2008-12-11,12.568354
2008-12-12,13.000971
2008-12-15,12.53528
2008-12-16,12.625243
2008-12-17,11.795731
2008-12-18,11.831452
2008-12-19,11.906862
2008-12-22,11.343271
2008-12-23,11.427942
2008-12-24,11.250661
2008-12-26,11.352532
2008-12-29,11.45837
2008-12-30,11.416035
2008-12-31,11.291674
2009-01-02,12.006086
2009-01-05,12.512789
2009-01-06,12.306404
2009-01-07,12.040484
2009-01-08,12.264068
2009-01-09,11.983595
2009-01-12,11.729582
2009-01-13,11.603899
2009-01-14,11.289028
2009-01-15,11.031046
2009-01-16,10.892133
2009-01-20,10.34574
2009-01-21,10.958282
2009-01-22,11.689893
2009-01-23,11.689893
2009-01-26,11.859235
2009-01-27,12.00344
2009-01-28,12.462516
2009-01-29,12.303758
2009-01-30,11.924061
2009-02-02,12.106633
2009-02-03,12.301112
2009-02-04,12.376522
2009-02-05,12.76151
2009-02-06,13.192803
2009-02-09,13.561917
2009-02-10,12.94276
2009-02-11,12.809138
2009-02-12,13.133269
2009-02-13,13.118716
2009-02-17,12.506174
2009-02-18,12.485006
2009-02-19,11.991533
2009-02-20,12.06562
2009-02-23,11.503352
2009-02-24,11.939937
2009-02-25,12.060328
2009-02-26,11.7997
2009-02-27,11.815576
2009-03-02,11.634327
2009-03-03,11.691216
2009-03-04,12.061651
2009-03-05,11.753396
2009-03-06,11.28506
2009-03-09,10.995326
2009-03-10,11.725613
2009-03-11,12.261422
2009-03-12,12.746958
2009-03-13,12.691392
2009-03-16,12.62392
2009-03-17,13.184865
2009-03-18,13.430941
2009-03-19,13.444171
2009-03-20,13.440202
2009-03-23,14.243253
2009-03-24,14.089787
2009-03-25,14.088464
2009-03-26,14.535633
2009-03-27,14.136092
2009-03-30,13.823867
2009-03-31,13.907215
2009-04-01,14.37952
2009-04-02,14.911361
2009-04-03,15.345299
2009-04-06,15.670754
2009-04-07,15.214324
2009-04-08,15.388959
2009-04-09,15.818928
2009-04-13,15.904922
2009-04-14,15.652232
2009-04-15,15.563591
2009-04-16,16.067649
2009-04-17,16.328276
2009-04-20,15.941965
2009-04-21,16.108661
2009-04-22,16.075588
2009-04-23,16.590228
2009-04-24,16.391781
2009-04-27,16.501588
2009-04-28,16.391781
2009-04-29,16.55583
2009-04-30,16.647116
2009-05-01,16.833657
2009-05-04,17.47266
2009-05-05,17.557329
2009-05-06,17.529547
2009-05-07,17.074441
2009-05-08,17.09164
2009-05-11,17.141913
2009-05-12,16.460576
2009-05-13,15.808344
2009-05-14,16.266096
2009-05-15,16.195978
2009-05-18,16.755601
2009-05-19,16.861441
2009-05-20,16.652408
2009-05-21,16.428824
2009-05-22,16.206563
2009-05-26,17.301994
2009-05-27,17.602312
2009-05-28,17.869553
2009-05-29,17.967455
2009-06-01,18.435792
2009-06-02,18.454314
2009-06-03,18.647469
2009-06-04,19.016583
2009-06-05,19.13962
2009-06-08,19.031134
2009-06-09,18.881637
2009-06-10,18.554861
2009-06-11,18.515171
2009-06-12,18.120921
2009-06-15,18.004499
2009-06-16,18.038896
2009-06-17,17.937027
2009-06-18,17.976716
2009-06-19,18.452991
2009-06-22,18.173842
2009-06-23,17.729317
2009-06-24,18.021697
2009-06-25,18.503264
2009-06-26,18.844594
2009-06-29,18.782414
2009-06-30,18.843271
2009-07-01,18.89619
2009-07-02,18.524432
2009-07-06,18.33789
2009-07-07,17.913213
2009-07-08,18.153996
2009-07-09,18.040219
2009-07-10,18.325985
2009-07-13,18.831364
2009-07-14,18.822103
2009-07-15,19.431999
2009-07-16,19.51667
2009-07-17,20.076293
2009-07-20,20.229759
2009-07-21,20.044541
2009-07-22,20.736462
2009-07-23,20.879344
2009-07-24,21.166433
2009-07-27,21.180986
2009-07-28,21.167756
2009-07-29,21.171725
2009-07-30,21.536868
2009-07-31,21.616247
2009-08-03,22.018435
2009-08-04,21.902011
2009-08-05,21.8438
2009-08-06,21.685042
2009-08-07,21.896719
2009-08-10,21.792205
2009-08-11,21.54216
2009-08-12,21.870261
2009-08-13,22.281708
2009-08-14,22.064739
2009-08-17,21.113512
2009-08-18,21.696949
2009-08-19,21.776328
2009-08-20,22.005205
2009-08-21,22.387547
2009-08-24,22.366379
2009-08-25,22.411361
2009-08-26,22.148087
2009-08-27,22.417977
2009-08-28,22.497354
2009-08-31,22.253926
2009-09-01,21.868938
2009-09-02,21.853061
2009-09-03,22.034309
2009-09-04,22.531752
2009-09-08,22.878375
2009-09-09,22.641561
2009-09-10,22.829424
2009-09-11,22.776505
2009-09-14,22.98289
2009-09-15,23.1734
2009-09-16,24.061122
2009-09-17,24.415683
2009-09-18,24.477863
2009-09-21,24.345565
2009-09-22,24.406422
2009-09-23,24.541366
2009-09-24,24.319105
2009-09-25,24.127271
2009-09-28,24.62736
2009-09-29,24.52549
2009-09-30,24.521521
2009-10-01,23.927501
2009-10-02,24.461987
2009-10-05,24.610161
2009-10-06,25.138032
2009-10-07,25.169784
2009-10-08,25.040132
2009-10-09,25.198889
2009-10-12,25.243871
2009-10-13,25.139355
2009-10-14,25.307374
2009-10-15,25.210796
2009-10-16,24.878728
2009-10-19,25.118188
2009-10-20,26.295643
2009-10-21,27.110603
2009-10-22,27.147646
2009-10-23,26.980949
2009-10-26,26.787794
2009-10-27,26.11175
2009-10-28,25.454225
2009-10-29,25.976804
2009-10-30,24.938262
2009-11-02,25.045422
2009-11-03,24.971335
2009-11-04,25.243871
2009-11-05,25.669871
2009-11-06,25.710885
2009-11-09,26.65285
2009-11-10,26.853943
2009-11-11,26.889665
2009-11-12,26.722968
2009-11-13,27.048423
2009-11-16,27.336833
2009-11-17,27.385783
2009-11-18,27.248193
2009-11-19,26.527166
2009-11-20,26.44911
2009-11-23,27.237609
2009-11-24,27.0471
2009-11-25,27.014025
2009-11-27,26.53775
2009-11-30,26.447787
2009-12-01,26.058829
2009-12-02,25.960929
2009-12-03,25.994003
2009-12-04,25.575941
2009-12-07,24.997796
2009-12-08,25.119511
2009-12-09,26.168637
2009-12-10,25.987388
2009-12-11,25.754543
2009-12-14,26.060152
2009-12-15,25.688394
2009-12-16,25.802171
2009-12-17,25.382784
2009-12-18,25.85509
2009-12-21,26.225525
2009-12-22,26.507322
2009-12-23,26.737522
2009-12-24,27.655673
2009-12-28,27.995679
2009-12-29,27.663611
2009-12-30,27.999648
2009-12-31,27.879257
2010-01-04,28.313195
2010-01-05,28.362145
2010-01-06,27.911008
2010-01-07,27.859412
2010-01-08,28.04463
2010-01-11,27.797232
2010-01-12,27.481038
2010-01-13,27.868673
2010-01-14,27.707269
2010-01-15,27.244224
2010-01-19,28.449462
2010-01-20,28.011555
2010-01-21,27.527342
2010-01-22,26.162022
2010-01-25,26.86585
2010-01-26,27.245547
2010-01-27,27.502206
2010-01-28,26.365761
2010-01-29,25.409244
2010-02-01,25.762481
2010-02-02,25.911978
2010-02-03,26.357823
2010-02-04,25.407921
2010-02-05,25.859059
2010-02-08,25.681779
2010-02-09,25.955637
2010-02-10,25.814078
2010-02-11,26.283736
2010-02-12,26.509966
2010-02-16,26.909508
2010-02-17,26.797055
2010-02-18,26.847328
2010-02-19,26.680632
2010-02-22,26.515259
2010-02-23,26.070736
2010-02-24,26.547011
2010-02-25,26.724291
2010-02-26,27.070912
2010-03-01,27.649058
2010-03-02,27.630535
2010-03-03,27.694038
2010-03-04,27.876611
2010-03-05,28.966751
2010-03-08,28.983948
2010-03-09,29.505205
2010-03-10,29.745987
2010-03-11,29.833306
2010-03-12,29.978834
2010-03-15,29.613689
2010-03-16,29.694391
2010-03-17,29.650733
2010-03-18,29.720851
2010-03-19,29.403335
2010-03-22,29.734082
2010-03-23,30.211678
2010-03-24,30.3453
2010-03-25,29.985447
2010-03-26,30.547716
2010-03-29,30.74484
2010-03-30,31.202593
2010-03-31,31.090142
2010-04-01,31.218469
2010-04-05,31.551862
2010-04-06,31.690777
2010-04-07,31.831013
2010-04-08,31.745018
2010-04-09,31.988446
2010-04-12,32.054597
2010-04-13,32.073119
2010-04-14,32.504411
2010-04-15,32.931736
2010-04-16,32.730642
2010-04-19,32.686984
2010-04-20,32.358883
2010-04-21,34.29441
2010-04-22,35.253573
2010-04-23,35.830393
2010-04-26,35.654438
2010-04-27,34.66749
2010-04-28,34.60928
2010-04-29,35.540662
2010-04-30,34.541808
2010-05-03,35.237697
2010-05-04,34.222967
2010-05-05,33.867085
2010-05-06,32.578497
2010-05-07,31.203918
2010-05-10,33.602489
2010-05-11,33.937203
2010-05-12,34.674106
2010-05-13,34.180634
2010-05-14,33.579996
2010-05-17,33.632917
2010-05-18,33.386843
2010-05-19,32.855001
2010-05-20,31.455283
2010-05-21,32.058564
2010-05-24,32.645969
2010-05-25,32.442231
2010-05-26,32.295378
2010-05-27,33.517816
2010-05-28,33.984832
2010-06-01,34.50741
2010-06-02,34.92018
2010-06-03,34.810372
2010-06-04,33.863118
2010-06-07,33.198978
2010-06-08,32.985978
2010-06-09,32.174989
2010-06-10,33.14209
2010-06-11,33.538984
2010-06-14,33.640855
2010-06-15,34.35659
2010-06-16,35.356764
2010-06-17,35.967983
2010-06-18,36.259043
2010-06-21,35.743078
2010-06-22,36.229937
2010-06-23,35.848915
2010-06-24,35.588287
2010-06-25,35.284
2010-06-28,35.495679
2010-06-29,33.890899
2010-06-30,33.277034
2010-07-01,32.873524
2010-07-02,32.669783
2010-07-06,32.893367
2010-07-07,34.221646
2010-07-08,34.144911
2010-07-09,34.347327
2010-07-12,34.039073
2010-07-13,33.312754
2010-07-14,33.435793
2010-07-15,33.26645
2010-07-16,33.061388
2010-07-19,32.48986
2010-07-20,33.324663
2010-07-21,33.635563
2010-07-22,34.26795
2010-07-23,34.389663
2010-07-26,34.302348
2010-07-27,34.937381
2010-07-28,34.524607
2010-07-29,34.147557
2010-07-30,34.033781
2010-08-02,34.642354
2010-08-03,34.652938
2010-08-04,34.791853
2010-08-05,34.622511
2010-08-06,34.40951
2010-08-09,34.629124
2010-08-10,34.319545
2010-08-11,33.099754
2010-08-12,33.311433
2010-08-13,32.955547
2010-08-16,32.762394
2010-08-17,33.335247
2010-08-18,33.480775
2010-08-19,33.058742
2010-08-20,33.02699
2010-08-23,32.518963
2010-08-24,31.742372
2010-08-25,32.133974
2010-08-26,31.788676
2010-08-27,31.965957
2010-08-30,32.082378
2010-08-31,32.161759
2010-09-01,33.118276
2010-09-02,33.361704
2010-09-03,34.234876
2010-09-07,34.10787
2010-09-08,34.783915
2010-09-09,34.803758
2010-09-10,34.848741
2010-09-13,35.328983
2010-09-14,35.463927
2010-09-15,35.749691
2010-09-16,36.589786
2010-09-17,36.431028
2010-09-20,37.470896
2010-09-21,37.542336
2010-09-22,38.068885
2010-09-23,38.223673
2010-09-24,38.673487
2010-09-27,38.520021
2010-09-28,37.951139
2010-09-29,38.018611
2010-09-30,37.53969
2010-10-01,37.376964
2010-10-04,36.863645
2010-10-05,38.226319
2010-10-06,38.259396
2010-10-07,38.263363
2010-10-08,38.905009
2010-10-11,39.075677
2010-10-12,39.496385
2010-10-13,39.708064
2010-10-14,39.995149
2010-10-15,41.63962
2010-10-18,42.070912
2010-10-19,40.945053
2010-10-20,41.082643
2010-10-21,40.949023
2010-10-22,40.677811
2010-10-25,40.859058
2010-10-26,40.754542
2010-10-27,40.725439
2010-10-28,40.382784
2010-10-29,39.819193
2010-11-01,40.242547
2010-11-02,40.927855
2010-11-03,41.382962
2010-11-04,42.106635
2010-11-05,41.955815
2010-11-08,42.152939
2010-11-09,41.8169
2010-11-10,42.074883
2010-11-11,41.89231
2010-11-12,40.751896
2010-11-15,40.620923
2010-11-16,39.899895
2010-11-17,39.755689
2010-11-18,40.804816
2010-11-19,40.579911
2010-11-22,41.457047
2010-11-23,40.844507
2010-11-24,41.647558
2010-11-26,41.674018
2010-11-29,41.921416
2010-11-30,41.16467
2010-12-01,41.859236
2010-12-02,42.090759
2010-12-03,41.996826
2010-12-06,42.355355
2010-12-07,42.098697
2010-12-08,42.46913
2010-12-09,42.303759
2010-12-10,42.409597
2010-12-13,42.556449
2010-12-14,42.373877
2010-12-15,42.383136
2010-12-16,42.500883
2010-12-17,42.416213
2010-12-20,42.627889
2010-12-21,42.891163
2010-12-22,43.018169
2010-12-23,42.811786
2010-12-27,42.954668
2010-12-28,43.059185
2010-12-29,43.03537
2010-12-30,42.819724
2010-12-31,42.674196
2011-01-03,43.601607
2011-01-04,43.829162
2011-01-05,44.18769
2011-01-06,44.151967
2011-01-07,44.468162
2011-01-10,45.305611
2011-01-11,45.198449
2011-01-12,45.56624
2011-01-13,45.732936
2011-01-14,46.10337
2011-01-18,45.067475
2011-01-19,44.828015
2011-01-20,44.013055
2011-01-21,43.224556
2011-01-24,44.644118
2011-01-25,45.166696
2011-01-26,45.49083
2011-01-27,45.406157
2011-01-28,44.465516
2011-01-31,44.891516
2011-02-01,45.646942
2011-02-02,45.553009
2011-02-03,45.436588
2011-02-04,45.84142
2011-02-07,46.553184
2011-02-08,46.992418
2011-02-09,47.38402
2011-02-10,46.905099
2011-02-11,47.21071
2011-02-14,47.518964
2011-02-15,47.614221
2011-02-16,48.041542
2011-02-17,47.402542
2011-02-18,46.378553
2011-02-22,44.797584
2011-02-23,45.3281
2011-02-24,45.362499
2011-02-25,46.061037
2011-02-28,46.729143
2011-03-01,46.213178
2011-03-02,46.584936
2011-03-03,47.569238
2011-03-04,47.627448
2011-03-07,47.013583
2011-03-08,47.066503
2011-03-09,46.63124
2011-03-10,45.863909
2011-03-11,46.567739
2011-03-14,46.775447
2011-03-15,45.699859
2011-03-16,43.659819
2011-03-17,44.272359
2011-03-18,43.747135
2011-03-21,44.88887
2011-03-22,45.140239
2011-03-23,44.874319
2011-03-24,45.639004
2011-03-25,46.508205
2011-03-28,46.362677
2011-03-29,46.43147
2011-03-30,46.123217
2011-03-31,46.10734
2011-04-01,45.584762
2011-04-04,45.138915
2011-04-05,44.834628
2011-04-06,44.722174
2011-04-07,44.727466
2011-04-08,44.327926
2011-04-11,43.764332
2011-04-12,43.976011
2011-04-13,44.469483
2011-04-14,43.978657
2011-04-15,43.322456
2011-04-18,43.903247
2011-04-19,44.698359
2011-04-20,45.300319
2011-04-21,46.397072
2011-04-25,46.702683
2011-04-26,46.360031
2011-04-27,46.324308
2011-04-28,45.874493
2011-04-29,46.321662
2011-05-02,45.812313
2011-05-03,46.066329
2011-05-04,46.247576
2011-05-05,45.874493
2011-05-06,45.862588
2011-05-09,45.986948
2011-05-10,46.2317
2011-05-11,45.937998
2011-05-12,45.850679
2011-05-13,45.047628
2011-05-16,44.095079
2011-05-17,44.470808
2011-05-18,44.96428
2011-05-19,45.051599
2011-05-20,44.349094
2011-05-23,44.240607
2011-05-24,43.948229
2011-05-25,44.555477
2011-05-26,44.319988
2011-05-27,44.638826
2011-05-31,46.017375
2011-06-01,45.710443
2011-06-02,45.788499
2011-06-03,45.436588
2011-06-06,44.722174
2011-06-07,43.928382
2011-06-08,43.954843
2011-06-09,43.855618
2011-06-10,43.116072
2011-06-13,43.20868
2011-06-14,43.981303
2011-06-15,43.228523
2011-06-16,43.018169
2011-06-17,42.369906
2011-06-20,41.716354
2011-06-21,43.036692
2011-06-22,42.680809
2011-06-23,43.821224
2011-06-24,43.175606
2011-06-27,43.928382
2011-06-28,44.354386
2011-06-29,44.192982
2011-06-30,44.408628
2011-07-01,45.412773
2011-07-05,46.229054
2011-07-06,46.537308
2011-07-07,47.257014
2011-07-08,47.589082
2011-07-11,46.83366
2011-07-12,46.800583
2011-07-13,47.365498
2011-07-14,47.332424
2011-07-15,48.278357
2011-07-18,49.453169
2011-07-19,49.856677
2011-07-20,51.186277
2011-07-21,51.237876
2011-07-22,52.032988
2011-07-25,52.720939
2011-07-26,53.370527
2011-07-27,51.939056
2011-07-28,51.837186
2011-07-29,51.659905
2011-08-01,52.489416
2011-08-02,51.452197
2011-08-03,51.93641
2011-08-04,49.925473
2011-08-05,49.429355
2011-08-08,46.729143
2011-08-09,49.480951
2011-08-10,48.115631
2011-08-11,49.439939
2011-08-12,49.875199
2011-08-15,50.724557
2011-08-16,50.336922
2011-08-17,50.33163
2011-08-18,48.427856
2011-08-19,47.102223
2011-08-22,47.156465
2011-08-23,49.426709
2011-08-24,49.76804
2011-08-25,49.442585
2011-08-26,50.747046
2011-08-29,51.592433
2011-08-30,51.595079
2011-08-31,50.912421
2011-09-01,50.409686
2011-09-02,49.486243
2011-09-06,50.239023
2011-09-07,50.79335
2011-09-08,50.821135
2011-09-09,49.940025
2011-09-12,50.265479
2011-09-13,50.884636
2011-09-14,51.503793
2011-09-15,51.988006
2011-09-16,52.985538
2011-09-19,54.45802
2011-09-20,54.698802
2011-09-21,54.525492
2011-09-22,53.160173
2011-09-23,53.48827
2011-09-26,53.338774
2011-09-27,52.821488
2011-09-28,52.523815
2011-09-29,51.671814
2011-09-30,50.448052
2011-10-03,49.559007
2011-10-04,49.281181
2011-10-05,50.041895
2011-10-06,49.925473
2011-10-07,48.923974
2011-10-10,51.438967
2011-10-11,52.957753
2011-10-12,53.209122
2011-10-13,54.034662
2011-10-14,55.829953
2011-10-17,55.564036
2011-10-18,55.861705
2011-10-19,52.736815
2011-10-20,52.298909
2011-10-21,51.976101
2011-10-24,53.682751
2011-10-25,52.624364
2011-10-26,52.998768
2011-10-27,53.539869
2011-10-28,53.574264
2011-10-31,53.551774
2011-11-01,52.457667
2011-11-02,52.576735
2011-11-03,53.325544
2011-11-04,52.95114
2011-11-07,52.883668
2011-11-08,53.743606
2011-11-09,52.294938
2011-11-10,50.964017
2011-11-11,50.884636
2011-11-14,50.175518
2011-11-15,51.441613
2011-11-16,50.904483
2011-11-17,49.930765
2011-11-18,49.60399
2011-11-21,48.819458
2011-11-22,49.811698
2011-11-23,48.552216
2011-11-25,48.099755
2011-11-28,49.760102
2011-11-29,49.373788
2011-11-30,50.564474
2011-12-01,51.322545
2011-12-02,51.556714
2011-12-05,51.994623
2011-12-06,51.722085
2011-12-07,51.476012
2011-12-08,51.68372
2011-12-09,52.075325
2011-12-12,51.839832
2011-12-13,51.438967
2011-12-14,50.298556
2011-12-15,50.133181
2011-12-16,50.408361
2011-12-19,50.565798
2011-12-20,52.383579
2011-12-21,52.449729
2011-12-22,52.727555
2011-12-23,53.359942
2011-12-27,53.783297
2011-12-28,53.268656
2011-12-29,53.596757
2011-12-30,53.580881
2012-01-03,54.4051
2012-01-04,54.697481
2012-01-05,55.304729
2012-01-06,55.882874
2012-01-09,55.794234
2012-01-10,55.994003
2012-01-11,55.902717
2012-01-12,55.749251
2012-01-13,55.540222
2012-01-17,56.18716
2012-01-18,56.770597
2012-01-19,56.590671
2012-01-20,55.605048
2012-01-23,56.545688
2012-01-24,55.619599
2012-01-25,59.092434
2012-01-26,58.823867
2012-01-27,59.174461
2012-01-30,59.932529
2012-01-31,60.391603
2012-02-01,60.353237
2012-02-02,60.21168
2012-02-03,60.814961
2012-02-06,61.382518
2012-02-07,62.025493
2012-02-08,63.064037
2012-02-09,65.245637
2012-02-10,65.278711
2012-02-13,66.493214
2012-02-14,67.400777
2012-02-15,65.840976
2012-02-16,66.441614
2012-02-17,66.429709
2012-02-21,68.11387
2012-02-22,67.874406
2012-02-23,68.317604
2012-02-24,69.114041
2012-02-27,69.557246
2012-02-28,70.833925
2012-02-29,71.763985
2012-03-01,72.032545
2012-03-02,72.126481
2012-03-05,70.536256
2012-03-06,70.152585
2012-03-07,70.209476
2012-03-08,71.704444
2012-03-09,72.125159
2012-03-12,73.028752
2012-03-13,75.15876
2012-03-14,78.000533
2012-03-15,77.468692
2012-03-16,77.470013
2012-03-19,79.524611
2012-03-20,80.167578
2012-03-21,79.709826
2012-03-22,79.291767
2012-03-23,78.856504
2012-03-26,80.302522
2012-03-27,81.294766
2012-03-28,81.710182
2012-03-29,80.683544
2012-03-30,79.319549
2012-04-02,81.843805
2012-04-03,83.258071
2012-04-04,82.595256
2012-04-05,83.834891
2012-04-09,84.172255
2012-04-10,83.141653
2012-04-11,82.845304
2012-04-12,82.391515
2012-04-13,80.071
2012-04-16,76.750314
2012-04-17,80.662375
2012-04-18,80.482452
2012-04-19,77.717411
2012-04-20,75.804378
2012-04-23,75.635032
2012-04-24,74.124187
2012-04-25,80.702069
2012-04-26,80.39778
2012-04-27,79.77598
2012-04-30,77.259659
2012-05-01,77.01491
2012-05-02,77.524255
2012-05-03,76.973895
2012-05-04,74.78171
2012-05-07,75.341333
2012-05-08,75.169344
2012-05-09,75.301646
2012-05-10,75.478919
2012-05-11,74.974867
2012-05-14,73.85165
2012-05-15,73.183543
2012-05-16,72.245548
2012-05-17,70.134066
2012-05-18,70.168461
2012-05-21,74.256482
2012-05-22,73.686282
2012-05-23,75.484211
2012-05-24,74.790973
2012-05-25,74.390105
2012-05-29,75.710442
2012-05-30,76.623304
2012-05-31,76.432797
2012-06-01,74.218116
2012-06-04,74.6547
2012-06-05,74.461551
2012-06-06,75.603286
2012-06-07,75.637681
2012-06-08,76.775446
2012-06-11,75.564914
2012-06-12,76.225086
2012-06-13,75.695894
2012-06-14,75.612542
2012-06-15,75.956519
2012-06-18,77.497794
2012-06-19,77.713448
2012-06-20,77.492502
2012-06-21,76.424856
2012-06-22,77.010939
2012-06-25,75.512
2012-06-26,75.678696
2012-06-27,76.005469
2012-06-28,75.284441
2012-06-29,77.262308
2012-07-02,78.389489
2012-07-03,79.301023
2012-07-05,80.694128
2012-07-06,80.156994
2012-07-09,81.216706
2012-07-10,80.465255
2012-07-11,79.965166
2012-07-12,79.233555
2012-07-13,80.036605
2012-07-16,80.293267
2012-07-17,80.29723
2012-07-18,80.207272
2012-07-19,81.273598
2012-07-20,79.947969
2012-07-23,79.885785
2012-07-24,79.500793
2012-07-25,76.067652
2012-07-26,76.055747
2012-07-27,77.415771
2012-07-30,78.72156
2012-07-31,80.802612
2012-08-01,80.280033
2012-08-02,80.409685
2012-08-03,81.45617
2012-08-06,82.362413
2012-08-07,82.145445
2012-08-08,82.00653
2012-08-09,82.474217
2012-08-10,82.603098
2012-08-13,83.705891
2012-08-14,83.930439
2012-08-15,83.816167
2012-08-16,84.548266
2012-08-17,86.112103
2012-08-20,88.37615
2012-08-21,87.168389
2012-08-22,88.870411
2012-08-23,88.041326
2012-08-24,88.119714
2012-08-27,89.775229
2012-08-28,89.658311
2012-08-29,89.481596
2012-08-30,88.206077
2012-08-31,88.388107
2012-09-04,89.680897
2012-09-05,89.051113
2012-09-06,89.853624
2012-09-07,90.407674
2012-09-10,88.055936
2012-09-11,87.770279
2012-09-12,88.992651
2012-09-13,90.745159
2012-09-14,91.847951
2012-09-17,92.977318
2012-09-18,93.260321
2012-09-19,93.285568
2012-09-20,92.833818
2012-09-21,93.018502
2012-09-24,91.782847
2012-09-25,89.490899
2012-09-26,88.380131
2012-09-27,90.524599
2012-09-28,88.635241
2012-10-01,87.610836
2012-10-02,87.865938
2012-10-03,89.213203
2012-10-04,88.595376
2012-10-05,86.707344
2012-10-08,84.791412
2012-10-09,84.483162
2012-10-10,85.155464
2012-10-11,83.453442
2012-10-12,83.66736
2012-10-15,84.338336
2012-10-16,86.335317
2012-10-17,85.64707
2012-10-18,84.056659
2012-10-19,81.027305
2012-10-22,84.241343
2012-10-23,81.494991
2012-10-24,81.956036
2012-10-25,80.98744
2012-10-26,80.251361
2012-10-31,79.098082
2012-11-01,79.260179
2012-11-02,76.637395
2012-11-05,77.67641
2012-11-06,77.441232
2012-11-07,74.478128
2012-11-08,71.775289
2012-11-09,73.017928
2012-11-12,72.453335
2012-11-13,72.46268
2012-11-14,71.65917
2012-11-15,70.156258
2012-11-16,70.431216
2012-11-19,75.509877
2012-11-20,74.866531
2012-11-21,74.97198
2012-11-23,76.28002
2012-11-26,78.686544
2012-11-27,78.052544
2012-11-28,77.806954
2012-11-29,78.663848
2012-11-30,78.119278
2012-12-03,78.240744
2012-12-04,76.860623
2012-12-05,71.914105
2012-12-06,73.04195
2012-12-07,71.174663
2012-12-10,70.716845
2012-12-11,72.261136
2012-12-12,71.942133
2012-12-13,70.699495
2012-12-14,68.043375
2012-12-17,69.249973
2012-12-18,71.261421
2012-12-19,70.248355
2012-12-20,69.637051
2012-12-21,69.316715
2012-12-24,69.428828
2012-12-26,68.471825
2012-12-27,68.746783
2012-12-28,68.016679
2012-12-31,71.030509
2013-01-02,73.280868
2013-01-03,72.355899
2013-01-04,70.340452
2013-01-07,69.926686
2013-01-08,70.114886
2013-01-09,69.019068
2013-01-10,69.874628
2013-01-11,69.446185
2013-01-14,66.970253
2013-01-15,64.85737
2013-01-16,67.549523
2013-01-17,67.094384
2013-01-18,66.736675
2013-01-22,67.373341
2013-01-23,68.606634
2013-01-24,60.129739
2013-01-25,58.712255
2013-01-28,60.040315
2013-01-29,61.166827
2013-01-30,60.974629
2013-01-31,60.795774
2013-02-01,60.546178
2013-02-04,59.037931
2013-02-05,61.109438
2013-02-06,61.044036
2013-02-07,62.859111
2013-02-08,63.766649
2013-02-11,64.431195
2013-02-12,62.81615
2013-02-13,62.696665
2013-02-14,62.640282
2013-02-15,61.777047
2013-02-19,61.754225
2013-02-20,60.258665
2013-02-21,59.884104
2013-02-22,60.521796
2013-02-25,59.446442
2013-02-26,60.274775
2013-02-27,59.684067
2013-02-28,59.258493
2013-03-01,57.791128
2013-03-04,56.392227
2013-03-05,57.881073
2013-03-06,57.145377
2013-03-07,57.805894
2013-03-08,57.95894
2013-03-11,58.784585
2013-03-12,57.517253
2013-03-13,57.506512
2013-03-14,58.063655
2013-03-15,59.561901
2013-03-18,61.180972
2013-03-19,61.015837
2013-03-20,60.692294
2013-03-21,60.779557
2013-03-22,62.011986
2013-03-25,62.236186
2013-03-26,61.908612
2013-03-27,60.692294
2013-03-28,59.42765
2013-04-01,57.581695
2013-04-02,57.699835
2013-04-03,57.995187
2013-04-04,57.421934
2013-04-05,56.815119
2013-04-08,57.219215
2013-04-09,57.322589
2013-04-10,58.491918
2013-04-11,58.309335
2013-04-12,57.701179
2013-04-15,56.36538
2013-04-16,57.223245
2013-04-17,54.076396
2013-04-18,52.633192
2013-04-19,52.429133
2013-04-22,53.521938
2013-04-23,54.52345
2013-04-24,54.433501
2013-04-25,54.825517
2013-04-26,56.009611
2013-04-29,57.744137
2013-04-30,59.443757
2013-05-01,58.975222
2013-05-02,59.811607
2013-05-03,60.41037
2013-05-06,61.850881
2013-05-07,61.575669
2013-05-08,62.271088
2013-05-09,61.727821
2013-05-10,61.214292
2013-05-13,61.453494
2013-05-14,59.983171
2013-05-15,57.954721
2013-05-16,58.729073
2013-05-17,58.550688
2013-05-20,59.85749
2013-05-21,59.415582
2013-05-22,59.643968
2013-05-23,59.750729
2013-05-24,60.1575
2013-05-28,59.656132
2013-05-29,60.130471
2013-05-30,61.026449
2013-05-31,60.776444
2013-06-03,60.91023
2013-06-04,60.71968
2013-06-05,60.152094
2013-06-06,59.253414
2013-06-07,59.706133
2013-06-10,59.311523
2013-06-11,59.137194
2013-06-12,58.406087
2013-06-13,58.915563
2013-06-14,58.116889
2013-06-17,58.380412
2013-06-18,58.349327
2013-06-19,57.164151
2013-06-20,56.331692
2013-06-21,55.880325
2013-06-24,54.399191
2013-06-25,54.411352
2013-06-26,53.795115
2013-06-27,53.215364
2013-06-28,53.587002
2013-07-01,55.301924
2013-07-02,56.554672
2013-07-03,56.866844
2013-07-05,56.410071
2013-07-08,56.089792
2013-07-09,57.07631
2013-07-10,56.857385
2013-07-11,57.743901
2013-07-12,57.638493
2013-07-15,57.764175
2013-07-16,58.137158
2013-07-17,58.152026
2013-07-18,58.347978
2013-07-19,57.427674
2013-07-22,57.611465
2013-07-23,56.62224
2013-07-24,59.530451
2013-07-25,59.258819
2013-07-26,59.59532
2013-07-29,60.51427
2013-07-30,61.261594
2013-07-31,61.15483
2013-08-01,61.71566
2013-08-02,62.50758
2013-08-05,63.441397
2013-08-06,62.873812
2013-08-07,62.837321
2013-08-08,62.712171
2013-08-09,61.819794
2013-08-12,63.575974
2013-08-13,66.597243
2013-08-14,67.812014
2013-08-15,67.731751
2013-08-16,68.333016
2013-08-19,69.06895
2013-08-20,68.161615
2013-08-21,68.337099
2013-08-22,68.418713
2013-08-23,68.154815
2013-08-26,68.420078
2013-08-27,66.463932
2013-08-28,66.778168
2013-08-29,66.886996
2013-08-30,66.277572
2013-09-03,66.462573
2013-09-04,67.837862
2013-09-05,67.372625
2013-09-06,67.773924
2013-09-09,68.855377
2013-09-10,67.286929
2013-09-11,63.623581
2013-09-12,64.301026
2013-09-13,63.241331
2013-09-16,61.230778
2013-09-17,61.938147
2013-09-18,63.211407
2013-09-19,64.24797
2013-09-20,63.582774
2013-09-23,66.742802
2013-09-24,66.533311
2013-09-25,65.503548
2013-09-26,66.141537
2013-09-27,65.669508
2013-09-30,64.853311
2013-10-01,66.378235
2013-10-02,66.595884
2013-10-03,65.759287
2013-10-04,65.707598
2013-10-07,66.349669
2013-10-08,65.423286
2013-10-09,66.191868
2013-10-10,66.606767
2013-10-11,67.037989
2013-10-14,67.477371
2013-10-15,67.836496
2013-10-16,68.167056
2013-10-17,68.628203
2013-10-18,69.225385
2013-10-21,70.92171
2013-10-22,70.719019
2013-10-23,71.411422
2013-10-24,72.356847
2013-10-25,71.547458
2013-10-28,72.080701
2013-10-29,70.28508
2013-10-30,71.403264
2013-10-31,71.103986
2013-11-01,70.740785
2013-11-04,71.65492
2013-11-05,71.478078
2013-11-06,71.275573
2013-11-07,70.122128
2013-11-08,71.226314
2013-11-11,71.019707
2013-11-12,71.151056
2013-11-13,71.235894
2013-11-14,72.266196
2013-11-15,71.832456
2013-11-18,70.962235
2013-11-19,71.088118
2013-11-20,70.465557
2013-11-21,71.305671
2013-11-22,71.122324
2013-11-25,71.661421
2013-11-26,72.98316
2013-11-27,74.701702
2013-11-29,76.085013
2013-12-02,75.42278
2013-12-03,77.487484
2013-12-04,77.306876
2013-12-05,77.703671
2013-12-06,76.625477
2013-12-09,77.502537
2013-12-10,77.382127
2013-12-11,76.808824
2013-12-12,76.696628
2013-12-13,75.86062
2013-12-16,76.280681
2013-12-17,75.937244
2013-12-18,75.359835
2013-12-19,74.496461
2013-12-20,75.120389
2013-12-23,78.003322
2013-12-24,77.672198
2013-12-26,77.156368
2013-12-27,76.635056
2013-12-30,75.872933
2013-12-31,76.762306
2014-01-02,75.682745
2014-01-03,74.020309
2014-01-06,74.423944
2014-01-07,73.891693
2014-01-08,74.359639
2014-01-09,73.410061
2014-01-10,72.920222
2014-01-13,73.301971
2014-01-14,74.76054
2014-01-15,76.261521
2014-01-16,75.835994
2014-01-17,73.97789
2014-01-21,75.127229
2014-01-22,75.461085
2014-01-23,76.100066
2014-01-24,74.716755
2014-01-27,75.322897
2014-01-28,69.302532
2014-01-29,68.515783
2014-01-30,68.38306
2014-01-31,68.495256
2014-02-03,68.622506
2014-02-04,69.61587
2014-02-05,70.135807
2014-02-06,70.544607
2014-02-07,71.531523
2014-02-10,72.813003
2014-02-11,73.772389
2014-02-12,73.766883
2014-02-13,74.938244
2014-02-14,74.877686
2014-02-18,75.152975
2014-02-19,73.966471
2014-02-20,73.110316
2014-02-21,72.298208
2014-02-24,72.614796
2014-02-25,71.859123
2014-02-26,71.210811
2014-02-27,72.631314
2014-02-28,72.434475
2014-03-03,72.643701
2014-03-04,73.122703
2014-03-05,73.276869
2014-03-06,73.055257
2014-03-07,73.012591
2014-03-10,73.078655
2014-03-11,73.790281
2014-03-12,73.861858
2014-03-13,73.041495
2014-03-14,72.221125
2014-03-17,72.503303
2014-03-18,73.144726
2014-03-19,73.125459
2014-03-20,72.773086
2014-03-21,73.347064
2014-03-24,74.216988
2014-03-25,75.015327
2014-03-26,74.298195
2014-03-27,73.978858
2014-03-28,73.896269
2014-03-31,73.879751
2014-04-01,74.555592
2014-04-02,74.679472
2014-04-03,74.161928
2014-04-04,73.202542
2014-04-07,72.053198
2014-04-08,72.049074
2014-04-09,72.996073
2014-04-10,72.05458
2014-04-11,71.521893
2014-04-14,71.806813
2014-04-15,71.294775
2014-04-16,71.439304
2014-04-17,72.255543
2014-04-21,73.113073
2014-04-22,73.186025
2014-04-23,72.229388
2014-04-24,78.150889
2014-04-25,78.724867
2014-04-28,81.773717
2014-04-29,81.531455
2014-04-30,81.22313
2014-05-01,81.414463
2014-05-02,81.565873
2014-05-05,82.719334
2014-05-06,81.817764
2014-05-07,81.531455
2014-05-08,81.386125
2014-05-09,81.047012
2014-05-12,82.05605
2014-05-13,82.184776
2014-05-14,82.199996
2014-05-15,81.501005
2014-05-16,82.703828
2014-05-19,83.6838
2014-05-20,83.70041
2014-05-21,83.92187
2014-05-22,84.054744
2014-05-23,85.004268
2014-05-27,86.596033
2014-05-28,86.371801
2014-05-29,87.945567
2014-05-30,87.616145
2014-06-02,87.014043
2014-06-03,88.244541
2014-06-04,89.252197
2014-06-05,89.602383
2014-06-06,89.356011
2014-06-09,90.785823
2014-06-10,91.318721
2014-06-11,90.940851
2014-06-12,89.41968
2014-06-13,88.44109
2014-06-16,89.332475
2014-06-17,89.216212
2014-06-18,89.3131
2014-06-19,89.003053
2014-06-20,88.082602
2014-06-23,88.005088
2014-06-24,87.472191
2014-06-25,87.549705
2014-06-26,88.072911
2014-06-27,89.119323
2014-06-30,90.039774
2014-07-01,90.611421
2014-07-02,90.572672
2014-07-03,91.105562
2014-07-07,92.985228
2014-07-08,92.384508
2014-07-09,92.423265
2014-07-10,92.084152
2014-07-11,92.258554
2014-07-14,93.450295
2014-07-15,92.355442
2014-07-16,91.832236
2014-07-17,90.194794
2014-07-18,91.493123
2014-07-21,91.018364
2014-07-22,91.774104
2014-07-23,94.167286
2014-07-24,94.012258
2014-07-25,94.632353
2014-07-28,95.940365
2014-07-29,95.32027
2014-07-30,95.097428
2014-07-31,92.626733
2014-08-01,93.140248
2014-08-04,92.617042
2014-08-05,92.161665
2014-08-06,92.006638
2014-08-07,91.996907
2014-08-08,92.250068
2014-08-11,93.467216
2014-08-12,93.447745
2014-08-13,94.684364
2014-08-14,94.937533
2014-08-15,95.404921
2014-08-18,96.553909
2014-08-19,97.887898
2014-08-20,97.926848
2014-08-21,97.936587
2014-08-22,98.657136
2014-08-25,98.871356
2014-08-26,98.238437
2014-08-27,99.445846
2014-08-28,99.562695
2014-08-29,99.806124
2014-09-02,100.585102
2014-09-03,96.33969
2014-09-04,95.541241
2014-09-05,96.3689
2014-09-08,95.774931
2014-09-09,95.414653
2014-09-10,98.345547
2014-09-11,98.764246
2014-09-12,98.988204
2014-09-15,98.958987
2014-09-16,98.209227
2014-09-17,98.910305
2014-09-18,99.114785
2014-09-19,98.306597
2014-09-22,98.403968
2014-09-23,99.942444
2014-09-24,99.075836
2014-09-25,95.297811
2014-09-26,98.102117
2014-09-29,97.478938
2014-09-30,98.102117
2014-10-01,96.57338
2014-10-02,97.274458
2014-10-03,97.001818
2014-10-06,97.001818
2014-10-07,96.154681
2014-10-08,98.150806
2014-10-09,98.365018
2014-10-10,98.082646
2014-10-13,97.18682
2014-10-14,96.154681
2014-10-15,94.976482
2014-10-16,93.730124
2014-10-17,95.103063
2014-10-20,97.138138
2014-10-21,99.776914
2014-10-22,100.283244
2014-10-23,102.07489
2014-10-24,102.454639
2014-10-27,102.34753
2014-10-28,103.934688
2014-10-29,104.518917
2014-10-30,104.168385
2014-10-31,105.161575
2014-11-03,106.524782
2014-11-04,105.745804
2014-11-05,105.998973
2014-11-06,106.302129
2014-11-07,106.605296
2014-11-10,106.429266
2014-11-11,107.28007
2014-11-12,108.795881
2014-11-13,110.331247
2014-11-14,111.661247
2014-11-17,111.475435
2014-11-18,112.922791
2014-11-19,112.140435
2014-11-20,113.744257
2014-11-21,113.900731
2014-11-24,116.013079
2014-11-25,115.005801
2014-11-26,116.374919
2014-11-28,116.306464
2014-12-01,112.531613
2014-12-02,112.101317
2014-12-03,113.372642
2014-12-04,112.942346
2014-12-05,112.463157
2014-12-08,109.920514
2014-12-09,111.602573
2014-12-10,109.480436
2014-12-11,109.157721
2014-12-12,107.309414
2014-12-15,105.842504
2014-12-16,104.395148
2014-12-17,106.996474
2014-12-18,110.164999
2014-12-19,109.314188
2014-12-22,110.448602
2014-12-23,110.057425
2014-12-24,109.539117
2014-12-26,111.475435
2014-12-29,111.397206
2014-12-30,110.037862
2014-12-31,107.94507
2015-01-02,106.918237
2015-01-05,103.906178
2015-01-06,103.91596
2015-01-07,105.373089
2015-01-08,109.421762
2015-01-09,109.539117
2015-01-12,106.84
2015-01-13,107.788603
2015-01-14,107.37787
2015-01-15,104.463604
2015-01-16,103.651911
2015-01-20,106.321692
2015-01-21,107.133385
2015-01-22,109.920514
2015-01-23,110.487721
2015-01-26,110.605069
2015-01-27,106.732426
2015-01-28,112.766317
2015-01-29,116.277127
2015-01-30,114.575513
2015-02-02,116.013079
2015-02-03,116.032642
2015-02-04,116.922564
2015-02-05,117.757104
2015-02-06,116.765484
2015-02-09,117.541107
2015-02-10,119.799242
2015-02-11,122.607191
2015-02-12,124.158437
2015-02-13,124.767156
2015-02-17,125.503506
2015-02-18,126.377307
2015-02-19,126.112217
2015-02-20,127.14311
2015-02-23,130.579411
2015-02-24,129.764515
2015-02-25,126.446026
2015-02-26,128.046365
2015-02-27,126.122045
2015-03-02,126.740569
2015-03-03,127.005659
2015-03-04,126.200576
2015-03-05,124.109352
2015-03-06,124.295889
2015-03-09,124.826061
2015-03-10,122.24393
2015-03-11,120.01524
2015-03-12,122.185017
2015-03-13,121.340668
2015-03-16,122.675917
2015-03-17,124.727883
2015-03-18,126.131857
2015-03-19,125.17951
2015-03-20,123.608631
2015-03-23,124.894787
2015-03-24,124.384254
2015-03-25,121.134491
2015-03-26,121.97884
2015-03-27,121.00686
2015-03-30,124.070079
2015-03-31,122.165384
2015-04-01,121.98866
2015-04-02,123.039186
2015-04-06,125.032239
2015-04-07,123.71663
2015-04-08,123.314088
2015-04-09,124.256616
2015-04-10,124.786789
2015-04-13,124.541339
2015-04-14,124.001353
2015-04-15,124.472613
2015-04-16,123.873714
2015-04-17,122.47956
2015-04-20,125.277689
2015-04-21,124.600252
2015-04-22,126.279121
2015-04-23,127.310014
2015-04-24,127.908913
2015-04-27,130.235775
2015-04-28,128.183816
2015-04-29,126.298762
2015-04-30,122.872281
2015-05-01,126.603117
2015-05-04,126.357667
2015-05-05,123.510453
2015-05-06,122.73483
2015-05-07,123.493969
2015-05-08,125.820696
2015-05-11,124.539021
2015-05-12,124.095369
2015-05-13,124.233394
2015-05-14,127.131938
2015-05-15,126.954483
2015-05-18,128.354461
2015-05-19,128.236158
2015-05-20,128.226289
2015-05-21,129.537539
2015-05-22,130.67132
2015-05-26,127.79249
2015-05-27,130.178369
2015-05-28,129.92204
2015-05-29,128.443189
2015-06-01,128.699517
2015-06-02,128.127708
2015-06-03,128.285441
2015-06-04,127.536161
2015-06-05,126.836165
2015-06-08,125.998158
2015-06-09,125.623511
2015-06-10,127.062933
2015-06-11,126.777013
2015-06-12,125.377036
2015-06-15,125.13056
2015-06-16,125.800973
2015-06-17,125.505208
2015-06-18,126.077025
2015-06-19,124.815072
2015-06-22,125.810835
2015-06-23,125.23901
2015-06-24,126.303785
2015-06-25,125.702385
2015-06-26,124.962959
2015-06-29,122.774258
2015-06-30,123.66157
2015-07-01,124.815072
2015-07-02,124.657332
2015-07-06,124.223533
2015-07-07,123.917906
2015-07-08,120.841892
2015-07-09,118.37714
2015-07-10,121.541881
2015-07-13,123.888331
2015-07-14,123.839032
2015-07-15,125.031972
2015-07-16,126.69814
2015-07-17,127.79249
2015-07-20,130.20796
2015-07-21,128.906563
2015-07-22,123.454532
2015-07-23,123.39538
2015-07-24,122.744682
2015-07-27,121.03907
2015-07-28,121.64047
2015-07-29,121.255969
2015-07-30,120.644715
2015-07-31,119.589801
2015-08-03,116.770124
2015-08-04,113.023697
2015-08-05,113.772984
2015-08-06,114.020572
2015-08-07,114.406813
2015-08-10,118.566345
2015-08-11,112.396376
2015-08-12,114.129513
2015-08-13,114.040384
2015-08-14,114.842576
2015-08-17,116.031017
2015-08-18,115.377373
2015-08-19,113.901733
2015-08-20,111.564474
2015-08-21,104.740869
2015-08-24,102.126309
2015-08-25,102.74033
2015-08-26,108.632999
2015-08-27,111.831869
2015-08-28,112.198306
2015-08-31,111.673415
2015-09-01,106.681981
2015-09-02,111.257456
2015-09-03,109.306446
2015-09-04,108.21704
2015-09-08,111.227747
2015-09-09,109.088565
2015-09-10,111.485243
2015-09-11,113.109439
2015-09-14,114.198838
2015-09-15,115.159492
2015-09-16,115.288244
2015-09-17,112.822233
2015-09-18,112.356761
2015-09-21,114.099803
2015-09-22,112.307247
2015-09-23,113.21838
2015-09-24,113.891827
2015-09-25,113.604621
2015-09-28,111.356499
2015-09-29,108.009065
2015-09-30,109.237121
2015-10-01,108.524058
2015-10-02,109.316344
2015-10-05,109.712491
2015-10-06,110.237383
2015-10-07,109.712491
2015-10-08,108.444827
2015-10-09,111.039583
2015-10-12,110.524589
2015-10-13,110.712761
2015-10-14,109.147984
2015-10-15,110.782086
2015-10-16,109.969988
2015-10-19,110.653341
2015-10-20,112.673677
2015-10-21,112.663779
2015-10-22,114.387009
2015-10-23,117.932513
2015-10-26,114.169128
2015-10-27,113.446167
2015-10-28,118.120677
2015-10-29,119.368538
2015-10-30,118.348464
2015-11-02,120.012276
2015-11-03,121.388881
2015-11-04,120.824373
2015-11-05,120.267398
2015-11-06,120.406641
2015-11-09,119.919288
2015-11-10,116.139793
2015-11-11,115.483359
2015-11-12,115.095465
2015-11-13,111.733702
2015-11-16,113.563775
2015-11-17,113.076422
2015-11-18,116.656991
2015-11-19,118.138948
2015-11-20,118.656145
2015-11-23,117.114508
2015-11-24,118.238406
2015-11-25,117.392995
2015-11-27,117.174181
2015-11-30,117.661542
2015-12-01,116.706717
2015-12-02,115.65244
2015-12-03,114.578267
2015-12-04,118.387598
2015-12-07,117.641646
2015-12-08,117.591921
2015-12-09,114.996006
2015-12-10,115.543033
2015-12-11,112.569172
2015-12-14,111.872953
2015-12-15,109.893688
2015-12-16,110.739099
2015-12-17,108.391842
2015-12-18,105.457759
2015-12-21,106.750746
2015-12-22,106.651287
2015-12-23,108.023837
2015-12-24,107.446965
2015-12-28,106.243496
2015-12-29,108.153132
2015-12-30,106.740798
2015-12-31,104.691918
2016-01-04,104.781429
2016-01-05,102.155677
2016-01-06,100.156523
2016-01-07,95.92946
2016-01-08,96.43671
2016-01-11,97.998236
2016-01-12,99.420519
2016-01-13,96.864389
2016-01-14,98.982891
2016-01-15,96.60579
2016-01-19,96.138333
2016-01-20,96.267629
2016-01-21,95.780276
2016-01-22,100.872638
2016-01-25,98.903329
2016-01-26,99.450356
2016-01-27,92.915814
2016-01-28,93.582196
2016-01-29,96.814656
2016-02-01,95.909571
2016-02-02,93.970098
2016-02-03,95.830001
2016-02-04,96.599998
2016-02-05,94.019997
2016-02-08,95.010002
2016-02-09,94.989998
2016-02-10,94.269997
2016-02-11,93.699997
2016-02-12,93.989998
2016-02-16,96.639999
2016-02-17,98.120003
2016-02-18,96.260002
2016-02-19,96.040001
2016-02-22,96.879997
2016-02-23,94.690002
2016-02-24,96.099998
2016-02-25,96.760002
2016-02-26,96.910004
2016-02-29,96.690002
"""


In [ ]:
MSFT_CSV = """Date,Close
2007-01-03,23.950705
2007-01-04,23.910599
2007-01-05,23.774242
2007-01-08,24.006852
2007-01-09,24.030914
2007-01-10,23.790284
2007-01-11,24.624469
2007-01-12,25.033538
2007-01-16,24.993434
2007-01-17,24.945309
2007-01-18,24.865098
2007-01-19,24.95333
2007-01-22,24.64051
2007-01-23,24.656552
2007-01-24,24.937287
2007-01-25,24.423944
2007-01-26,24.544259
2007-01-29,24.488112
2007-01-30,24.448006
2007-01-31,24.752805
2007-02-01,24.512174
2007-02-02,24.215398
2007-02-05,23.75018
2007-02-06,23.66997
2007-02-07,23.557676
2007-02-08,23.469445
2007-02-09,23.244856
2007-02-12,23.212773
2007-02-13,23.349603
2007-02-14,23.663506
2007-02-15,23.711798
2007-02-16,23.132284
2007-02-20,23.204724
2007-02-21,23.623262
2007-02-22,23.655457
2007-02-23,23.261065
2007-02-26,23.397895
2007-02-27,22.432038
2007-02-28,22.673502
2007-03-01,22.609112
2007-03-02,22.343501
2007-03-05,22.174475
2007-03-06,22.399842
2007-03-07,22.222769
2007-03-08,21.989353
2007-03-09,21.965207
2007-03-12,22.085939
2007-03-13,21.506424
2007-03-14,22.053743
2007-03-15,21.957158
2007-03-16,21.997402
2007-03-19,22.399842
2007-03-20,22.407891
2007-03-21,22.955211
2007-03-22,22.753991
2007-03-23,22.55277
2007-03-26,22.713746
2007-03-27,22.311305
2007-03-28,22.246915
2007-03-29,22.335452
2007-03-30,22.432038
2007-04-02,22.327403
2007-04-03,22.432038
2007-04-04,22.939113
2007-04-05,22.979356
2007-04-09,22.995454
2007-04-10,22.858625
2007-04-11,22.62521
2007-04-12,22.971309
2007-04-13,23.02765
2007-04-16,23.124235
2007-04-17,23.220822
2007-04-18,23.019601
2007-04-19,23.092041
2007-04-20,23.357652
2007-04-23,23.16448
2007-04-24,23.172529
2007-04-25,23.333505
2007-04-26,23.422042
2007-04-27,24.243021
2007-04-30,24.098142
2007-05-01,24.468387
2007-05-02,24.637413
2007-05-03,24.927169
2007-05-04,24.597168
2007-05-07,24.7179
2007-05-08,24.750096
2007-05-09,24.774243
2007-05-10,24.613266
2007-05-11,24.862778
2007-05-14,24.927169
2007-05-15,24.951394
2007-05-16,25.088667
2007-05-17,25.015993
2007-05-18,24.89487
2007-05-21,25.072517
2007-05-22,24.781823
2007-05-23,24.692998
2007-05-24,24.361928
2007-05-25,24.612249
2007-05-29,24.862572
2007-05-30,25.120968
2007-05-31,24.781823
2007-06-01,24.701073
2007-06-04,24.806046
2007-06-05,24.692998
2007-06-06,24.458827
2007-06-07,23.91781
2007-06-08,24.265029
2007-06-11,24.240805
2007-06-12,24.103532
2007-06-13,24.539575
2007-06-14,24.644549
2007-06-15,24.620324
2007-06-18,24.636474
2007-06-19,24.596099
2007-06-20,24.23273
2007-06-21,24.402302
2007-06-22,23.812836
2007-06-25,23.812836
2007-06-26,23.837061
2007-06-27,24.119682
2007-06-28,24.087382
2007-06-29,23.796686
2007-07-02,24.014708
2007-07-03,24.240805
2007-07-05,24.21658
2007-07-06,24.20043
2007-07-09,24.119682
2007-07-10,23.683638
2007-07-11,23.812836
2007-07-12,24.281179
2007-07-13,24.079307
2007-07-16,24.24888
2007-07-17,24.854497
2007-07-18,24.967545
2007-07-19,25.443963
2007-07-20,25.161342
2007-07-23,25.185567
2007-07-24,24.870645
2007-07-25,24.797971
2007-07-26,24.208505
2007-07-27,23.732087
2007-07-30,23.740162
2007-07-31,23.409091
2007-08-01,23.659412
2007-08-02,23.837061
2007-08-03,23.384866
2007-08-06,23.853211
2007-08-07,23.861285
2007-08-08,24.224655
2007-08-09,23.659412
2007-08-10,23.182994
2007-08-13,23.118395
2007-08-14,22.907714
2007-08-15,22.769959
2007-08-16,22.534966
2007-08-17,22.891507
2007-08-20,22.89961
2007-08-21,22.745649
2007-08-22,22.867197
2007-08-23,22.932022
2007-08-24,23.345285
2007-08-27,23.085983
2007-08-28,22.632205
2007-08-29,23.167015
2007-08-30,23.053571
2007-08-31,23.28046
2007-09-04,23.345285
2007-09-05,23.07788
2007-09-06,23.426317
2007-09-07,23.045468
2007-09-10,23.07788
2007-09-11,23.442524
2007-09-12,23.442524
2007-09-13,23.628897
2007-09-14,23.531659
2007-09-17,23.28046
2007-09-18,23.442524
2007-09-19,23.231841
2007-09-20,23.029261
2007-09-21,23.215634
2007-09-24,23.564071
2007-09-25,23.953024
2007-09-26,23.904405
2007-09-27,23.896302
2007-09-28,23.871992
2007-10-01,24.123192
2007-10-02,24.06647
2007-10-03,23.86389
2007-10-04,24.074572
2007-10-05,24.179914
2007-10-08,24.179914
2007-10-09,24.390597
2007-10-10,24.495938
2007-10-11,24.236636
2007-10-12,24.447319
2007-10-15,24.341978
2007-10-16,24.568867
2007-10-17,25.184709
2007-10-18,25.249535
2007-10-19,24.447319
2007-10-22,24.722828
2007-10-23,25.038851
2007-10-24,25.322463
2007-10-25,25.922099
2007-10-26,28.385468
2007-10-29,28.012722
2007-10-30,28.82304
2007-10-31,29.827837
2007-11-01,30.030417
2007-11-02,30.030417
2007-11-05,29.76301
2007-11-06,29.503708
2007-11-07,28.782525
2007-11-08,28.150477
2007-11-09,27.332054
2007-11-12,27.048443
2007-11-13,28.015908
2007-11-14,27.58502
2007-11-15,27.446809
2007-11-16,27.7151
2007-11-19,27.609409
2007-11-20,28.11347
2007-11-21,27.828919
2007-11-23,27.73136
2007-11-26,26.804543
2007-11-27,26.877713
2007-11-28,27.398031
2007-11-29,27.308601
2007-11-30,27.316729
2007-12-03,26.763891
2007-12-04,26.641943
2007-12-05,27.763881
2007-12-06,28.089078
2007-12-07,28.072818
2007-12-10,28.259807
2007-12-11,27.723228
2007-12-12,28.02404
2007-12-13,28.633788
2007-12-14,28.706958
2007-12-17,27.958999
2007-12-18,28.24355
2007-12-19,28.284199
2007-12-20,28.877687
2007-12-21,29.316707
2007-12-24,29.739466
2007-12-26,29.763855
2007-12-27,29.243537
2007-12-28,29.365485
2007-12-31,28.942725
2008-01-02,28.633788
2008-01-03,28.755736
2008-01-04,27.95087
2008-01-07,28.137859
2008-01-08,27.194782
2008-01-09,27.999648
2008-01-10,27.910221
2008-01-11,27.56876
2008-01-14,27.958999
2008-01-15,27.64193
2008-01-16,27.015921
2008-01-17,26.918362
2008-01-18,26.837061
2008-01-22,25.983413
2008-01-23,25.959024
2008-01-24,27.032181
2008-01-25,26.780151
2008-01-28,26.601293
2008-01-29,26.503732
2008-01-30,26.178534
2008-01-31,26.503732
2008-02-01,24.755788
2008-02-04,24.544408
2008-02-05,23.63385
2008-02-06,23.186702
2008-02-07,22.861503
2008-02-08,23.219221
2008-02-11,22.934671
2008-02-12,23.040362
2008-02-13,23.54442
2008-02-14,23.170441
2008-02-15,23.105401
2008-02-19,22.991141
2008-02-20,23.031948
2008-02-21,22.93401
2008-02-22,22.591224
2008-02-25,22.721809
2008-02-26,23.162533
2008-02-27,23.064595
2008-02-28,22.795263
2008-02-29,22.199469
2008-03-03,22.028075
2008-03-04,22.51777
2008-03-05,22.950333
2008-03-06,22.501446
2008-03-07,22.746294
2008-03-10,22.893201
2008-03-11,23.897075
2008-03-12,23.366572
2008-03-13,23.358412
2008-03-14,22.819747
2008-03-17,23.09724
2008-03-18,24.011337
2008-03-19,23.358412
2008-03-20,23.815459
2008-03-24,23.807298
2008-03-25,23.782812
2008-03-26,23.309441
2008-03-27,22.893201
2008-03-28,22.77894
2008-03-31,23.162533
2008-04-01,24.076629
2008-04-02,23.799136
2008-04-03,23.668551
2008-04-04,23.799136
2008-04-07,23.799136
2008-04-08,23.464512
2008-04-09,23.578773
2008-04-10,23.758329
2008-04-11,23.080918
2008-04-14,22.901363
2008-04-15,23.056433
2008-04-16,23.627744
2008-04-17,23.848105
2008-04-18,24.484708
2008-04-21,24.827494
2008-04-22,24.688747
2008-04-23,25.668136
2008-04-24,25.95379
2008-04-25,24.345961
2008-04-28,23.660389
2008-04-29,23.374734
2008-04-30,23.276796
2008-05-01,23.995013
2008-05-02,23.864428
2008-05-05,23.733843
2008-05-06,24.239861
2008-05-07,23.839943
2008-05-08,23.888914
2008-05-09,23.986851
2008-05-12,24.476546
2008-05-13,24.394629
2008-05-14,24.517503
2008-05-15,24.943468
2008-05-16,24.566652
2008-05-19,24.132496
2008-05-20,23.559084
2008-05-21,23.141312
2008-05-22,23.321527
2008-05-23,22.977479
2008-05-27,23.296953
2008-05-28,23.083971
2008-05-29,23.190461
2008-05-30,23.198653
2008-06-02,22.772688
2008-06-03,22.371299
2008-06-04,22.559708
2008-06-05,23.182269
2008-06-06,22.518749
2008-06-09,22.698964
2008-06-10,22.846413
2008-06-11,22.21566
2008-06-12,23.13312
2008-06-13,23.813024
2008-06-16,23.698342
2008-06-17,23.59185
2008-06-18,23.313335
2008-06-19,23.698342
2008-06-20,23.124928
2008-06-23,22.911946
2008-06-24,22.715347
2008-06-25,23.223228
2008-06-26,22.731731
2008-06-27,22.633431
2008-06-30,22.535132
2008-07-01,22.01087
2008-07-02,21.199898
2008-07-03,21.281815
2008-07-07,21.322774
2008-07-08,21.175324
2008-07-09,20.667444
2008-07-10,20.84766
2008-07-11,20.683827
2008-07-14,20.601911
2008-07-15,21.421072
2008-07-16,22.330342
2008-07-17,22.543324
2008-07-18,21.183516
2008-07-21,21.0033
2008-07-22,21.134365
2008-07-23,21.650438
2008-07-24,20.839468
2008-07-25,21.429264
2008-07-28,20.888618
2008-07-29,21.388307
2008-07-30,21.486605
2008-07-31,21.068833
2008-08-01,20.839468
2008-08-04,20.708403
2008-08-05,21.470222
2008-08-06,22.133743
2008-08-07,22.436832
2008-08-08,23.043012
2008-08-11,22.854605
2008-08-12,23.034821
2008-08-13,22.862797
2008-08-14,22.862797
2008-08-15,22.78088
2008-08-18,22.682582
2008-08-19,22.46875
2008-08-20,22.444078
2008-08-21,22.353611
2008-08-22,22.896414
2008-08-25,22.748376
2008-08-26,22.427629
2008-08-27,22.666133
2008-08-28,22.978657
2008-08-29,22.444078
2008-09-02,22.287817
2008-09-03,22.12333
2008-09-04,21.670995
2008-09-05,21.095294
2008-09-08,21.481837
2008-09-09,21.465388
2008-09-10,21.745014
2008-09-11,22.485199
2008-09-12,22.71548
2008-09-15,22.057536
2008-09-16,21.37492
2008-09-17,20.207072
2008-09-18,20.774548
2008-09-19,20.692305
2008-09-22,20.889687
2008-09-23,20.922585
2008-09-24,21.152864
2008-09-25,21.884827
2008-09-26,22.534545
2008-09-29,20.568941
2008-09-30,21.950621
2008-10-01,21.77791
2008-10-02,21.588752
2008-10-03,21.646322
2008-10-06,20.486697
2008-10-07,19.105017
2008-10-08,18.924083
2008-10-09,18.340158
2008-10-10,17.682216
2008-10-13,20.97193
2008-10-14,19.820531
2008-10-15,18.636233
2008-10-16,19.894549
2008-10-17,19.680718
2008-10-20,20.330436
2008-10-21,19.211934
2008-10-22,17.706889
2008-10-23,18.356607
2008-10-24,18.060532
2008-10-27,17.419039
2008-10-28,18.998102
2008-10-29,18.915859
2008-10-30,18.61156
2008-10-31,18.364832
2008-11-03,18.603337
2008-11-04,19.351747
2008-11-05,18.159224
2008-11-06,17.172309
2008-11-07,17.682216
2008-11-10,17.517729
2008-11-11,17.435488
2008-11-12,16.695301
2008-11-13,17.476609
2008-11-14,16.497918
2008-11-17,15.889321
2008-11-18,16.245362
2008-11-19,15.144122
2008-11-20,14.514842
2008-11-21,16.295042
2008-11-24,17.131322
2008-11-25,16.551721
2008-11-26,16.965721
2008-11-28,16.742161
2008-12-01,15.409082
2008-12-02,15.856201
2008-12-03,16.452362
2008-12-04,15.823082
2008-12-05,16.452362
2008-12-08,17.396282
2008-12-09,17.056802
2008-12-10,17.065082
2008-12-11,16.104602
2008-12-12,16.030082
2008-12-15,15.765122
2008-12-16,16.651082
2008-12-17,16.278481
2008-12-18,15.980401
2008-12-19,15.831362
2008-12-22,15.881042
2008-12-23,15.963842
2008-12-24,15.872761
2008-12-26,15.839641
2008-12-29,15.698881
2008-12-30,16.013521
2008-12-31,16.096322
2009-01-02,16.833241
2009-01-05,16.990562
2009-01-06,17.189282
2009-01-07,16.154282
2009-01-08,16.659362
2009-01-09,16.162562
2009-01-12,16.121161
2009-01-13,16.410961
2009-01-14,15.806521
2009-01-15,15.930721
2009-01-16,16.319881
2009-01-20,15.301441
2009-01-21,16.046641
2009-01-22,14.167082
2009-01-23,14.241602
2009-01-26,14.597641
2009-01-27,14.622481
2009-01-28,14.937122
2009-01-29,14.564521
2009-01-30,14.158801
2009-02-02,14.763241
2009-02-03,15.318001
2009-02-04,15.425641
2009-02-05,15.765122
2009-02-06,16.278481
2009-02-09,16.096322
2009-02-10,15.566401
2009-02-11,15.905881
2009-02-12,15.947282
2009-02-13,15.806521
2009-02-17,15.081222
2009-02-18,15.106233
2009-02-19,14.93116
2009-02-20,15.006191
2009-02-23,14.347585
2009-02-24,14.314239
2009-02-25,14.139166
2009-02-26,13.688981
2009-02-27,13.463888
2009-03-02,13.163764
2009-03-03,13.238795
2009-03-04,13.438879
2009-03-05,12.730253
2009-03-06,12.738589
2009-03-09,12.630211
2009-03-10,13.739001
2009-03-11,14.264219
2009-03-12,14.180851
2009-03-13,13.880727
2009-03-16,13.547256
2009-03-17,14.089146
2009-03-18,14.139166
2009-03-19,14.289228
2009-03-20,14.222534
2009-03-23,15.281305
2009-03-24,14.947834
2009-03-25,14.906149
2009-03-26,15.698143
2009-03-27,15.114569
2009-03-30,14.572679
2009-03-31,15.314653
2009-04-01,16.098308
2009-04-02,16.081636
2009-04-03,15.631449
2009-04-06,15.639786
2009-04-07,15.639786
2009-04-08,15.998268
2009-04-09,16.398432
2009-04-13,16.331738
2009-04-14,16.131656
2009-04-15,15.698143
2009-04-16,16.473463
2009-04-17,16.006605
2009-04-20,15.514735
2009-04-21,15.814858
2009-04-22,15.65646
2009-04-23,15.773174
2009-04-24,17.432192
2009-04-27,17.007016
2009-04-28,16.615189
2009-04-29,16.881965
2009-04-30,16.890302
2009-05-01,16.873628
2009-05-04,16.831945
2009-05-05,16.498474
2009-05-06,16.498474
2009-05-07,16.106645
2009-05-08,16.190013
2009-05-11,16.106645
2009-05-12,16.581841
2009-05-13,16.465127
2009-05-14,16.723566
2009-05-15,16.856954
2009-05-18,17.173753
2009-05-19,17.039517
2009-05-20,17.098244
2009-05-21,16.628421
2009-05-22,16.569693
2009-05-26,17.064686
2009-05-27,16.888502
2009-05-28,17.156974
2009-05-29,17.52612
2009-06-01,17.953996
2009-06-02,17.953996
2009-06-03,18.230857
2009-06-04,18.314754
2009-06-05,18.574835
2009-06-08,18.499327
2009-06-09,18.524497
2009-06-10,18.918813
2009-06-11,19.153726
2009-06-12,19.573212
2009-06-15,19.648719
2009-06-16,19.673889
2009-06-17,19.866852
2009-06-18,19.715837
2009-06-19,20.194051
2009-06-22,19.531264
2009-06-23,19.581602
2009-06-24,19.690667
2009-06-25,19.95914
2009-06-26,19.589992
2009-06-29,20.017868
2009-06-30,19.94236
2009-07-01,20.168883
2009-07-02,19.606772
2009-07-06,19.464146
2009-07-07,18.902035
2009-07-08,18.927203
2009-07-09,18.826527
2009-07-10,18.784578
2009-07-13,19.489314
2009-07-14,19.388639
2009-07-15,20.236
2009-07-16,20.504471
2009-07-17,20.378626
2009-07-20,20.579979
2009-07-21,20.83167
2009-07-22,20.8065
2009-07-23,21.444119
2009-07-24,19.673889
2009-07-27,19.388639
2009-07-28,19.690667
2009-07-29,19.967528
2009-07-30,19.975918
2009-07-31,19.732617
2009-08-03,19.992698
2009-08-04,19.94236
2009-08-05,19.975918
2009-08-06,19.682278
2009-08-07,19.766175
2009-08-10,19.648719
2009-08-11,19.405417
2009-08-12,19.741007
2009-08-13,19.816514
2009-08-14,19.875242
2009-08-17,19.506094
2009-08-18,19.89419
2009-08-19,19.953248
2009-08-20,19.970122
2009-08-21,20.594452
2009-08-24,20.7885
2009-08-25,20.7885
2009-08-26,20.712568
2009-08-27,20.830685
2009-08-28,20.822248
2009-08-31,20.796937
2009-09-01,20.248539
2009-09-02,20.130423
2009-09-03,20.341346
2009-09-04,20.771627
2009-09-08,20.940364
2009-09-09,20.906618
2009-09-10,21.092229
2009-09-11,20.974113
2009-09-14,21.092229
2009-09-15,21.260967
2009-09-16,21.260967
2009-09-17,21.345335
2009-09-18,21.311588
2009-09-21,21.345335
2009-09-22,21.74187
2009-09-23,21.691247
2009-09-24,21.885297
2009-09-25,21.556257
2009-09-28,21.792491
2009-09-29,21.724995
2009-09-30,21.699684
2009-10-01,20.990985
2009-10-02,21.05848
2009-10-05,20.7885
2009-10-06,21.185035
2009-10-07,21.176598
2009-10-08,21.6575
2009-10-09,21.556257
2009-10-12,21.699684
2009-10-13,21.775616
2009-10-14,21.902169
2009-10-15,22.534936
2009-10-16,22.357762
2009-10-19,22.239646
2009-10-20,22.248083
2009-10-21,22.425257
2009-10-22,22.433694
2009-10-23,23.64017
2009-10-26,24.197005
2009-10-27,24.121073
2009-10-28,23.64017
2009-10-29,23.808907
2009-10-30,23.3955
2009-11-02,23.522053
2009-11-03,23.226763
2009-11-04,23.673917
2009-11-05,24.019829
2009-11-06,24.062015
2009-11-09,24.458548
2009-11-10,24.475422
2009-11-11,24.568229
2009-11-12,24.770714
2009-11-13,24.998509
2009-11-16,24.922578
2009-11-17,25.422555
2009-11-18,25.515771
2009-11-19,25.236123
2009-11-20,25.100536
2009-11-23,25.37171
2009-11-24,25.346287
2009-11-25,25.244598
2009-11-27,24.761568
2009-11-30,24.922578
2009-12-01,25.431029
2009-12-02,25.236123
2009-12-03,25.278493
2009-12-04,25.405606
2009-12-07,25.244598
2009-12-08,25.058164
2009-12-09,25.176802
2009-12-10,25.312391
2009-12-11,25.295442
2009-12-14,25.515771
2009-12-15,25.439503
2009-12-16,25.507297
2009-12-17,25.083588
2009-12-18,25.727626
2009-12-21,25.863213
2009-12-22,26.117438
2009-12-23,26.20218
2009-12-24,26.269973
2009-12-28,26.414034
2009-12-29,26.600466
2009-12-30,26.236076
2009-12-31,25.829315
2010-01-04,26.227603
2010-01-05,26.236076
2010-01-06,26.075067
2010-01-07,25.803894
2010-01-08,25.981851
2010-01-11,25.651358
2010-01-12,25.481874
2010-01-13,25.719151
2010-01-14,26.236076
2010-01-15,26.151335
2010-01-19,26.354715
2010-01-20,25.922532
2010-01-21,25.431029
2010-01-22,24.541239
2010-01-25,24.84631
2010-01-26,24.998845
2010-01-27,25.142907
2010-01-28,24.710723
2010-01-29,23.880253
2010-02-01,24.075159
2010-02-02,24.117529
2010-02-03,24.261591
2010-02-04,23.592131
2010-02-05,23.744666
2010-02-08,23.49044
2010-02-09,23.736192
2010-02-10,23.719243
2010-02-11,23.829409
2010-02-12,23.668399
2010-02-16,24.136659
2010-02-17,24.34099
2010-02-18,24.664514
2010-02-19,24.494239
2010-02-22,24.460183
2010-02-23,24.119631
2010-02-24,24.375044
2010-02-25,24.349504
2010-02-26,24.4091
2010-03-01,24.707084
2010-03-02,24.230309
2010-03-03,24.230309
2010-03-04,24.375044
2010-03-05,24.34099
2010-03-08,24.375044
2010-03-09,24.519779
2010-03-10,24.664514
2010-03-11,24.843305
2010-03-12,24.919929
2010-03-15,24.936957
2010-03-16,25.005068
2010-03-17,25.226425
2010-03-18,25.209399
2010-03-19,25.192371
2010-03-22,25.200885
2010-03-23,25.439271
2010-03-24,25.243454
2010-03-25,25.549951
2010-03-26,25.251968
2010-03-29,25.192371
2010-03-30,25.34562
2010-03-31,24.936957
2010-04-01,24.826277
2010-04-05,24.919929
2010-04-06,24.962498
2010-04-07,24.98804
2010-04-08,25.473327
2010-04-09,25.830907
2010-04-12,25.813879
2010-04-13,25.924559
2010-04-14,26.23957
2010-04-15,26.28214
2010-04-16,26.111863
2010-04-19,26.426875
2010-04-20,26.699316
2010-04-21,26.673774
2010-04-22,26.724857
2010-04-23,26.358762
2010-04-26,26.486471
2010-04-27,26.265112
2010-04-28,26.316194
2010-04-29,26.392819
2010-04-30,26.001184
2010-05-03,26.273626
2010-05-04,25.652116
2010-05-05,25.41373
2010-05-06,24.673028
2010-05-07,24.017464
2010-05-10,24.638974
2010-05-11,24.58789
2010-05-12,25.064664
2010-05-13,24.894387
2010-05-14,24.63046
2010-05-17,24.638974
2010-05-18,24.459376
2010-05-19,24.151496
2010-05-20,23.185094
2010-05-21,22.954184
2010-05-24,22.466707
2010-05-25,22.295662
2010-05-26,21.389126
2010-05-27,22.235796
2010-05-28,22.064751
2010-06-01,22.141721
2010-06-02,22.629198
2010-06-03,22.971289
2010-06-04,22.0562
2010-06-07,21.628589
2010-06-08,21.474649
2010-06-09,21.200977
2010-06-10,21.380574
2010-06-11,21.945021
2010-06-14,21.808185
2010-06-15,22.731826
2010-06-16,22.509468
2010-06-17,22.55223
2010-06-18,22.612095
2010-06-21,22.193036
2010-06-22,22.039096
2010-06-23,21.645692
2010-06-24,21.380574
2010-06-25,20.978619
2010-06-28,20.790469
2010-06-29,19.935246
2010-06-30,19.67868
2010-07-01,19.806963
2010-07-02,19.901038
2010-07-06,20.37141
2010-07-07,20.781917
2010-07-08,20.875992
2010-07-09,20.756261
2010-07-12,21.235186
2010-07-13,21.491752
2010-07-14,21.756872
2010-07-15,21.816737
2010-07-16,21.286498
2010-07-19,21.577274
2010-07-20,21.79108
2010-07-21,21.483201
2010-07-22,22.098961
2010-07-23,22.073304
2010-07-26,22.321319
2010-07-27,22.372632
2010-07-28,22.193036
2010-07-29,22.261454
2010-07-30,22.073304
2010-08-02,22.51802
2010-08-03,22.372632
2010-08-04,22.004886
2010-08-05,21.697007
2010-08-06,21.850945
2010-08-09,21.90226
2010-08-10,21.440439
2010-08-11,21.260843
2010-08-12,20.94441
2010-08-13,20.867439
2010-08-16,20.952962
2010-08-17,21.245289
2010-08-18,21.339865
2010-08-19,21.013148
2010-08-20,20.832592
2010-08-23,20.875582
2010-08-24,20.669234
2010-08-25,20.720821
2010-08-26,20.48008
2010-08-27,20.574657
2010-08-30,20.325319
2010-08-31,20.179155
2010-09-01,20.548863
2010-09-02,20.583255
2010-09-03,20.88418
2010-09-07,20.60045
2010-09-08,20.574657
2010-09-09,20.64344
2010-09-10,20.505874
2010-09-13,21.589204
2010-09-14,21.520421
2010-09-15,21.597802
2010-09-16,21.778356
2010-09-17,21.683779
2010-09-20,21.864335
2010-09-21,21.623594
2010-09-22,21.159311
2010-09-23,21.00455
2010-09-24,21.305475
2010-09-27,21.262485
2010-09-28,21.219496
2010-09-29,21.064734
2010-09-30,21.056136
2010-10-01,20.96156
2010-10-04,20.557461
2010-10-05,20.935767
2010-10-06,21.00455
2010-10-07,21.090529
2010-10-08,21.124919
2010-10-11,21.142115
2010-10-12,21.348463
2010-10-13,21.786954
2010-10-14,21.692377
2010-10-15,21.958912
2010-10-18,22.199651
2010-10-19,21.580606
2010-10-20,21.76116
2010-10-21,21.855737
2010-10-22,21.821345
2010-10-25,21.657987
2010-10-26,22.268433
2010-10-27,22.397401
2010-10-28,22.595152
2010-10-29,22.930468
2010-11-01,23.171209
2010-11-02,23.549513
2010-11-03,23.239991
2010-11-04,23.334567
2010-11-05,23.08523
2010-11-08,23.050838
2010-11-09,23.171209
2010-11-10,23.162611
2010-11-11,22.939066
2010-11-12,22.586554
2010-11-15,22.52637
2010-11-16,22.327404
2010-11-17,22.119787
2010-11-18,22.353356
2010-11-19,22.223596
2010-11-22,22.258198
2010-11-23,21.730508
2010-11-24,21.946775
2010-11-26,21.842966
2010-11-29,21.89487
2010-11-30,21.851617
2010-12-01,22.52637
2010-12-02,23.261677
2010-12-03,23.374137
2010-12-06,23.218424
2010-12-07,23.244377
2010-12-08,23.5558
2010-12-09,23.42604
2010-12-10,23.650958
2010-12-13,23.573102
2010-12-14,23.893178
2010-12-15,24.092143
2010-12-16,24.213252
2010-12-17,24.135396
2010-12-20,24.057539
2010-12-21,24.282457
2010-12-22,24.386266
2010-12-23,24.481423
2010-12-27,24.282457
2010-12-28,24.230554
2010-12-29,24.19595
2010-12-30,24.092143
2010-12-31,24.144047
2011-01-03,24.204601
2011-01-04,24.299759
2011-01-05,24.221903
2011-01-06,24.931258
2011-01-07,24.740944
2011-01-10,24.412217
2011-01-11,24.317061
2011-01-12,24.69769
2011-01-13,24.386266
2011-01-14,24.481423
2011-01-18,24.792848
2011-01-19,24.628484
2011-01-20,24.524677
2011-01-21,24.239205
2011-01-24,24.550628
2011-01-25,24.611184
2011-01-26,24.896656
2011-01-27,24.974513
2011-01-28,24.005636
2011-01-31,23.988334
2011-02-01,24.213252
2011-02-02,24.169999
2011-02-03,23.919129
2011-02-04,24.022938
2011-02-07,24.394917
2011-02-08,24.464122
2011-02-09,24.19595
2011-02-10,23.789369
2011-02-11,23.573102
2011-02-14,23.5558
2011-02-15,23.46008
2011-02-16,23.512292
2011-02-17,23.677625
2011-02-18,23.547098
2011-02-22,23.138114
2011-02-23,23.138114
2011-02-24,23.294746
2011-02-25,23.103306
2011-02-28,23.129412
2011-03-01,22.763935
2011-03-02,22.694321
2011-03-03,22.798743
2011-03-04,22.581198
2011-03-07,22.381055
2011-03-08,22.54639
2011-03-09,22.528986
2011-03-10,22.1113
2011-03-11,22.346249
2011-03-14,22.354951
2011-03-15,22.093896
2011-03-16,21.571788
2011-03-17,21.563086
2011-03-18,21.580489
2011-03-21,22.041685
2011-03-22,22.015579
2011-03-23,22.224424
2011-03-24,22.459372
2011-03-25,22.294039
2011-03-28,22.1113
2011-03-29,22.180914
2011-03-30,22.285337
2011-03-31,22.093896
2011-04-01,22.172212
2011-04-04,22.233125
2011-04-05,22.433267
2011-04-06,22.755233
2011-04-07,22.798743
2011-04-08,22.685619
2011-04-11,22.607303
2011-04-12,22.311441
2011-04-13,22.302739
2011-04-14,22.120002
2011-04-15,22.076493
2011-04-18,21.82414
2011-04-19,21.885052
2011-04-20,22.415863
2011-04-21,22.20702
2011-04-25,22.285337
2011-04-26,22.790041
2011-04-27,22.955375
2011-04-28,23.242534
2011-04-29,22.555092
2011-05-02,22.328845
2011-05-03,22.459372
2011-05-04,22.676917
2011-05-05,22.441969
2011-05-06,22.511584
2011-05-09,22.476776
2011-05-10,22.337547
2011-05-11,22.067791
2011-05-12,22.032983
2011-05-13,21.780632
2011-05-16,21.380347
2011-05-17,21.476695
2011-05-18,21.625595
2011-05-19,21.651871
2011-05-20,21.450418
2011-05-23,21.170135
2011-05-24,21.152617
2011-05-25,21.187653
2011-05-26,21.608077
2011-05-27,21.686907
2011-05-31,21.905878
2011-06-01,21.397865
2011-06-02,21.213929
2011-06-03,20.942405
2011-06-06,21.029994
2011-06-07,21.073787
2011-06-08,20.968682
2011-06-09,20.986198
2011-06-10,20.767227
2011-06-13,21.056271
2011-06-14,21.213929
2011-06-15,20.793504
2011-06-16,21.021235
2011-06-17,21.248965
2011-06-20,21.4329
2011-06-21,21.686907
2011-06-22,21.590559
2011-06-23,21.573041
2011-06-24,21.283999
2011-06-27,22.072297
2011-06-28,22.597826
2011-06-29,22.440169
2011-06-30,22.773004
2011-07-01,22.790522
2011-07-05,22.799281
2011-07-06,23.062046
2011-07-07,23.447436
2011-07-08,23.578818
2011-07-11,23.324811
2011-07-12,23.245983
2011-07-13,23.324811
2011-07-14,23.184669
2011-07-15,23.456195
2011-07-18,23.289776
2011-07-19,24.121867
2011-07-20,23.701441
2011-07-21,23.736478
2011-07-22,24.113108
2011-07-25,24.445944
2011-07-26,24.594844
2011-07-27,23.937931
2011-07-28,24.279525
2011-07-29,23.999242
2011-08-01,23.885378
2011-08-02,23.473711
2011-08-03,23.578818
2011-08-04,22.720451
2011-08-05,22.492721
2011-08-08,21.441659
2011-08-09,22.405132
2011-08-10,21.196412
2011-08-11,22.063538
2011-08-12,21.984708
2011-08-15,22.343821
2011-08-16,22.343821
2011-08-17,22.255679
2011-08-18,21.74446
2011-08-19,21.197983
2011-08-22,21.136284
2011-08-23,21.78853
2011-08-24,21.947184
2011-08-25,21.656318
2011-08-26,22.255679
2011-08-29,22.775713
2011-08-30,23.119464
2011-08-31,23.445587
2011-09-01,23.101835
2011-09-02,22.740456
2011-09-06,22.484847
2011-09-07,22.916739
2011-09-08,23.110649
2011-09-09,22.687572
2011-09-12,22.819783
2011-09-13,22.951996
2011-09-14,23.357446
2011-09-15,23.789338
2011-09-16,23.903922
2011-09-19,23.983248
2011-09-20,23.780523
2011-09-21,22.907925
2011-09-22,22.08821
2011-09-23,22.08821
2011-09-26,22.423148
2011-09-27,22.625873
2011-09-28,22.546546
2011-09-29,22.431963
2011-09-30,21.93837
2011-10-03,21.621063
2011-10-04,22.335007
2011-10-05,22.819783
2011-10-06,23.21642
2011-10-07,23.137092
2011-10-10,23.745268
2011-10-11,23.798152
2011-10-12,23.762895
2011-10-13,23.956807
2011-10-14,24.036134
2011-10-17,23.780523
2011-10-18,24.07139
2011-10-19,23.912735
2011-10-20,23.833409
2011-10-21,23.939178
2011-10-24,23.965621
2011-10-25,23.630683
2011-10-26,23.436773
2011-10-27,24.018505
2011-10-28,23.780523
2011-10-31,23.472029
2011-11-01,22.907925
2011-11-02,22.925553
2011-11-03,23.383889
2011-11-04,23.137092
2011-11-07,23.621869
2011-11-08,23.939178
2011-11-09,23.093022
2011-11-10,23.163535
2011-11-11,23.718825
2011-11-14,23.586613
2011-11-15,23.746462
2011-11-16,23.151468
2011-11-17,22.680803
2011-11-18,22.467669
2011-11-21,22.201255
2011-11-22,22.014765
2011-11-23,21.730588
2011-11-25,21.579619
2011-11-28,22.085809
2011-11-29,22.059167
2011-11-30,22.716324
2011-12-01,22.44991
2011-12-02,22.396625
2011-12-05,22.822891
2011-12-06,22.787368
2011-12-07,22.734085
2011-12-08,22.556475
2011-12-09,22.822891
2011-12-12,22.654161
2011-12-13,22.876173
2011-12-14,22.725205
2011-12-15,22.698563
2011-12-16,23.089305
2011-12-19,22.671922
2011-12-20,23.115947
2011-12-21,22.876173
2011-12-22,22.920575
2011-12-23,23.115947
2011-12-27,23.124828
2011-12-28,22.929456
2011-12-29,23.107067
2011-12-30,23.053782
2012-01-03,23.773104
2012-01-04,24.332575
2012-01-05,24.58123
2012-01-06,24.963092
2012-01-09,24.634512
2012-01-10,24.723318
2012-01-11,24.616751
2012-01-12,24.865406
2012-01-13,25.087418
2012-01-17,25.096299
2012-01-18,25.069657
2012-01-19,24.971972
2012-01-20,26.383971
2012-01-23,26.401732
2012-01-24,26.055393
2012-01-25,26.250763
2012-01-26,26.197481
2012-01-27,25.957707
2012-01-30,26.295167
2012-01-31,26.224123
2012-02-01,26.54382
2012-02-02,26.597104
2012-02-03,26.854638
2012-02-06,26.819117
2012-02-07,26.952324
2012-02-08,27.227619
2012-02-09,27.325305
2012-02-10,27.085531
2012-02-13,27.156575
2012-02-14,27.040368
2012-02-15,26.861588
2012-02-16,27.970021
2012-02-17,27.934264
2012-02-21,28.104105
2012-02-22,27.952143
2012-02-23,28.041533
2012-02-24,28.13986
2012-02-27,28.023654
2012-02-28,28.488481
2012-02-29,28.372273
2012-03-01,28.863918
2012-03-02,28.6762
2012-03-05,28.425907
2012-03-06,28.211372
2012-03-07,28.461663
2012-03-08,28.613624
2012-03-09,28.595748
2012-03-12,28.640443
2012-03-13,29.203596
2012-03-14,29.292987
2012-03-15,29.364497
2012-03-16,29.141023
2012-03-19,28.783467
2012-03-20,28.595748
2012-03-21,28.524236
2012-03-22,28.604687
2012-03-23,28.613624
2012-03-26,29.132086
2012-03-27,29.069513
2012-03-28,28.774526
2012-03-29,28.711953
2012-03-30,28.837098
2012-04-02,28.863918
2012-04-03,28.551053
2012-04-04,27.898508
2012-04-05,28.175617
2012-04-09,27.80018
2012-04-10,27.237025
2012-04-11,27.129758
2012-04-12,27.692912
2012-04-13,27.540949
2012-04-16,27.782302
2012-04-17,28.104105
2012-04-18,27.835935
2012-04-19,27.719729
2012-04-20,28.980122
2012-04-23,28.711953
2012-04-24,28.533175
2012-04-25,28.783467
2012-04-26,28.703016
2012-04-27,28.586808
2012-04-30,28.622565
2012-05-01,28.613624
2012-05-02,28.425907
2012-05-03,28.390152
2012-05-04,27.692912
2012-05-07,27.397926
2012-05-08,27.263842
2012-05-09,27.496255
2012-05-10,27.478377
2012-05-11,27.853814
2012-05-14,27.424744
2012-05-15,27.181807
2012-05-16,26.902881
2012-05-17,26.740924
2012-05-18,26.336032
2012-05-21,26.767917
2012-05-22,26.776915
2012-05-23,26.19207
2012-05-24,26.156079
2012-05-25,26.147081
2012-05-29,26.596962
2012-05-30,26.399015
2012-05-31,26.264051
2012-06-01,25.598227
2012-06-04,25.688202
2012-06-05,25.652212
2012-06-06,26.408013
2012-06-07,26.30004
2012-06-08,26.67794
2012-06-11,26.003119
2012-06-12,26.354027
2012-06-13,26.210064
2012-06-14,26.399015
2012-06-15,27.010853
2012-06-18,26.848896
2012-06-19,27.622691
2012-06-20,27.829636
2012-06-21,27.118824
2012-06-22,27.622691
2012-06-25,26.875889
2012-06-26,27.010853
2012-06-27,27.145817
2012-06-28,26.911879
2012-06-29,27.523717
2012-07-02,27.496724
2012-07-03,27.676677
2012-07-05,27.622691
2012-07-06,27.163813
2012-07-09,26.992857
2012-07-10,26.758919
2012-07-11,26.363023
2012-07-12,25.760183
2012-07-13,26.444002
2012-07-16,26.488991
2012-07-17,26.686938
2012-07-18,27.397751
2012-07-19,27.595698
2012-07-20,27.10083
2012-07-23,26.345029
2012-07-24,26.228059
2012-07-25,25.940136
2012-07-26,26.237057
2012-07-27,26.776915
2012-07-30,26.668943
2012-07-31,26.515983
2012-08-01,26.461998
2012-08-02,26.264051
2012-08-03,26.767917
2012-08-06,26.94787
2012-08-07,27.226796
2012-08-08,27.289779
2012-08-09,27.442738
2012-08-10,27.370758
2012-08-13,27.343764
2012-08-14,27.289422
2012-08-15,27.352824
2012-08-16,27.878143
2012-08-17,27.986829
2012-08-20,27.841913
2012-08-21,27.896256
2012-08-22,27.66077
2012-08-23,27.407167
2012-08-24,27.678883
2012-08-27,27.796628
2012-08-28,27.742283
2012-08-29,27.760398
2012-08-30,27.46151
2012-08-31,27.914371
2012-09-04,27.52491
2012-09-05,27.52491
2012-09-06,28.394404
2012-09-07,28.032116
2012-09-10,27.823798
2012-09-11,27.8872
2012-09-12,27.878143
2012-09-13,28.023058
2012-09-14,28.267602
2012-09-17,28.267602
2012-09-18,28.240432
2012-09-19,28.122687
2012-09-20,28.484977
2012-09-21,28.249489
2012-09-24,27.878143
2012-09-25,27.52491
2012-09-26,27.325651
2012-09-27,27.316594
2012-09-28,26.954305
2012-10-01,26.70976
2012-10-02,26.863733
2012-10-03,27.044878
2012-10-04,27.198851
2012-10-05,27.03582
2012-10-08,26.97242
2012-10-09,26.519559
2012-10-10,26.247841
2012-10-11,26.22067
2012-10-12,26.447101
2012-10-15,26.727875
2012-10-16,26.70976
2012-10-17,26.800332
2012-10-18,26.718817
2012-10-19,25.939895
2012-10-22,25.360233
2012-10-23,25.405519
2012-10-24,25.269661
2012-10-25,25.251546
2012-10-26,25.550434
2012-10-31,25.849324
2012-11-01,26.736932
2012-11-02,26.718817
2012-11-05,26.83656
2012-11-06,27.044878
2012-11-07,26.338414
2012-11-08,26.093868
2012-11-09,26.111983
2012-11-12,25.559492
2012-11-13,24.737642
2012-11-14,24.509351
2012-11-15,24.344981
2012-11-16,24.217139
2012-11-19,24.408903
2012-11-20,24.390639
2012-11-21,24.6098
2012-11-23,25.294674
2012-11-26,25.011591
2012-11-27,24.728511
2012-11-28,24.984197
2012-11-29,24.6098
2012-11-30,24.308456
2012-12-03,24.134954
2012-12-04,24.080164
2012-12-05,24.354113
2012-12-06,24.408903
2012-12-07,24.162348
2012-12-10,24.600668
2012-12-11,24.94767
2012-12-12,24.874617
2012-12-13,24.755906
2012-12-14,24.481956
2012-12-17,24.746774
2012-12-18,25.166829
2012-12-19,24.938538
2012-12-20,25.27641
2012-12-21,25.066382
2012-12-24,24.710247
2012-12-26,24.527615
2012-12-27,24.61893
2012-12-28,24.244533
2012-12-31,24.390639
2013-01-02,25.221621
2013-01-03,24.883749
2013-01-04,24.418034
2013-01-07,24.372377
2013-01-08,24.244533
2013-01-09,24.381509
2013-01-10,24.162348
2013-01-11,24.500219
2013-01-14,24.555009
2013-01-15,24.847221
2013-01-16,24.691985
2013-01-17,24.883749
2013-01-18,24.883749
2013-01-22,24.792432
2013-01-23,25.212489
2013-01-24,25.230751
2013-01-25,25.459042
2013-01-28,25.486438
2013-01-29,25.577754
2013-01-30,25.431648
2013-01-31,25.066382
2013-02-01,25.504701
2013-02-04,25.057251
2013-02-05,25.11204
2013-02-06,24.965934
2013-02-07,24.911144
2013-02-08,25.157698
2013-02-11,25.44078
2013-02-12,25.459042
2013-02-13,25.596018
2013-02-14,25.60515
2013-02-15,25.577754
2013-02-19,25.82635
2013-02-20,25.660621
2013-02-21,25.310744
2013-02-22,25.559341
2013-02-25,25.200258
2013-02-26,25.200258
2013-02-27,25.605376
2013-02-28,25.596169
2013-03-01,25.734279
2013-03-04,25.918423
2013-03-05,26.102569
2013-03-06,25.86318
2013-03-07,25.909216
2013-03-08,25.780315
2013-03-11,25.660621
2013-03-12,25.697449
2013-03-13,25.706657
2013-03-14,25.909216
2013-03-15,25.817145
2013-03-18,25.872388
2013-03-19,25.946046
2013-03-20,26.074947
2013-03-21,25.881595
2013-03-22,26.010496
2013-03-25,25.927631
2013-03-26,25.927631
2013-03-27,26.120984
2013-03-28,26.341958
2013-04-01,26.341958
2013-04-02,26.516894
2013-04-03,26.29592
2013-04-04,26.33275
2013-04-05,26.424823
2013-04-08,26.323543
2013-04-09,27.262683
2013-04-10,27.87957
2013-04-11,26.645797
2013-04-12,26.507689
2013-04-15,26.415616
2013-04-16,26.673418
2013-04-17,26.544517
2013-04-18,26.507689
2013-04-19,27.409999
2013-04-22,28.385968
2013-04-23,28.174201
2013-04-24,29.242243
2013-04-25,29.407974
2013-04-26,29.269865
2013-04-29,30.02486
2013-04-30,30.476013
2013-05-01,30.12614
2013-05-02,30.531258
2013-05-03,30.835099
2013-05-06,31.074486
2013-05-07,30.669368
2013-05-08,30.374737
2013-05-09,30.070895
2013-05-10,30.098516
2013-05-13,30.411563
2013-05-14,31.088406
2013-05-15,31.385104
2013-05-16,31.598359
2013-05-17,32.33083
2013-05-20,32.525541
2013-05-21,32.312286
2013-05-22,32.089764
2013-05-23,31.663261
2013-05-24,31.774522
2013-05-28,32.469908
2013-05-29,32.340104
2013-05-30,32.479179
2013-05-31,32.358648
2013-06-03,32.998402
2013-06-04,32.442094
2013-06-05,32.247383
2013-06-06,32.414276
2013-06-07,33.072575
2013-06-10,32.887141
2013-06-11,32.303015
2013-06-12,32.451364
2013-06-13,32.191755
2013-06-14,31.895057
2013-06-17,32.451364
2013-06-18,32.43282
2013-06-19,32.07122
2013-06-20,31.051321
2013-06-21,30.84734
2013-06-24,31.264573
2013-06-25,31.218211
2013-06-26,31.848695
2013-06-27,32.099034
2013-06-28,32.024862
2013-07-01,31.857969
2013-07-02,31.46855
2013-07-03,31.533453
2013-07-05,31.71889
2013-07-08,31.830154
2013-07-09,31.848695
2013-07-10,32.173211
2013-07-11,33.091119
2013-07-12,33.072575
2013-07-15,33.536165
2013-07-16,33.628886
2013-07-17,33.137481
2013-07-18,32.859323
2013-07-19,29.113509
2013-07-22,29.679089
2013-07-23,29.502926
2013-07-24,29.632731
2013-07-25,29.104237
2013-07-26,29.317491
2013-07-29,29.243316
2013-07-30,29.530742
2013-07-31,29.52147
2013-08-01,29.363849
2013-08-02,29.567828
2013-08-05,29.391665
2013-08-06,29.280402
2013-08-07,29.725451
2013-08-08,30.49501
2013-08-09,30.318847
2013-08-12,30.476466
2013-08-13,30.093643
2013-08-14,30.205688
2013-08-15,29.68281
2013-08-16,29.692145
2013-08-19,29.309322
2013-08-20,29.524078
2013-08-21,29.514741
2013-08-22,30.243037
2013-08-23,32.446606
2013-08-26,31.886378
2013-08-27,31.055369
2013-08-28,30.831279
2013-08-29,31.326147
2013-08-30,31.186092
2013-09-03,29.766842
2013-09-04,29.131918
2013-09-05,29.159928
2013-09-06,29.085231
2013-09-09,29.561426
2013-09-10,30.243037
2013-09-11,30.56984
2013-09-12,30.523151
2013-09-13,30.840615
2013-09-16,30.625861
2013-09-17,30.747245
2013-09-18,31.111393
2013-09-19,31.410182
2013-09-20,30.616525
2013-09-23,30.56984
2013-09-24,30.299062
2013-09-25,30.355082
2013-09-26,30.59785
2013-09-27,31.064708
2013-09-30,31.074044
2013-10-01,31.354161
2013-10-02,31.671621
2013-10-03,31.6156
2013-10-04,31.634275
2013-10-07,31.092718
2013-10-08,30.82194
2013-10-09,30.877964
2013-10-10,31.522226
2013-10-11,31.867704
2013-10-14,32.166492
2013-10-15,32.203842
2013-10-16,32.343897
2013-10-17,32.605336
2013-10-18,32.642685
2013-10-21,32.670699
2013-10-22,32.287876
2013-10-23,31.522226
2013-10-24,31.484881
2013-10-25,33.361647
2013-10-28,33.212252
2013-10-29,33.165567
2013-10-30,33.184242
2013-10-31,33.062858
2013-11-01,33.174903
2013-11-04,33.557726
2013-11-05,34.211327
2013-11-06,35.64925
2013-11-07,35.014323
2013-11-08,35.275762
2013-11-11,35.098358
2013-11-12,34.883604
2013-11-13,35.630575
2013-11-14,35.499856
2013-11-15,35.331786
2013-11-18,34.734209
2013-11-19,34.564867
2013-11-20,34.884738
2013-11-21,35.185793
2013-11-22,35.345726
2013-11-25,35.411582
2013-11-26,35.13875
2013-11-27,35.373949
2013-11-29,35.872574
2013-12-02,36.173628
2013-12-03,36.041917
2013-12-04,36.634617
2013-12-05,35.750269
2013-12-06,36.088956
2013-12-09,36.418234
2013-12-10,35.853757
2013-12-11,35.383359
2013-12-12,35.016449
2013-12-13,34.517824
2013-12-16,34.705984
2013-12-17,34.357891
2013-12-18,34.41434
2013-12-19,34.103875
2013-12-20,34.621312
2013-12-23,34.451969
2013-12-24,34.884738
2013-12-26,35.223422
2013-12-27,35.082304
2013-12-30,35.082304
2013-12-31,35.195199
2014-01-02,34.96
2014-01-03,34.724801
2014-01-06,33.990981
2014-01-07,34.254402
2014-01-08,33.642883
2014-01-09,33.4265
2014-01-10,33.906309
2014-01-13,32.909063
2014-01-14,33.6617
2014-01-15,34.58368
2014-01-16,34.705984
2014-01-17,34.22618
2014-01-21,34.02861
2014-01-22,33.80282
2014-01-23,33.925125
2014-01-24,34.630722
2014-01-27,33.896899
2014-01-28,34.122691
2014-01-29,34.489602
2014-01-30,34.677762
2014-01-31,35.599742
2014-02-03,34.320258
2014-02-04,34.197953
2014-02-05,33.699332
2014-02-06,34.03802
2014-02-07,34.395523
2014-02-10,34.621312
2014-02-11,34.969406
2014-02-12,35.251648
2014-02-13,35.383359
2014-02-14,35.392765
2014-02-18,35.468592
2014-02-19,35.553898
2014-02-20,35.781384
2014-02-21,35.99939
2014-02-24,35.724512
2014-02-25,35.582336
2014-02-26,35.515987
2014-02-27,35.885649
2014-02-28,36.312182
2014-03-03,35.809819
2014-03-04,36.406966
2014-03-05,36.122611
2014-03-06,36.160526
2014-03-07,35.923563
2014-03-10,35.847734
2014-03-11,36.037305
2014-03-12,36.274267
2014-03-13,35.914083
2014-03-14,35.733992
2014-03-17,36.065739
2014-03-18,37.487516
2014-03-19,37.222119
2014-03-20,38.226842
2014-03-21,38.065706
2014-03-24,38.387975
2014-03-25,38.236319
2014-03-26,37.715002
2014-03-27,37.307425
2014-03-28,38.198404
2014-03-31,38.852424
2014-04-01,39.259997
2014-04-02,39.193647
2014-04-03,38.871378
2014-04-04,37.790828
2014-04-07,37.724479
2014-04-08,37.743436
2014-04-09,38.359541
2014-04-10,37.307425
2014-04-11,37.165246
2014-04-14,37.136812
2014-04-15,37.677087
2014-04-16,38.293192
2014-04-17,37.923527
2014-04-21,37.857177
2014-04-22,37.904573
2014-04-23,37.620214
2014-04-24,37.781351
2014-04-25,37.828743
2014-04-28,38.738679
2014-04-29,38.397452
2014-04-30,38.293192
2014-05-01,37.91405
2014-05-02,37.620214
2014-05-05,37.373775
2014-05-06,37.023071
2014-05-07,37.364294
2014-05-08,37.572823
2014-05-09,37.478039
2014-05-12,37.885615
2014-05-13,38.582426
2014-05-14,38.410613
2014-05-15,37.799705
2014-05-16,38.019252
2014-05-19,37.942887
2014-05-20,37.87607
2014-05-21,38.515609
2014-05-22,38.276974
2014-05-23,38.296065
2014-05-27,38.362883
2014-05-28,38.191066
2014-05-29,38.506065
2014-05-30,39.078786
2014-06-02,38.935608
2014-06-03,38.458339
2014-06-04,38.486974
2014-06-05,39.336512
2014-06-06,39.594238
2014-06-09,39.393786
2014-06-10,39.24106
2014-06-11,39.002425
2014-06-12,38.735156
2014-06-13,39.355603
2014-06-16,39.613329
2014-06-17,39.785146
2014-06-18,39.756511
2014-06-19,39.622873
2014-06-20,39.785146
2014-06-23,40.081054
2014-06-24,39.851963
2014-06-25,40.119233
2014-06-26,39.823328
2014-06-27,40.329232
2014-06-30,39.804237
2014-07-01,39.966507
2014-07-02,39.995146
2014-07-03,39.89969
2014-07-07,40.081054
2014-07-08,39.880598
2014-07-09,39.775599
2014-07-10,39.79469
2014-07-11,40.176507
2014-07-14,40.224233
2014-07-15,40.520141
2014-07-16,42.076039
2014-07-17,42.505578
2014-07-18,42.658304
2014-07-21,42.801486
2014-07-22,42.791942
2014-07-23,42.830121
2014-07-24,42.381491
2014-07-25,42.476943
2014-07-28,41.971039
2014-07-29,41.894674
2014-07-30,41.59877
2014-07-31,41.197862
2014-08-01,40.911501
2014-08-04,41.398314
2014-08-05,41.121501
2014-08-06,40.796958
2014-08-07,41.264679
2014-08-08,41.236044
2014-08-11,41.236044
2014-08-12,41.541496
2014-08-13,42.076039
2014-08-14,42.2574
2014-08-15,42.75376
2014-08-18,43.059212
2014-08-19,43.539461
2014-08-20,43.17447
2014-08-21,43.433806
2014-08-22,43.366571
2014-08-25,43.385778
2014-08-26,43.232098
2014-08-27,43.097629
2014-08-28,43.107236
2014-08-29,43.63551
2014-09-02,43.30894
2014-09-03,43.184074
2014-09-04,43.472223
2014-09-05,44.096549
2014-09-08,44.634431
2014-09-09,44.912973
2014-09-10,44.989815
2014-09-11,45.143494
2014-09-12,44.855345
2014-09-15,44.413516
2014-09-16,44.912973
2014-09-17,44.682455
2014-09-18,44.836135
2014-09-19,45.642955
2014-09-22,45.201126
2014-09-23,44.720876
2014-09-24,45.220336
2014-09-25,44.221415
2014-09-26,44.576799
2014-09-29,44.605613
2014-09-30,44.528775
2014-10-01,44.086946
2014-10-02,43.952473
2014-10-03,44.26944
2014-10-06,44.26944
2014-10-07,43.731558
2014-10-08,44.932183
2014-10-09,44.038918
2014-10-10,42.290809
2014-10-13,41.925821
2014-10-14,42.002659
2014-10-15,41.512806
2014-10-16,41.051767
2014-10-17,41.906611
2014-10-20,42.338837
2014-10-21,43.107236
2014-10-22,42.626986
2014-10-23,43.241705
2014-10-24,44.307861
2014-10-27,44.096549
2014-10-28,44.653641
2014-10-29,44.778504
2014-10-30,44.231019
2014-10-31,45.09547
2014-11-03,45.566113
2014-11-04,45.690979
2014-11-05,45.969525
2014-11-06,46.776345
2014-11-07,46.757135
2014-11-10,46.958839
2014-11-11,46.939628
2014-11-12,46.853183
2014-11-13,47.6504
2014-11-14,47.621586
2014-11-17,47.506323
2014-11-18,47.110038
2014-11-19,46.607427
2014-11-20,47.071374
2014-11-21,46.375451
2014-11-24,45.998494
2014-11-25,45.882508
2014-11-26,46.153144
2014-11-28,46.211138
2014-12-01,46.994048
2014-12-02,46.839399
2014-12-03,46.472109
2014-12-04,47.206692
2014-12-05,46.800736
2014-12-08,46.104816
2014-12-09,45.998494
2014-12-10,45.331571
2014-12-11,45.592538
2014-12-12,45.379898
2014-12-15,45.109259
2014-12-16,43.649758
2014-12-17,44.210364
2014-12-18,45.930836
2014-12-19,46.066153
2014-12-22,46.375451
2014-12-23,46.829735
2014-12-24,46.530101
2014-12-26,46.278797
2014-12-29,45.863177
2014-12-30,45.447557
2014-12-31,44.896619
2015-01-02,45.19625
2015-01-05,44.780633
2015-01-06,44.123373
2015-01-07,44.683975
2015-01-08,45.998494
2015-01-09,45.61187
2015-01-12,45.0416
2015-01-13,44.809629
2015-01-14,44.423004
2015-01-15,43.959056
2015-01-16,44.693643
2015-01-20,44.838624
2015-01-21,44.384341
2015-01-22,45.553879
2015-01-23,45.602206
2015-01-26,45.437889
2015-01-27,41.233363
2015-01-28,39.812522
2015-01-29,40.605099
2015-01-30,39.048944
2015-02-02,39.899512
2015-02-03,40.20881
2015-02-04,40.440786
2015-02-05,41.030387
2015-02-06,40.991724
2015-02-09,40.943397
2015-02-10,41.175368
2015-02-11,40.962728
2015-02-12,41.648983
2015-02-13,42.402898
2015-02-17,42.422369
2015-02-18,42.373694
2015-02-19,42.344492
2015-02-20,42.69493
2015-02-23,42.977227
2015-02-24,42.91882
2015-02-25,42.821477
2015-02-26,42.889618
2015-02-27,42.685193
2015-03-02,42.714399
2015-03-03,42.130335
2015-03-04,41.916181
2015-03-05,41.964852
2015-03-06,41.234775
2015-03-09,41.711757
2015-03-10,40.913539
2015-03-11,40.864868
2015-03-12,39.93037
2015-03-13,40.280807
2015-03-16,40.456026
2015-03-17,40.592307
2015-03-18,41.371055
2015-03-19,41.166634
2015-03-20,41.740962
2015-03-23,41.721493
2015-03-24,41.760431
2015-03-25,40.35868
2015-03-26,40.115321
2015-03-27,39.881698
2015-03-30,39.871962
2015-03-31,39.579932
2015-04-01,39.638339
2015-04-02,39.219761
2015-04-06,40.44629
2015-04-07,40.426821
2015-04-08,40.319742
2015-04-09,40.378149
2015-04-10,40.611776
2015-04-13,40.650711
2015-04-14,40.543636
2015-04-15,41.137429
2015-04-16,41.040087
2015-04-17,40.51443
2015-04-20,41.770164
2015-04-21,41.507336
2015-04-22,41.848041
2015-04-23,42.188742
2015-04-24,46.598409
2015-04-27,46.754158
2015-04-28,47.854143
2015-04-29,47.756801
2015-04-30,47.347955
2015-05-01,47.367425
2015-05-04,46.958583
2015-05-05,46.33558
2015-05-06,45.050644
2015-05-07,45.45949
2015-05-08,46.481597
2015-05-11,46.11169
2015-05-12,46.092221
2015-05-13,46.364786
2015-05-14,47.425832
2015-05-15,47.016987
2015-05-18,46.734689
2015-05-19,46.617121
2015-05-20,46.617121
2015-05-21,46.460355
2015-05-22,45.950882
2015-05-26,45.647154
2015-05-27,46.646513
2015-05-28,46.489751
2015-05-29,45.911691
2015-06-01,46.274202
2015-06-02,45.970474
2015-06-03,45.901891
2015-06-04,45.421809
2015-06-05,45.20626
2015-06-08,44.804558
2015-06-09,44.726178
2015-06-10,45.66675
2015-06-11,45.500188
2015-06-12,45.039702
2015-06-15,44.559617
2015-06-16,44.902536
2015-06-17,45.039702
2015-06-18,45.774524
2015-06-19,45.167069
2015-06-22,45.294439
2015-06-23,44.980915
2015-06-24,44.716379
2015-06-25,44.726178
2015-06-26,44.344068
2015-06-29,43.472079
2015-06-30,43.256534
2015-07-01,43.550462
2015-07-02,43.501475
2015-07-06,43.491675
2015-07-07,43.403496
2015-07-08,43.344713
2015-07-09,43.619045
2015-07-10,43.707224
2015-07-13,44.618404
2015-07-14,44.696783
2015-07-15,44.833949
2015-07-16,45.715737
2015-07-17,45.676546
2015-07-20,45.970474
2015-07-21,46.323189
2015-07-22,44.618404
2015-07-23,45.176868
2015-07-24,45.010307
2015-07-27,44.432247
2015-07-28,44.422451
2015-07-29,45.353226
2015-07-30,45.931286
2015-07-31,45.754929
2015-08-03,45.862703
2015-08-04,46.57793
2015-08-05,46.617121
2015-08-06,45.676546
2015-08-07,45.79412
2015-08-10,46.37218
2015-08-11,45.470797
2015-08-12,45.79412
2015-08-13,45.78432
2015-08-14,46.048857
2015-08-17,46.362381
2015-08-18,46.6188
2015-08-19,45.967893
2015-08-20,45.030979
2015-08-21,42.476659
2015-08-24,41.105809
2015-08-25,39.912479
2015-08-26,42.121618
2015-08-27,43.295227
2015-08-28,43.324813
2015-08-31,42.920461
2015-09-01,41.24388
2015-09-02,42.762665
2015-09-03,42.900736
2015-09-04,42.022997
2015-09-08,43.285363
2015-09-09,42.476659
2015-09-10,42.69363
2015-09-11,42.881011
2015-09-14,42.447074
2015-09-15,43.374123
2015-09-16,43.689714
2015-09-17,43.640404
2015-09-18,42.881011
2015-09-21,43.502333
2015-09-22,43.295227
2015-09-23,43.265638
2015-09-24,43.305088
2015-09-25,43.334673
2015-09-28,42.69363
2015-09-29,42.841561
2015-09-30,43.650264
2015-10-01,43.995445
2015-10-02,44.942219
2015-10-05,45.987618
2015-10-06,46.105963
2015-10-07,46.155274
2015-10-08,46.796321
2015-10-09,46.461005
2015-10-12,46.352519
2015-10-13,46.244034
2015-10-14,46.036928
2015-10-15,46.36238
2015-10-16,46.855492
2015-10-19,46.963977
2015-10-20,47.111912
2015-10-21,46.549765
2015-10-22,47.368329
2015-10-23,52.141652
2015-10-26,53.502642
2015-10-27,52.950355
2015-10-28,53.236361
2015-10-29,52.624903
2015-10-30,51.914821
2015-11-02,52.506558
2015-11-03,53.404021
2015-11-04,53.650577
2015-11-05,53.630852
2015-11-06,54.16341
2015-11-09,53.413882
2015-11-10,52.772835
2015-11-11,52.910909
2015-11-12,52.585454
2015-11-13,52.112067
2015-11-16,53.029255
2015-11-17,52.592393
2015-11-18,53.466117
2015-11-19,53.555476
2015-11-20,53.803694
2015-11-23,53.803694
2015-11-24,53.863267
2015-11-25,53.307258
2015-11-27,53.545549
2015-11-30,53.962553
2015-12-01,54.826354
2015-12-02,54.816423
2015-12-03,53.813624
2015-12-04,55.511433
2015-12-07,55.412148
2015-12-08,55.39229
2015-12-09,54.588063
2015-12-10,54.875996
2015-12-11,53.674623
2015-12-14,54.746922
2015-12-15,54.806496
2015-12-16,55.729866
2015-12-17,55.302931
2015-12-18,53.744124
2015-12-21,54.439134
2015-12-22,54.955424
2015-12-23,55.422075
2015-12-24,55.273143
2015-12-28,55.551149
2015-12-29,56.14687
2015-12-30,55.908583
2015-12-31,55.084498
2016-01-04,54.409346
2016-01-05,54.657563
2016-01-06,53.664692
2016-01-07,51.798093
2016-01-08,51.956956
2016-01-11,51.927167
2016-01-12,52.403745
2016-01-13,51.271873
2016-01-14,52.731395
2016-01-15,50.626508
2016-01-19,50.199574
2016-01-20,50.427933
2016-01-21,50.120142
2016-01-22,51.91724
2016-01-25,51.420805
2016-01-26,51.798093
2016-01-27,50.854868
2016-01-28,51.68888
2016-01-29,54.697279
2016-02-01,54.319987
2016-02-02,52.622178
2016-02-03,51.788166
2016-02-04,51.629307
2016-02-05,49.802423
2016-02-08,49.05777
2016-02-09,48.928696
2016-02-10,49.355631
2016-02-11,49.335773
2016-02-12,50.14
2016-02-16,51.09
2016-02-17,52.419998
2016-02-18,52.189999
2016-02-19,51.82
2016-02-22,52.650002
2016-02-23,51.18
2016-02-24,51.360001
2016-02-25,52.099998
2016-02-26,51.299999
2016-02-29,50.880001
"""


In [ ]:
IBM_CSV = """Date,Close
2007-01-03,80.517962
2007-01-04,81.378851
2007-01-05,80.642129
2007-01-08,81.867244
2007-01-09,82.835742
2007-01-10,81.858964
2007-01-11,81.660299
2007-01-12,82.231462
2007-01-16,83.456576
2007-01-17,82.794351
2007-01-18,82.322518
2007-01-19,79.607407
2007-01-22,80.38552
2007-01-23,80.360688
2007-01-24,80.625577
2007-01-25,80.716633
2007-01-26,80.666962
2007-01-29,81.569243
2007-01-30,82.2563
2007-01-31,82.074188
2007-02-01,81.95002
2007-02-02,82.090741
2007-02-05,83.092352
2007-02-06,82.65363
2007-02-07,82.645331
2007-02-08,82.711754
2007-02-09,81.823363
2007-02-12,81.84827
2007-02-13,81.60749
2007-02-14,82.363035
2007-02-15,82.13056
2007-02-16,82.188678
2007-02-20,82.487577
2007-02-21,82.271704
2007-02-22,81.781847
2007-02-23,81.14254
2007-02-26,80.461716
2007-02-27,78.012409
2007-02-28,77.165533
2007-03-01,76.609246
2007-03-02,75.471777
2007-03-05,76.227321
2007-03-06,77.879568
2007-03-07,77.995806
2007-03-08,77.215348
2007-03-09,77.447823
2007-03-12,78.136951
2007-03-13,76.974568
2007-03-14,77.846357
2007-03-15,77.588968
2007-03-16,77.422916
2007-03-19,78.037316
2007-03-20,78.460756
2007-03-21,79.174791
2007-03-22,79.033647
2007-03-23,78.9008
2007-03-26,78.875893
2007-03-27,78.651722
2007-03-28,78.261493
2007-03-29,78.518875
2007-03-30,78.261493
2007-04-02,79.050249
2007-04-03,79.789191
2007-04-04,79.880522
2007-04-05,80.137904
2007-04-09,80.220937
2007-04-10,80.08809
2007-04-11,79.008739
2007-04-12,79.432174
2007-04-13,78.817774
2007-04-16,79.855615
2007-04-17,80.636073
2007-04-18,78.709841
2007-04-19,78.2864
2007-04-20,78.52718
2007-04-23,79.050249
2007-04-24,81.773542
2007-04-25,84.239453
2007-04-26,83.774502
2007-04-27,83.998673
2007-04-30,84.862157
2007-05-01,85.659218
2007-05-02,84.870462
2007-05-03,85.352021
2007-05-04,85.484862
2007-05-07,85.65092
2007-05-08,86.092674
2007-05-09,87.001191
2007-05-10,87.251245
2007-05-11,88.334803
2007-05-14,87.993063
2007-05-15,87.376272
2007-05-16,88.243117
2007-05-17,87.77635
2007-05-18,90.010143
2007-05-21,89.218316
2007-05-22,88.934921
2007-05-23,88.0014
2007-05-24,86.642784
2007-05-25,87.667997
2007-05-29,88.276458
2007-05-30,89.12663
2007-05-31,88.851572
2007-06-01,88.801564
2007-06-04,88.543179
2007-06-05,88.218107
2007-06-06,85.359193
2007-06-07,84.850754
2007-06-08,85.909302
2007-06-11,86.034329
2007-06-12,85.300841
2007-06-13,85.95098
2007-06-14,86.559435
2007-06-15,87.592978
2007-06-18,87.793024
2007-06-19,88.768223
2007-06-20,88.35147
2007-06-21,88.851572
2007-06-22,87.051206
2007-06-25,87.601315
2007-06-26,87.793024
2007-06-27,87.876373
2007-06-28,88.309793
2007-06-29,87.726342
2007-07-02,87.526303
2007-07-03,88.834905
2007-07-05,90.060157
2007-07-06,90.876988
2007-07-09,90.82698
2007-07-10,90.543585
2007-07-11,90.935333
2007-07-12,91.085364
2007-07-13,90.518581
2007-07-16,91.4021
2007-07-17,92.327284
2007-07-18,92.585675
2007-07-19,96.569825
2007-07-20,95.694643
2007-07-23,97.003244
2007-07-24,96.828209
2007-07-25,98.436873
2007-07-26,97.128271
2007-07-27,96.369785
2007-07-30,95.452925
2007-07-31,92.227267
2007-08-01,93.385838
2007-08-02,94.37771
2007-08-03,93.260811
2007-08-06,94.92782
2007-08-07,94.627758
2007-08-08,94.502297
2007-08-09,92.620281
2007-08-10,94.2179
2007-08-13,94.276451
2007-08-14,93.724397
2007-08-15,93.038507
2007-08-16,91.75037
2007-08-17,92.762476
2007-08-20,91.357237
2007-08-21,91.206676
2007-08-22,92.009668
2007-08-23,93.222521
2007-08-24,94.71977
2007-08-27,94.887064
2007-08-28,93.682571
2007-08-29,95.832252
2007-08-30,96.501415
2007-08-31,97.605531
2007-09-04,98.860208
2007-09-05,98.600904
2007-09-06,98.383431
2007-09-07,96.651977
2007-09-10,96.86109
2007-09-11,98.157586
2007-09-12,97.028378
2007-09-13,96.986552
2007-09-14,96.300662
2007-09-17,95.790426
2007-09-18,97.55534
2007-09-19,97.588799
2007-09-20,97.747726
2007-09-21,97.680809
2007-09-24,97.23749
2007-09-25,97.45497
2007-09-26,98.115767
2007-09-27,98.458709
2007-09-28,98.533993
2007-10-01,99.562825
2007-10-02,99.002404
2007-10-03,97.362959
2007-10-04,96.76908
2007-10-05,97.279316
2007-10-08,98.508894
2007-10-09,98.952219
2007-10-10,99.219883
2007-10-11,98.743106
2007-10-12,98.542353
2007-10-15,98.726373
2007-10-16,100.039602
2007-10-17,96.844357
2007-10-18,96.024638
2007-10-19,93.916777
2007-10-22,94.828512
2007-10-23,95.924262
2007-10-24,94.477198
2007-10-25,94.360095
2007-10-26,95.129635
2007-10-29,96.024638
2007-10-30,95.455851
2007-10-31,97.128754
2007-11-01,95.062718
2007-11-02,95.848978
2007-11-05,94.853605
2007-11-06,94.661218
2007-11-07,93.242605
2007-11-08,89.070694
2007-11-09,84.151702
2007-11-12,85.159001
2007-11-13,88.36558
2007-11-14,86.829449
2007-11-15,86.963752
2007-11-16,87.962662
2007-11-19,85.805357
2007-11-20,86.812657
2007-11-21,85.805357
2007-11-23,87.341494
2007-11-26,85.595502
2007-11-27,87.156821
2007-11-28,90.128364
2007-11-29,90.237486
2007-11-30,88.290035
2007-12-03,88.835658
2007-12-04,89.507189
2007-12-05,90.791505
2007-12-06,92.084204
2007-12-07,91.379095
2007-12-10,91.823986
2007-12-11,89.80938
2007-12-12,91.051723
2007-12-13,90.80829
2007-12-14,88.785289
2007-12-17,87.744412
2007-12-18,89.238576
2007-12-19,89.935294
2007-12-20,91.362303
2007-12-21,93.217424
2007-12-24,93.721074
2007-12-26,93.645523
2007-12-27,92.000263
2007-12-28,92.411576
2007-12-31,90.741135
2008-01-02,87.878722
2008-01-03,88.054999
2008-01-04,84.890388
2008-01-07,83.983821
2008-01-08,81.918845
2008-01-09,82.523228
2008-01-10,83.874692
2008-01-11,81.986001
2008-01-14,86.401343
2008-01-15,85.477984
2008-01-16,85.310097
2008-01-17,84.865206
2008-01-18,86.795871
2008-01-22,84.965939
2008-01-23,89.062298
2008-01-24,89.742232
2008-01-25,87.736016
2008-01-28,88.122154
2008-01-29,89.062298
2008-01-30,88.684563
2008-01-31,89.910113
2008-02-01,91.563768
2008-02-04,90.598436
2008-02-05,88.155725
2008-02-06,87.287813
2008-02-07,86.234528
2008-02-08,87.018172
2008-02-11,88.593889
2008-02-12,89.765141
2008-02-13,91.357707
2008-02-14,89.428089
2008-02-15,89.453373
2008-02-19,88.475921
2008-02-20,90.877409
2008-02-21,90.102193
2008-02-22,91.062789
2008-02-25,92.756472
2008-02-26,96.379768
2008-02-27,98.132436
2008-02-28,97.104429
2008-02-29,95.941604
2008-03-03,96.253379
2008-03-04,97.500465
2008-03-05,97.230824
2008-03-06,94.81248
2008-03-07,96.009016
2008-03-10,96.068
2008-03-11,98.157713
2008-03-12,98.646439
2008-03-13,97.668994
2008-03-14,97.096007
2008-03-17,97.365648
2008-03-18,99.775564
2008-03-19,98.5369
2008-03-20,99.708152
2008-03-24,100.323267
2008-03-25,99.404805
2008-03-26,98.511622
2008-03-27,97.340363
2008-03-28,96.539869
2008-03-31,97.020167
2008-04-01,98.157713
2008-04-02,96.742098
2008-04-03,97.761677
2008-04-04,97.542599
2008-04-07,98.00604
2008-04-08,97.972334
2008-04-09,98.393648
2008-04-10,100.087332
2008-04-11,97.744828
2008-04-14,98.82339
2008-04-15,98.730701
2008-04-16,101.511375
2008-04-17,103.710634
2008-04-18,104.822903
2008-04-21,104.780769
2008-04-22,104.207781
2008-04-23,104.148798
2008-04-24,104.645951
2008-04-25,103.710634
2008-04-28,102.539382
2008-04-29,103.516827
2008-04-30,101.705176
2008-05-01,104.157226
2008-05-02,103.794896
2008-05-05,102.825872
2008-05-06,103.491549
2008-05-07,105.031402
2008-05-08,105.691337
2008-05-09,104.963715
2008-05-12,105.96208
2008-05-13,107.09582
2008-05-14,107.891124
2008-05-15,108.68644
2008-05-16,108.144948
2008-05-19,107.01967
2008-05-20,105.911318
2008-05-21,104.591447
2008-05-22,105.5052
2008-05-23,105.082164
2008-05-27,107.721912
2008-05-28,109.600187
2008-05-29,109.74403
2008-05-30,109.507118
2008-06-02,107.755756
2008-06-03,108.161867
2008-06-04,107.916511
2008-06-05,108.694896
2008-06-06,105.708262
2008-06-09,106.486647
2008-06-10,106.554335
2008-06-11,104.278398
2008-06-12,104.78604
2008-06-13,106.732009
2008-06-16,107.205807
2008-06-17,105.843631
2008-06-18,105.048327
2008-06-19,105.775943
2008-06-20,103.8469
2008-06-23,104.456073
2008-06-24,104.456073
2008-06-25,105.403676
2008-06-26,102.484723
2008-06-27,101.57097
2008-06-30,100.284936
2008-07-01,100.911028
2008-07-02,100.767197
2008-07-03,101.139471
2008-07-07,102.797772
2008-07-08,104.811421
2008-07-09,101.867094
2008-07-10,104.219173
2008-07-11,103.322339
2008-07-14,102.831616
2008-07-15,104.236092
2008-07-16,106.554335
2008-07-17,107.045052
2008-07-18,109.896317
2008-07-21,108.855652
2008-07-22,109.989386
2008-07-23,109.583275
2008-07-24,109.989386
2008-07-25,108.745658
2008-07-28,106.816615
2008-07-29,108.00958
2008-07-30,109.024864
2008-07-31,108.280323
2008-08-01,107.146583
2008-08-04,107.924967
2008-08-05,109.03332
2008-08-06,109.704325
2008-08-07,109.610894
2008-08-08,109.407041
2008-08-11,107.529939
2008-08-12,106.357814
2008-08-13,106.850449
2008-08-14,107.818728
2008-08-15,107.326093
2008-08-18,105.822708
2008-08-19,104.098493
2008-08-20,104.056029
2008-08-21,104.463722
2008-08-22,106.111497
2008-08-25,104.353306
2008-08-26,104.047533
2008-08-27,104.794974
2008-08-28,105.814219
2008-08-29,103.393523
2008-09-02,100.573623
2008-09-03,100.514161
2008-09-04,97.677276
2008-09-05,97.108201
2008-09-08,99.622329
2008-09-09,97.711252
2008-09-10,100.259354
2008-09-11,101.244618
2008-09-12,101.049267
2008-09-15,97.838658
2008-09-16,98.569115
2008-09-17,94.679009
2008-09-18,97.779203
2008-09-19,100.94734
2008-09-22,98.70501
2008-09-23,97.983049
2008-09-24,98.917352
2008-09-25,102.017545
2008-09-26,101.431479
2008-09-29,97.218617
2008-09-30,99.342036
2008-10-01,93.540853
2008-10-02,88.962763
2008-10-03,87.858589
2008-10-06,85.463372
2008-10-07,81.242014
2008-10-08,76.91024
2008-10-09,75.593718
2008-10-10,74.532009
2008-10-13,78.320187
2008-10-14,79.500808
2008-10-15,74.990668
2008-10-16,77.734122
2008-10-17,77.105591
2008-10-20,78.575
2008-10-21,75.474807
2008-10-22,71.007132
2008-10-23,71.644157
2008-10-24,69.7076
2008-10-27,67.660628
2008-10-28,74.132805
2008-10-29,74.914221
2008-10-30,77.029151
2008-10-31,78.965709
2008-11-03,78.719391
2008-11-04,79.330937
2008-11-05,76.392126
2008-11-06,72.72797
2008-11-07,73.684576
2008-11-10,71.634703
2008-11-11,70.669548
2008-11-12,68.1072
2008-11-13,71.925099
2008-11-14,68.611132
2008-11-17,66.176903
2008-11-18,68.397603
2008-11-19,64.887186
2008-11-20,61.274273
2008-11-21,63.956196
2008-11-24,68.235319
2008-11-25,68.884449
2008-11-26,69.755644
2008-11-28,69.695856
2008-12-01,65.681514
2008-12-02,68.192611
2008-12-03,68.901528
2008-12-04,66.142738
2008-12-05,68.833198
2008-12-08,72.480276
2008-12-09,70.626846
2008-12-10,70.772044
2008-12-11,68.824661
2008-12-12,70.208325
2008-12-15,70.69517
2008-12-16,73.795615
2008-12-17,73.317306
2008-12-18,71.745736
2008-12-19,71.335757
2008-12-22,70.028961
2008-12-23,68.841741
2008-12-24,68.77341
2008-12-26,69.465248
2008-12-29,69.396917
2008-12-30,71.361386
2008-12-31,71.882398
2009-01-02,74.624109
2009-01-05,74.154342
2009-01-06,76.212765
2009-01-07,74.982836
2009-01-08,74.461825
2009-01-09,72.343614
2009-01-12,73.206273
2009-01-13,72.890248
2009-01-14,71.053904
2009-01-15,71.848232
2009-01-16,72.531521
2009-01-20,70.020425
2009-01-21,78.083274
2009-01-22,76.930219
2009-01-23,76.43483
2009-01-26,78.237015
2009-01-27,78.288267
2009-01-28,80.98727
2009-01-29,79.014264
2009-01-30,78.279724
2009-02-02,77.664759
2009-02-03,79.842758
2009-02-04,79.287581
2009-02-05,78.928854
2009-02-06,82.561415
2009-02-09,83.145374
2009-02-10,80.096765
2009-02-11,81.719832
2009-02-12,81.64254
2009-02-13,80.586259
2009-02-17,77.863984
2009-02-18,78.585348
2009-02-19,76.369739
2009-02-20,76.249513
2009-02-23,72.453785
2009-02-24,74.197071
2009-02-25,73.76769
2009-02-26,76.40409
2009-02-27,79.031901
2009-03-02,76.472793
2009-03-03,75.373572
2009-03-04,76.850644
2009-03-05,75.124536
2009-03-06,73.690398
2009-03-09,71.689487
2009-03-10,74.927018
2009-03-11,76.103525
2009-03-12,77.632121
2009-03-13,77.597769
2009-03-16,78.336306
2009-03-17,79.787616
2009-03-18,78.963199
2009-03-19,79.572926
2009-03-20,79.44411
2009-03-23,84.768434
2009-03-24,84.416345
2009-03-25,84.115773
2009-03-26,84.828547
2009-03-27,80.85248
2009-03-30,81.170218
2009-03-31,83.205487
2009-04-01,83.823797
2009-04-02,86.580423
2009-04-03,87.782692
2009-04-06,87.215906
2009-04-07,84.802785
2009-04-08,86.898168
2009-04-09,87.336132
2009-04-13,85.833298
2009-04-14,85.249339
2009-04-15,84.88866
2009-04-16,87.104269
2009-04-17,86.966864
2009-04-20,86.245506
2009-04-21,87.859977
2009-04-22,88.066085
2009-04-23,87.095679
2009-04-24,85.944941
2009-04-27,85.833298
2009-04-28,87.542239
2009-04-29,89.345639
2009-04-30,88.632865
2009-05-01,89.835133
2009-05-04,91.19198
2009-05-05,90.899997
2009-05-06,90.312995
2009-05-07,88.560596
2009-05-08,87.611025
2009-05-11,88.828207
2009-05-12,89.725986
2009-05-13,88.275729
2009-05-14,87.2312
2009-05-15,87.507439
2009-05-18,90.278464
2009-05-19,91.081285
2009-05-20,89.820944
2009-05-21,88.759146
2009-05-22,87.956325
2009-05-26,90.658289
2009-05-27,88.854104
2009-05-28,90.373422
2009-05-29,91.745983
2009-06-01,93.550174
2009-06-02,92.220772
2009-06-03,91.927264
2009-06-04,91.789148
2009-06-05,92.5747
2009-06-08,92.790512
2009-06-09,93.351624
2009-06-10,93.532906
2009-06-11,94.439318
2009-06-12,93.412051
2009-06-15,92.902738
2009-06-16,92.643761
2009-06-17,92.367522
2009-06-18,91.789148
2009-06-19,91.409317
2009-06-22,90.226665
2009-06-23,90.15761
2009-06-24,89.907267
2009-06-25,91.556067
2009-06-26,91.228035
2009-06-29,91.357524
2009-06-30,90.140341
2009-07-01,90.502904
2009-07-02,87.818209
2009-07-06,87.749147
2009-07-07,86.488806
2009-07-08,86.911796
2009-07-09,88.120344
2009-07-10,87.041284
2009-07-13,89.449747
2009-07-14,89.130343
2009-07-15,92.557438
2009-07-16,95.509744
2009-07-17,99.636068
2009-07-20,100.516584
2009-07-21,101.034532
2009-07-22,99.765556
2009-07-23,101.051794
2009-07-24,101.552479
2009-07-27,101.543845
2009-07-28,101.241709
2009-07-29,101.224447
2009-07-30,101.742395
2009-07-31,101.802822
2009-08-03,103.520683
2009-08-04,103.244444
2009-08-05,102.268976
2009-08-06,101.800639
2009-08-07,103.491828
2009-08-10,102.945441
2009-08-11,102.156225
2009-08-12,103.457136
2009-08-13,103.708646
2009-08-14,102.832697
2009-08-17,101.349659
2009-08-18,102.017458
2009-08-19,102.832697
2009-08-20,103.162259
2009-08-21,103.986174
2009-08-24,103.483153
2009-08-25,103.058191
2009-08-26,103.613246
2009-08-27,103.578554
2009-08-28,102.529153
2009-08-31,102.381718
2009-09-01,101.202224
2009-09-02,100.681855
2009-09-03,100.890005
2009-09-04,101.870023
2009-09-08,101.609844
2009-09-09,101.262933
2009-09-10,102.052149
2009-09-11,102.381718
2009-09-14,103.10155
2009-09-15,103.50917
2009-09-16,105.651339
2009-09-17,105.703373
2009-09-18,105.902849
2009-09-21,105.43452
2009-09-22,105.469212
2009-09-23,104.784064
2009-09-24,104.88814
2009-09-25,105.009558
2009-09-28,103.491828
2009-09-29,103.040841
2009-09-30,103.734663
2009-10-01,102.251625
2009-10-02,103.222968
2009-10-05,103.856081
2009-10-06,105.243719
2009-10-07,106.483921
2009-10-08,106.058958
2009-10-09,109.215836
2009-10-12,110.178511
2009-10-13,110.161162
2009-10-14,111.314644
2009-10-15,110.993751
2009-10-16,105.495229
2009-10-19,106.726757
2009-10-20,106.518613
2009-10-21,104.827431
2009-10-22,106.405869
2009-10-23,104.385119
2009-10-26,104.168301
2009-10-27,104.636629
2009-10-28,105.373811
2009-10-29,106.561979
2009-10-30,104.601938
2009-11-02,104.558571
2009-11-03,105.078941
2009-11-04,105.191684
2009-11-05,106.761448
2009-11-06,107.580345
2009-11-09,109.766975
2009-11-10,110.55974
2009-11-11,110.803665
2009-11-12,109.99348
2009-11-13,110.664275
2009-11-16,111.692259
2009-11-17,112.058147
2009-11-18,111.639978
2009-11-19,111.108572
2009-11-20,110.603294
2009-11-23,111.683538
2009-11-24,111.448326
2009-11-25,110.882067
2009-11-27,109.505622
2009-11-30,110.071882
2009-12-01,111.45704
2009-12-02,110.821086
2009-12-03,111.117286
2009-12-04,110.855933
2009-12-07,110.672989
2009-12-08,110.463911
2009-12-09,111.849062
2009-12-10,112.676668
2009-12-11,112.972861
2009-12-14,113.190653
2009-12-15,111.936184
2009-12-16,112.127842
2009-12-17,110.986609
2009-12-18,111.430906
2009-12-21,112.075561
2009-12-22,113.190653
2009-12-23,113.251641
2009-12-24,113.748212
2009-12-28,115.264033
2009-12-29,114.863304
2009-12-30,115.490545
2009-12-31,114.035685
2010-01-04,115.385996
2010-01-05,113.992138
2010-01-06,113.251641
2010-01-07,112.859619
2010-01-08,113.992138
2010-01-11,112.798631
2010-01-12,113.695931
2010-01-13,113.452005
2010-01-14,115.264033
2010-01-15,114.802316
2010-01-19,116.858269
2010-01-20,113.469432
2010-01-21,112.380474
2010-01-22,109.331392
2010-01-25,109.871517
2010-01-26,109.549183
2010-01-27,110.054461
2010-01-28,107.80685
2010-01-29,106.622063
2010-02-01,108.608322
2010-02-02,109.357526
2010-02-03,109.470782
2010-02-04,107.153476
2010-02-05,107.606479
2010-02-08,106.652662
2010-02-09,107.816497
2010-02-10,107.466471
2010-02-11,108.271533
2010-02-12,108.507797
2010-02-16,109.584128
2010-02-17,110.546696
2010-02-18,111.841785
2010-02-19,111.29925
2010-02-22,111.001725
2010-02-23,110.660451
2010-02-24,111.64927
2010-02-25,111.19424
2010-02-26,111.272999
2010-03-01,112.506841
2010-03-02,111.500511
2010-03-03,111.027976
2010-03-04,110.88797
2010-03-05,111.351752
2010-03-08,110.616702
2010-03-09,109.864148
2010-03-10,109.925402
2010-03-11,111.658023
2010-03-12,111.955547
2010-03-15,111.85929
2010-03-16,112.59434
2010-03-17,111.798036
2010-03-18,112.340577
2010-03-19,111.75428
2010-03-22,111.990551
2010-03-23,113.206881
2010-03-24,112.471831
2010-03-25,113.093132
2010-03-26,113.110623
2010-03-29,112.524333
2010-03-30,112.681851
2010-03-31,112.226815
2010-04-01,112.226815
2010-04-05,113.189389
2010-04-06,112.821851
2010-04-07,112.428075
2010-04-08,111.666775
2010-04-09,112.673092
2010-04-12,112.323072
2010-04-13,112.909363
2010-04-14,114.852003
2010-04-15,114.53698
2010-04-16,114.309469
2010-04-19,115.709561
2010-04-20,113.486907
2010-04-21,112.874366
2010-04-22,112.996874
2010-04-23,113.749429
2010-04-26,114.396967
2010-04-27,112.725607
2010-04-28,113.845686
2010-04-29,114.16071
2010-04-30,112.883112
2010-05-03,113.408155
2010-05-04,112.113052
2010-05-05,111.535514
2010-05-06,108.993618
2010-05-07,107.39284
2010-05-10,111.060555
2010-05-11,111.605877
2010-05-12,116.698456
2010-05-13,115.643
2010-05-14,115.387937
2010-05-17,114.728276
2010-05-18,114.297293
2010-05-19,113.338588
2010-05-20,108.888076
2010-05-21,110.31294
2010-05-24,109.459777
2010-05-25,109.521345
2010-05-26,108.386734
2010-05-27,111.166103
2010-05-28,110.172216
2010-06-01,109.363026
2010-06-02,112.063246
2010-06-03,112.546993
2010-06-04,110.189804
2010-06-07,109.178322
2010-06-08,108.817711
2010-06-09,108.97603
2010-06-10,112.300721
2010-06-11,112.97797
2010-06-14,113.02195
2010-06-15,114.156562
2010-06-16,114.64912
2010-06-17,115.203226
2010-06-18,114.4732
2010-06-21,114.912974
2010-06-22,113.725592
2010-06-23,114.438024
2010-06-24,112.749293
2010-06-25,111.808176
2010-06-28,113.44413
2010-06-29,110.022688
2010-06-30,108.606621
2010-07-01,107.806229
2010-07-02,107.18175
2010-07-06,108.589026
2010-07-07,111.702628
2010-07-08,112.555791
2010-07-09,112.546993
2010-07-12,113.171472
2010-07-13,114.763452
2010-07-14,114.974549
2010-07-15,114.974549
2010-07-16,112.608562
2010-07-19,114.156562
2010-07-20,111.306834
2010-07-21,110.181007
2010-07-22,112.116017
2010-07-23,112.916409
2010-07-26,112.942794
2010-07-27,113.136296
2010-07-28,112.960376
2010-07-29,112.599771
2010-07-30,112.93399
2010-08-02,115.009725
2010-08-03,114.666701
2010-08-04,115.458303
2010-08-05,115.950848
2010-08-06,115.031589
2010-08-09,116.675656
2010-08-10,116.534228
2010-08-11,114.75758
2010-08-12,113.405205
2010-08-13,113.025124
2010-08-16,112.936728
2010-08-17,113.537785
2010-08-18,114.368659
2010-08-19,113.93554
2010-08-20,112.698077
2010-08-23,111.787654
2010-08-24,110.399921
2010-08-25,110.726962
2010-08-26,108.526037
2010-08-27,110.249659
2010-08-30,109.074061
2010-08-31,108.835403
2010-09-01,111.168915
2010-09-02,110.523668
2010-09-03,112.768791
2010-09-07,111.328019
2010-09-08,111.442931
2010-09-09,111.690424
2010-09-10,113.131189
2010-09-13,114.56312
2010-09-14,113.891356
2010-09-15,114.40401
2010-09-16,114.616152
2010-09-17,115.075787
2010-09-20,116.49003
2010-09-21,116.657974
2010-09-22,117.179489
2010-09-23,116.383965
2010-09-24,118.540699
2010-09-27,119.018003
2010-09-28,119.230145
2010-09-29,119.751647
2010-09-30,118.567215
2010-10-01,119.893075
2010-10-04,119.548352
2010-10-05,121.67857
2010-10-06,121.837666
2010-10-07,122.615508
2010-10-08,122.730421
2010-10-11,123.446383
2010-10-12,123.614327
2010-10-13,124.073949
2010-10-14,125.072767
2010-10-15,124.683846
2010-10-18,126.248365
2010-10-19,122.005611
2010-10-20,122.924881
2010-10-21,123.596645
2010-10-22,123.455217
2010-10-25,123.605479
2010-10-26,124.339123
2010-10-27,125.010888
2010-10-28,124.542418
2010-10-29,126.928976
2010-11-01,126.681484
2010-11-02,127.141105
2010-11-03,127.432796
2010-11-04,129.748627
2010-11-05,129.863539
2010-11-08,130.032228
2010-11-09,129.748114
2010-11-10,130.11213
2010-11-11,129.117746
2010-11-12,127.617317
2010-11-15,127.528528
2010-11-16,126.285566
2010-11-17,126.028086
2010-11-18,128.16777
2010-11-19,128.780378
2010-11-22,129.082239
2010-11-23,127.120119
2010-11-24,129.455127
2010-11-26,127.75936
2010-11-29,126.862653
2010-11-30,125.593056
2010-12-01,128.212165
2010-12-02,128.895787
2010-12-03,129.073365
2010-12-06,128.72711
2010-12-07,127.86591
2010-12-08,128.718223
2010-12-09,128.114502
2010-12-10,128.57618
2010-12-13,128.096742
2010-12-14,129.464014
2010-12-15,128.487391
2010-12-16,128.336461
2010-12-17,128.735984
2010-12-20,128.30094
2010-12-21,129.392986
2010-12-22,129.579424
2010-12-23,129.526156
2010-12-27,129.037844
2010-12-28,129.366352
2010-12-29,130.085496
2010-12-30,130.218665
2010-12-31,130.298567
2011-01-03,130.937809
2011-01-04,131.079866
2011-01-05,130.556047
2011-01-06,131.985461
2011-01-07,131.337332
2011-01-10,131.079866
2011-01-11,130.760245
2011-01-12,132.37611
2011-01-13,132.127517
2011-01-14,133.175156
2011-01-18,133.752242
2011-01-19,138.226935
2011-01-20,138.324598
2011-01-21,138.058245
2011-01-24,141.725005
2011-01-25,143.331983
2011-01-26,142.976841
2011-01-27,143.003489
2011-01-28,141.352116
2011-01-31,143.829168
2011-02-01,145.214187
2011-02-02,144.983355
2011-02-03,145.187553
2011-02-04,145.604837
2011-02-07,146.332867
2011-02-08,148.008606
2011-02-09,146.760708
2011-02-10,146.261555
2011-02-11,146.04764
2011-02-14,145.486085
2011-02-15,145.147368
2011-02-16,145.646521
2011-02-17,146.395265
2011-02-18,146.930067
2011-02-22,144.354067
2011-02-23,142.776375
2011-02-24,143.302281
2011-02-25,144.648214
2011-02-28,144.29168
2011-03-01,142.589199
2011-03-02,142.758557
2011-03-03,145.717831
2011-03-04,144.24711
2011-03-07,142.553537
2011-03-08,144.648214
2011-03-09,147.839247
2011-03-10,144.416468
2011-03-11,144.781911
2011-03-14,143.854914
2011-03-15,141.742419
2011-03-16,136.376491
2011-03-17,137.428277
2011-03-18,138.952491
2011-03-21,140.548001
2011-03-22,140.833239
2011-03-23,142.197003
2011-03-24,142.651586
2011-03-25,144.559074
2011-03-28,143.837083
2011-03-29,145.183029
2011-03-30,145.824802
2011-03-31,145.352388
2011-04-01,146.422005
2011-04-04,146.404174
2011-04-05,146.172428
2011-04-06,146.216985
2011-04-07,146.520054
2011-04-08,146.225907
2011-04-11,146.136766
2011-04-12,145.512824
2011-04-13,146.136766
2011-04-14,147.045947
2011-04-15,148.151225
2011-04-18,147.910557
2011-04-19,147.429221
2011-04-20,146.849849
2011-04-21,149.996312
2011-04-25,149.452588
2011-04-26,150.183501
2011-04-27,151.859229
2011-04-28,152.224686
2011-04-29,152.046419
2011-05-02,153.445831
2011-05-03,154.087603
2011-05-04,152.082067
2011-05-05,150.156762
2011-05-06,151.213252
2011-05-09,151.401278
2011-05-10,152.547307
2011-05-11,151.759407
2011-05-12,154.212632
2011-05-13,152.135446
2011-05-16,151.186393
2011-05-17,152.654743
2011-05-18,152.601025
2011-05-19,152.73532
2011-05-20,152.350332
2011-05-23,150.649186
2011-05-24,150.407455
2011-05-25,150.192569
2011-05-26,149.682221
2011-05-27,149.968735
2011-05-31,151.249059
2011-06-01,149.127118
2011-06-02,148.706309
2011-06-03,147.775166
2011-06-06,147.506562
2011-06-07,146.557508
2011-06-08,147.139471
2011-06-09,147.587139
2011-06-10,146.100878
2011-06-13,146.09193
2011-06-14,146.942496
2011-06-15,145.339851
2011-06-16,145.644262
2011-06-17,147.22901
2011-06-20,147.748307
2011-06-21,148.822707
2011-06-22,148.339218
2011-06-23,148.733168
2011-06-24,147.793076
2011-06-27,150.076171
2011-06-28,152.216023
2011-06-29,152.69055
2011-06-30,153.594848
2011-07-01,156.271893
2011-07-05,157.068741
2011-07-06,159.110119
2011-07-07,158.008846
2011-07-08,158.017809
2011-07-11,156.674805
2011-07-12,155.833187
2011-07-13,156.074932
2011-07-14,155.994341
2011-07-15,157.167229
2011-07-18,156.934446
2011-07-19,165.825137
2011-07-20,164.428402
2011-07-21,165.547572
2011-07-22,165.798265
2011-07-25,164.473172
2011-07-26,163.783759
2011-07-27,162.369141
2011-07-28,162.772039
2011-07-29,162.816809
2011-08-01,161.831934
2011-08-02,159.41453
2011-08-03,160.112891
2011-08-04,153.532168
2011-08-05,154.875171
2011-08-08,149.470775
2011-08-09,153.418414
2011-08-10,146.161585
2011-08-11,149.92938
2011-08-12,151.251256
2011-08-15,155.558597
2011-08-16,153.984937
2011-08-17,154.200744
2011-08-18,147.321605
2011-08-19,141.665412
2011-08-22,142.960312
2011-08-23,147.762235
2011-08-24,149.956356
2011-08-25,148.895266
2011-08-26,152.096538
2011-08-29,155.225871
2011-08-30,155.126955
2011-08-31,154.587422
2011-09-01,153.16663
2011-09-02,150.154188
2011-09-06,148.472624
2011-09-07,150.450938
2011-09-08,148.598516
2011-09-09,145.109482
2011-09-12,146.053681
2011-09-13,146.961903
2011-09-14,150.387998
2011-09-15,152.950809
2011-09-16,155.558597
2011-09-19,155.684489
2011-09-20,157.114269
2011-09-21,155.585573
2011-09-22,151.628933
2011-09-23,152.276383
2011-09-26,156.925424
2011-09-27,159.802986
2011-09-28,159.659105
2011-09-29,161.11586
2011-09-30,157.249149
2011-10-03,155.828357
2011-10-04,157.132258
2011-10-05,159.029643
2011-10-06,163.381935
2011-10-07,164.011397
2011-10-10,167.815155
2011-10-11,166.3584
2011-10-12,167.365538
2011-10-13,167.995013
2011-10-14,171.331166
2011-10-17,167.788179
2011-10-18,160.873063
2011-10-19,159.515224
2011-10-20,159.389332
2011-10-21,163.327984
2011-10-24,163.885505
2011-10-25,162.185952
2011-10-26,163.63372
2011-10-27,167.149731
2011-10-28,168.561522
2011-10-31,166.025687
2011-11-01,163.076199
2011-11-02,165.387225
2011-11-03,168.426642
2011-11-04,167.599348
2011-11-07,168.444631
2011-11-08,169.058557
2011-11-09,164.535286
2011-11-10,165.53745
2011-11-11,169.175932
2011-11-14,169.148847
2011-11-15,170.412831
2011-11-16,168.489757
2011-11-17,167.686222
2011-11-18,167.243834
2011-11-21,163.849112
2011-11-22,163.695629
2011-11-23,160.662054
2011-11-25,159.858519
2011-11-28,164.508202
2011-11-29,163.361579
2011-11-30,169.735694
2011-12-01,171.044823
2011-12-02,171.234427
2011-12-05,172.299783
2011-12-06,174.195772
2011-12-07,175.197936
2011-12-08,172.967897
2011-12-09,175.658384
2011-12-12,173.509598
2011-12-13,172.579664
2011-12-14,170.385747
2011-12-15,169.266208
2011-12-16,165.736078
2011-12-19,165.122133
2011-12-20,169.049533
2011-12-21,163.840088
2011-12-22,164.354705
2011-12-23,166.801433
2011-12-27,166.982
2011-12-28,166.115273
2011-12-29,168.092501
2011-12-30,166.015959
2012-01-03,168.200853
2012-01-04,167.514678
2012-01-05,166.72018
2012-01-06,164.80613
2012-01-09,163.948426
2012-01-10,163.695629
2012-01-11,164.607516
2012-01-12,163.009468
2012-01-13,161.754508
2012-01-17,162.512898
2012-01-18,163.478954
2012-01-19,162.982384
2012-01-20,170.205179
2012-01-23,171.523332
2012-01-24,173.283886
2012-01-25,173.103318
2012-01-26,172.426181
2012-01-27,171.95671
2012-01-30,173.798516
2012-01-31,173.888807
2012-02-01,173.906854
2012-02-02,172.922751
2012-02-03,174.827764
2012-02-06,174.087435
2012-02-07,174.565944
2012-02-08,174.883166
2012-02-09,175.046319
2012-02-10,174.402793
2012-02-13,174.584063
2012-02-14,174.221523
2012-02-15,174.248713
2012-02-16,174.946618
2012-02-17,175.309158
2012-02-21,175.281969
2012-02-22,175.71702
2012-02-23,179.106831
2012-02-24,179.24278
2012-02-27,179.03432
2012-02-28,179.442182
2012-02-29,178.309225
2012-03-01,179.03432
2012-03-02,180.194466
2012-03-05,181.871248
2012-03-06,178.789597
2012-03-07,179.251853
2012-03-08,181.100832
2012-03-09,181.834985
2012-03-12,182.179408
2012-03-13,184.699103
2012-03-14,185.551088
2012-03-15,186.711234
2012-03-16,186.720293
2012-03-19,186.457453
2012-03-20,185.125095
2012-03-21,185.523898
2012-03-22,186.248993
2012-03-23,186.239921
2012-03-26,188.315505
2012-03-27,187.780739
2012-03-28,187.880439
2012-03-29,188.768687
2012-03-30,189.113097
2012-04-02,189.856323
2012-04-03,189.883513
2012-04-04,186.756555
2012-04-05,186.230862
2012-04-09,185.75049
2012-04-10,183.384876
2012-04-11,183.611467
2012-04-12,186.094913
2012-04-13,183.810869
2012-04-16,183.738358
2012-04-17,188.025461
2012-04-18,181.390875
2012-04-19,180.828919
2012-04-20,180.910503
2012-04-23,180.022255
2012-04-24,181.273043
2012-04-25,184.508774
2012-04-26,186.330563
2012-04-27,187.445388
2012-04-30,187.690111
2012-05-01,188.523965
2012-05-02,188.578345
2012-05-03,187.835132
2012-05-04,185.795811
2012-05-07,184.671913
2012-05-08,183.379478
2012-05-09,183.151938
2012-05-10,182.578545
2012-05-11,183.09733
2012-05-14,181.522753
2012-05-15,181.15868
2012-05-16,181.786694
2012-05-17,180.111999
2012-05-18,178.282578
2012-05-21,179.993674
2012-05-22,179.138133
2012-05-23,178.501008
2012-05-24,178.473705
2012-05-25,176.84452
2012-05-29,178.810474
2012-05-30,177.053854
2012-05-31,175.570285
2012-06-01,172.093472
2012-06-04,171.601977
2012-06-05,172.202687
2012-06-06,176.562372
2012-06-07,176.971942
2012-06-08,177.609053
2012-06-11,175.215322
2012-06-12,177.072061
2012-06-13,175.752328
2012-06-14,177.572653
2012-06-15,181.213302
2012-06-18,180.476059
2012-06-19,181.058562
2012-06-20,180.922043
2012-06-21,176.016269
2012-06-22,176.298417
2012-06-25,175.533884
2012-06-26,174.705634
2012-06-27,175.661307
2012-06-28,174.205042
2012-06-29,178.009527
2012-07-02,178.237067
2012-07-03,178.328075
2012-07-05,177.745572
2012-07-06,174.214152
2012-07-09,172.630465
2012-07-10,169.526808
2012-07-11,168.607549
2012-07-12,166.641596
2012-07-13,169.299268
2012-07-16,168.188869
2012-07-17,167.151284
2012-07-18,171.338036
2012-07-19,177.791083
2012-07-20,175.160715
2012-07-23,173.686256
2012-07-24,173.240272
2012-07-25,173.913797
2012-07-26,176.525958
2012-07-27,178.746756
2012-07-30,179.010697
2012-07-31,178.373586
2012-08-01,177.645454
2012-08-02,176.981039
2012-08-03,180.685406
2012-08-06,180.903836
2012-08-07,181.968724
2012-08-08,181.923035
2012-08-09,181.365465
2012-08-10,182.160682
2012-08-13,181.90475
2012-08-14,181.246634
2012-08-15,181.34718
2012-08-16,183.57746
2012-08-17,183.924803
2012-08-20,183.266687
2012-08-21,181.575692
2012-08-22,180.29603
2012-08-23,178.879252
2012-08-24,180.771339
2012-08-27,178.870117
2012-08-28,178.120591
2012-08-29,178.312547
2012-08-30,176.749518
2012-08-31,178.10232
2012-09-04,177.818953
2012-09-05,178.275977
2012-09-06,181.987025
2012-09-07,182.352639
2012-09-10,183.678006
2012-09-11,185.798605
2012-09-12,186.255629
2012-09-13,188.623011
2012-09-14,189.03433
2012-09-17,189.345103
2012-09-18,189.271991
2012-09-19,188.686987
2012-09-20,188.458475
2012-09-21,188.275668
2012-09-24,187.644972
2012-09-25,187.36162
2012-09-26,186.465856
2012-09-27,188.211692
2012-09-28,189.61932
2012-10-01,192.37975
2012-10-02,191.803895
2012-10-03,192.416306
2012-10-04,192.306624
2012-10-05,192.489431
2012-10-08,191.785624
2012-10-09,190.112914
2012-10-10,188.129431
2012-10-11,188.074576
2012-10-12,189.939242
2012-10-15,190.972108
2012-10-16,192.864194
2012-10-17,183.385518
2012-10-18,178.202865
2012-10-19,176.740383
2012-10-22,177.690987
2012-10-23,174.81174
2012-10-24,174.327296
2012-10-25,175.131663
2012-10-26,176.658122
2012-10-31,177.809817
2012-11-01,180.20462
2012-11-02,176.804359
2012-11-05,177.453339
2012-11-06,178.303411
2012-11-07,175.494178
2012-11-08,174.521048
2012-11-09,174.09874
2012-11-12,173.740701
2012-11-13,172.886923
2012-11-14,170.307194
2012-11-15,170.619342
2012-11-16,171.620011
2012-11-19,174.75056
2012-11-20,173.694796
2012-11-21,174.695466
2012-11-23,177.633233
2012-11-26,177.073222
2012-11-27,175.558434
2012-11-28,176.24697
2012-11-29,175.833851
2012-11-30,174.493508
2012-12-03,173.951849
2012-12-04,173.841687
2012-12-05,173.189867
2012-12-06,174.15382
2012-12-07,176.21943
2012-12-10,176.834521
2012-12-11,178.285039
2012-12-12,177.137478
2012-12-13,176.25616
2012-12-14,176.044998
2012-12-17,177.752569
2012-12-18,179.652937
2012-12-19,179.092927
2012-12-20,178.808334
2012-12-21,177.568962
2012-12-24,176.632549
2012-12-26,176.21943
2012-12-27,176.917156
2012-12-28,174.273171
2012-12-31,175.852216
2013-01-02,180.258852
2013-01-03,179.267358
2013-01-04,178.092257
2013-01-07,177.31191
2013-01-08,177.064033
2013-01-09,176.559117
2013-01-10,177.073222
2013-01-11,178.514551
2013-01-14,176.834521
2013-01-15,176.724359
2013-01-16,176.80698
2013-01-17,177.78011
2013-01-18,178.532916
2013-01-22,180.010975
2013-01-23,187.942915
2013-01-24,187.667497
2013-01-25,188.172427
2013-01-28,188.135697
2013-01-29,187.190108
2013-01-30,186.841259
2013-01-31,186.42814
2013-02-01,188.365209
2013-02-04,187.089122
2013-02-05,186.171074
2013-02-06,185.322925
2013-02-07,184.142878
2013-02-08,185.931377
2013-02-11,184.530079
2013-02-12,184.41944
2013-02-13,184.465539
2013-02-14,184.059895
2013-02-15,185.28604
2013-02-19,184.677589
2013-02-20,183.746448
2013-02-21,182.842977
2013-02-22,185.387451
2013-02-25,182.087002
2013-02-26,183.589724
2013-02-27,186.530628
2013-02-28,185.147759
2013-03-01,187.06534
2013-03-04,189.1673
2013-03-05,190.40266
2013-03-06,192.108204
2013-03-07,193.066987
2013-03-08,193.95203
2013-03-11,193.675453
2013-03-12,194.108753
2013-03-13,195.500836
2013-03-14,198.948796
2013-03-15,198.137508
2013-03-18,196.561045
2013-03-19,196.773081
2013-03-20,198.266575
2013-03-21,195.685216
2013-03-22,195.519279
2013-03-25,194.283919
2013-03-26,195.777413
2013-03-27,194.4222
2013-03-28,196.644013
2013-04-01,195.795855
2013-04-02,197.621239
2013-04-03,196.05399
2013-04-04,194.809402
2013-04-05,193.057773
2013-04-08,192.974804
2013-04-09,192.882607
2013-04-10,195.445524
2013-04-11,196.293682
2013-04-12,194.873943
2013-04-15,192.919478
2013-04-16,195.445524
2013-04-17,193.297465
2013-04-18,190.974241
2013-04-19,175.163441
2013-04-22,173.162892
2013-04-23,176.647722
2013-04-24,176.739918
2013-04-25,178.804994
2013-04-26,179.136883
2013-04-29,183.598938
2013-04-30,186.724222
2013-05-01,184.041467
2013-05-02,186.585941
2013-05-03,188.540392
2013-05-06,186.945486
2013-05-07,187.729118
2013-05-08,189.711256
2013-05-09,188.247805
2013-05-10,189.387069
2013-05-13,187.534601
2013-05-14,188.220019
2013-05-15,188.321905
2013-05-16,189.590841
2013-05-17,193.064219
2013-05-20,192.286185
2013-05-21,193.25872
2013-05-22,191.721182
2013-05-23,190.952406
2013-05-24,190.544861
2013-05-28,192.452901
2013-05-29,192.582573
2013-05-30,193.916352
2013-05-31,192.675202
2013-06-03,193.536593
2013-06-04,190.980192
2013-06-05,187.784688
2013-06-06,188.766493
2013-06-07,191.128393
2013-06-10,189.8965
2013-06-11,188.933209
2013-06-12,186.358279
2013-06-13,188.738708
2013-06-14,187.284513
2013-06-17,188.062547
2013-06-18,189.757556
2013-06-19,187.043698
2013-06-20,182.792287
2013-06-21,181.041705
2013-06-24,179.263323
2013-06-25,180.597103
2013-06-26,180.485959
2013-06-27,181.217678
2013-06-28,177.012582
2013-07-01,177.17004
2013-07-02,177.373812
2013-07-03,178.994722
2013-07-05,180.550788
2013-07-08,180.597103
2013-07-09,177.188568
2013-07-10,178.068488
2013-07-11,178.577919
2013-07-12,177.901772
2013-07-15,179.689397
2013-07-16,179.550468
2013-07-17,180.198829
2013-07-18,183.385076
2013-07-19,179.263323
2013-07-22,179.772755
2013-07-23,180.597103
2013-07-24,182.106869
2013-07-25,182.671872
2013-07-26,182.792287
2013-07-29,181.736381
2013-07-30,181.551123
2013-07-31,180.652674
2013-08-01,181.365879
2013-08-02,180.763832
2013-08-05,181.078748
2013-08-06,176.901438
2013-08-07,175.523754
2013-08-08,174.937305
2013-08-09,174.834924
2013-08-12,176.017111
2013-08-13,175.393434
2013-08-14,174.564965
2013-08-15,172.945256
2013-08-16,172.52637
2013-08-19,171.49311
2013-08-20,171.800297
2013-08-21,172.079559
2013-08-22,172.386746
2013-08-23,172.600841
2013-08-26,171.96786
2013-08-27,170.106132
2013-08-28,169.566229
2013-08-29,170.01304
2013-08-30,169.668624
2013-09-03,171.241787
2013-09-04,170.469168
2013-09-05,171.418639
2013-09-06,170.376076
2013-09-09,172.191258
2013-09-10,173.699268
2013-09-11,177.515803
2013-09-12,177.543728
2013-09-13,178.884174
2013-09-16,179.796417
2013-09-17,178.874871
2013-09-18,180.978619
2013-09-19,180.01983
2013-09-20,176.882822
2013-09-23,177.785762
2013-09-24,176.836276
2013-09-25,176.370844
2013-09-26,177.068992
2013-09-27,173.997137
2013-09-30,172.377428
2013-10-01,173.494477
2013-10-02,172.172651
2013-10-03,171.148695
2013-10-04,171.372108
2013-10-07,169.426591
2013-10-08,166.364054
2013-10-09,168.784306
2013-10-10,171.995785
2013-10-11,173.289686
2013-10-14,174.043683
2013-10-15,171.893389
2013-10-16,173.820271
2013-10-17,162.742992
2013-10-18,161.765582
2013-10-21,160.909189
2013-10-22,162.873313
2013-10-23,163.618007
2013-10-24,165.50766
2013-10-25,164.623342
2013-10-28,165.088774
2013-10-29,169.528986
2013-10-30,167.695183
2013-10-31,166.820182
2013-11-01,166.838789
2013-11-04,167.806896
2013-11-05,165.554206
2013-11-06,167.697329
2013-11-07,168.455375
2013-11-08,168.446022
2013-11-11,171.150666
2013-11-12,171.328482
2013-11-13,171.777693
2013-11-14,170.523639
2013-11-15,171.440781
2013-11-18,172.638685
2013-11-19,173.368657
2013-11-20,173.312508
2013-11-21,172.320495
2013-11-22,169.672
2013-11-25,167.463363
2013-11-26,165.937901
2013-11-27,167.491437
2013-11-29,168.155892
2013-12-02,166.096996
2013-12-03,164.786793
2013-12-04,164.468603
2013-12-05,164.786793
2013-12-06,166.274812
2013-12-09,166.078289
2013-12-10,165.760085
2013-12-11,163.963229
2013-12-12,162.250598
2013-12-13,161.717163
2013-12-16,166.443275
2013-12-17,164.48731
2013-12-18,167.23875
2013-12-19,168.661266
2013-12-20,168.474097
2013-12-23,170.542346
2013-12-24,171.468856
2013-12-26,173.462249
2013-12-27,173.209562
2013-12-30,174.454262
2013-12-31,175.539867
2014-01-02,173.630698
2014-01-03,174.669506
2014-01-06,174.070555
2014-01-07,177.542613
2014-01-08,175.914206
2014-01-09,175.36205
2014-01-10,175.249737
2014-01-13,172.34857
2014-01-14,173.995684
2014-01-15,175.698962
2014-01-16,176.653532
2014-01-17,177.898232
2014-01-21,176.344695
2014-01-22,170.561068
2014-01-23,171.010278
2014-01-24,168.118464
2014-01-27,166.490057
2014-01-28,165.507412
2014-01-29,165.086262
2014-01-30,165.984697
2014-01-31,165.348303
2014-02-03,161.810741
2014-02-04,161.754592
2014-02-05,163.064809
2014-02-06,164.363372
2014-02-07,166.791138
2014-02-10,166.687628
2014-02-11,169.09657
2014-02-12,169.604714
2014-02-13,171.110296
2014-02-14,172.85114
2014-02-18,172.380643
2014-02-19,172.154799
2014-02-20,173.387499
2014-02-21,172.004237
2014-02-24,172.625296
2014-02-25,172.418276
2014-02-26,173.199303
2014-02-27,174.337912
2014-02-28,174.243807
2014-03-03,173.387499
2014-03-04,175.438873
2014-03-05,176.097565
2014-03-06,176.568062
2014-03-07,176.605696
2014-03-10,175.39182
2014-03-11,175.739983
2014-03-12,175.231853
2014-03-13,173.048741
2014-03-14,171.458473
2014-03-17,174.846042
2014-03-18,175.787036
2014-03-19,173.810957
2014-03-20,176.812715
2014-03-21,175.655297
2014-03-24,177.142069
2014-03-25,183.53141
2014-03-26,181.254207
2014-03-27,178.628841
2014-03-28,179.212252
2014-03-31,181.131888
2014-04-01,183.02328
2014-04-02,182.129339
2014-04-03,181.320083
2014-04-04,180.454371
2014-04-07,183.042104
2014-04-08,181.884671
2014-04-09,185.037006
2014-04-10,184.133645
2014-04-11,183.672568
2014-04-14,186.100333
2014-04-15,185.394588
2014-04-16,184.811162
2014-04-17,178.798213
2014-04-21,180.924868
2014-04-22,180.811939
2014-04-23,180.416723
2014-04-24,178.995828
2014-04-25,178.440645
2014-04-28,181.743528
2014-04-29,183.597287
2014-04-30,184.877039
2014-05-01,182.110515
2014-05-02,180.143841
2014-05-05,179.974455
2014-05-06,178.817037
2014-05-07,179.167229
2014-05-08,178.798106
2014-05-09,179.905477
2014-05-12,182.262198
2014-05-13,181.902534
2014-05-14,178.618274
2014-05-15,176.479251
2014-05-16,177.047126
2014-05-19,176.99034
2014-05-20,174.993283
2014-05-21,176.412991
2014-05-22,175.740989
2014-05-23,175.987082
2014-05-27,174.88917
2014-05-28,173.28017
2014-05-29,173.923764
2014-05-30,174.491653
2014-06-02,175.750463
2014-06-03,174.501113
2014-06-04,174.633619
2014-06-05,176.024934
2014-06-06,176.394058
2014-06-09,176.252093
2014-06-10,174.425393
2014-06-11,172.494596
2014-06-12,171.519731
2014-06-13,172.788
2014-06-16,172.589249
2014-06-17,172.504056
2014-06-18,173.77234
2014-06-19,173.034092
2014-06-20,171.832068
2014-06-23,172.390484
2014-06-24,171.197933
2014-06-25,171.046494
2014-06-26,170.715223
2014-06-27,171.983507
2014-06-30,171.567057
2014-07-01,176.375139
2014-07-02,178.305936
2014-07-03,178.438442
2014-07-07,177.974665
2014-07-08,177.198565
2014-07-09,178.334329
2014-07-10,177.652868
2014-07-11,177.936812
2014-07-14,179.697252
2014-07-15,178.400589
2014-07-16,182.063433
2014-07-17,182.186479
2014-07-18,182.195938
2014-07-21,180.634265
2014-07-22,183.700826
2014-07-23,183.265457
2014-07-24,184.789278
2014-07-25,183.99423
2014-07-28,185.300367
2014-07-29,184.155143
2014-07-30,183.615647
2014-07-31,181.410364
2014-08-01,179.02525
2014-08-04,179.489027
2014-08-05,177.084993
2014-08-06,177.056436
2014-08-07,175.466482
2014-08-08,177.684806
2014-08-11,178.484541
2014-08-12,178.360768
2014-08-13,178.941531
2014-08-14,178.874893
2014-08-15,178.398858
2014-08-18,180.283953
2014-08-19,180.959929
2014-08-20,180.98849
2014-08-21,182.064319
2014-08-22,181.283629
2014-08-25,181.997681
2014-08-26,183.739971
2014-08-27,183.035434
2014-08-28,182.797417
2014-08-29,183.083041
2014-09-02,182.378504
2014-09-03,182.74981
2014-09-04,181.540677
2014-09-05,182.035758
2014-09-08,181.026566
2014-09-09,180.883761
2014-09-10,182.359458
2014-09-11,182.530838
2014-09-12,182.111925
2014-09-15,182.616521
2014-09-16,183.71141
2014-09-17,183.559075
2014-09-18,184.463539
2014-09-19,184.701556
2014-09-22,183.854215
2014-09-23,182.435625
2014-09-24,183.092556
2014-09-25,179.950723
2014-09-26,180.950399
2014-09-29,180.550531
2014-09-30,180.731427
2014-10-01,178.198917
2014-10-02,177.951384
2014-10-03,179.627022
2014-10-06,179.979283
2014-10-07,176.808904
2014-10-08,180.283953
2014-10-09,177.484865
2014-10-10,177.018346
2014-10-13,174.723868
2014-10-14,174.990447
2014-10-15,173.0387
2014-10-16,171.220243
2014-10-17,173.324324
2014-10-20,160.995022
2014-10-21,155.406362
2014-10-22,154.035379
2014-10-23,154.406686
2014-10-24,154.311488
2014-10-27,154.111547
2014-10-28,155.758638
2014-10-29,155.625349
2014-10-30,156.47269
2014-10-31,156.520282
2014-11-03,156.482206
2014-11-04,154.85416
2014-11-05,154.063955
2014-11-06,154.773308
2014-11-07,155.358046
2014-11-10,156.719236
2014-11-11,156.537103
2014-11-12,155.214249
2014-11-13,156.048215
2014-11-14,157.361487
2014-11-17,157.361487
2014-11-18,155.185493
2014-11-19,154.744537
2014-11-20,153.98726
2014-11-21,154.255663
2014-11-24,155.43472
2014-11-25,155.060872
2014-11-26,155.243006
2014-11-28,155.453896
2014-12-01,154.849982
2014-12-02,155.933189
2014-12-03,157.706579
2014-12-04,157.256042
2014-12-05,156.508346
2014-12-08,155.156737
2014-12-09,156.239943
2014-12-10,153.86264
2014-12-11,154.39946
2014-12-12,148.945103
2014-12-15,146.721177
2014-12-16,145.139515
2014-12-17,145.63797
2014-12-18,151.149839
2014-12-19,151.945467
2014-12-22,154.754132
2014-12-23,155.521004
2014-12-24,155.1184
2014-12-26,155.616854
2014-12-29,153.86264
2014-12-30,153.421698
2014-12-31,153.795546
2015-01-02,155.348451
2015-01-05,152.904054
2015-01-06,149.60653
2015-01-07,148.628768
2015-01-08,151.859198
2015-01-09,152.520625
2015-01-12,149.961202
2015-01-13,150.315874
2015-01-14,149.347708
2015-01-15,148.168651
2015-01-16,150.632209
2015-01-20,150.450076
2015-01-21,145.791347
2015-01-22,148.954684
2015-01-23,149.414801
2015-01-26,149.884513
2015-01-27,147.305915
2015-01-28,145.273717
2015-01-29,149.040953
2015-01-30,146.960823
2015-02-02,148.25492
2015-02-03,151.90713
2015-02-04,150.459671
2015-02-05,151.370325
2015-02-06,151.283437
2015-02-09,150.347085
2015-02-10,153.059604
2015-02-11,152.712092
2015-02-12,153.020998
2015-02-13,154.835772
2015-02-17,155.376358
2015-02-18,156.563686
2015-02-19,158.20471
2015-02-20,157.97303
2015-02-23,157.25871
2015-02-24,159.112104
2015-02-25,157.162173
2015-02-26,155.289469
2015-02-27,156.322358
2015-03-02,154.912998
2015-03-03,155.443922
2015-03-04,153.889772
2015-03-05,155.588713
2015-03-06,153.001688
2015-03-09,155.192947
2015-03-10,152.335622
2015-03-11,151.360663
2015-03-12,152.499723
2015-03-13,148.928077
2015-03-16,151.630949
2015-03-17,151.515117
2015-03-18,154.266242
2015-03-19,154.266242
2015-03-20,157.229752
2015-03-23,158.919045
2015-03-24,157.345585
2015-03-25,153.677402
2015-03-26,155.019183
2015-03-27,154.835772
2015-03-30,157.02703
2015-03-31,154.932309
2015-04-01,153.658092
2015-04-02,154.88404
2015-04-06,156.41888
2015-04-07,156.447853
2015-04-08,156.235484
2015-04-09,156.708476
2015-04-10,157.210442
2015-04-13,156.747097
2015-04-14,156.66987
2015-04-15,158.43639
2015-04-16,157.47108
2015-04-17,155.09641
2015-04-20,160.395969
2015-04-21,158.56187
2015-04-22,159.623718
2015-04-23,164.334437
2015-04-24,163.890388
2015-04-27,164.807429
2015-04-28,167.886772
2015-04-29,168.350117
2015-04-30,165.348001
2015-05-01,167.645444
2015-05-04,167.93504
2015-05-05,167.075915
2015-05-06,165.393296
2015-05-07,166.307557
2015-05-08,167.951265
2015-05-11,166.433987
2015-05-12,165.879604
2015-05-13,167.562225
2015-05-14,169.283758
2015-05-15,168.515384
2015-05-18,168.320864
2015-05-19,168.72936
2015-05-20,169.001692
2015-05-21,168.593195
2015-05-22,167.50387
2015-05-26,165.471107
2015-05-27,167.289893
2015-05-28,167.007841
2015-05-29,165.004241
2015-06-01,165.519726
2015-06-02,165.004241
2015-06-03,165.266851
2015-06-04,163.769029
2015-06-05,162.815855
2015-06-08,160.81227
2015-06-09,161.142955
2015-06-10,164.294235
2015-06-11,164.15807
2015-06-12,162.417094
2015-06-15,161.707074
2015-06-16,162.271193
2015-06-17,162.592158
2015-06-18,163.642585
2015-06-19,162.417094
2015-06-22,163.13682
2015-06-23,164.002448
2015-06-24,162.397638
2015-06-25,161.53201
2015-06-26,160.928994
2015-06-29,158.507175
2015-06-30,158.205667
2015-07-01,159.985555
2015-07-02,160.569116
2015-07-06,160.218973
2015-07-07,160.481584
2015-07-08,158.691975
2015-07-09,159.363082
2015-07-10,162.378181
2015-07-13,164.741645
2015-07-14,163.992727
2015-07-15,163.914916
2015-07-16,166.317278
2015-07-17,167.785922
2015-07-20,168.476486
2015-07-21,158.604443
2015-07-22,155.958927
2015-07-23,157.301126
2015-07-24,155.375352
2015-07-27,154.71398
2015-07-28,155.667139
2015-07-29,156.678653
2015-07-30,156.552223
2015-07-31,157.554016
2015-08-03,154.363838
2015-08-04,153.284234
2015-08-05,153.576007
2015-08-06,153.301422
2015-08-07,152.124582
2015-08-10,153.723111
2015-08-11,152.507051
2015-08-12,153.144508
2015-08-13,152.07556
2015-08-14,152.742421
2015-08-17,153.291605
2015-08-18,152.997395
2015-08-19,150.967375
2015-08-20,149.712094
2015-08-21,145.975668
2015-08-24,140.699553
2015-08-25,138.238027
2015-08-26,143.867177
2015-08-27,145.671642
2015-08-28,145.122458
2015-08-31,145.0342
2015-09-01,139.9248
2015-09-02,142.249045
2015-09-03,143.945634
2015-09-04,140.925108
2015-09-08,144.386941
2015-09-09,142.249045
2015-09-10,143.376832
2015-09-11,144.524237
2015-09-14,142.837449
2015-09-15,144.681151
2015-09-16,145.544163
2015-09-17,145.279372
2015-09-18,141.719464
2015-09-21,143.651424
2015-09-22,141.641007
2015-09-23,140.885887
2015-09-24,141.621404
2015-09-25,142.611895
2015-09-28,139.767901
2015-09-29,139.718863
2015-09-30,142.170588
2015-10-01,140.817231
2015-10-02,141.788119
2015-10-05,146.161987
2015-10-06,145.907013
2015-10-07,147.191714
2015-10-08,149.339427
2015-10-09,149.447303
2015-10-12,148.221441
2015-10-13,146.730789
2015-10-14,147.113257
2015-10-15,147.191714
2015-10-16,147.485924
2015-10-19,146.338519
2015-10-20,137.924199
2015-10-21,138.198791
2015-10-22,141.307576
2015-10-23,141.886179
2015-10-26,140.885887
2015-10-27,135.197883
2015-10-28,138.110533
2015-10-29,137.835941
2015-10-30,137.375016
2015-11-02,137.659409
2015-11-03,139.14026
2015-11-04,138.895088
2015-11-05,137.188682
2015-11-06,136.852125
2015-11-09,133.941849
2015-11-10,134.100235
2015-11-11,133.654788
2015-11-12,131.694798
2015-11-13,130.417848
2015-11-16,132.358036
2015-11-17,132.466925
2015-11-18,134.446702
2015-11-19,135.357398
2015-11-20,137.099597
2015-11-23,137.060008
2015-11-24,137.198592
2015-11-25,136.604653
2015-11-27,137.060008
2015-11-30,138.010293
2015-12-01,139.851487
2015-12-02,138.287461
2015-12-03,137.515349
2015-12-04,139.010075
2015-12-07,138.138983
2015-12-08,136.65415
2015-12-09,135.228708
2015-12-10,135.396987
2015-12-11,133.209341
2015-12-14,134.555576
2015-12-15,136.396769
2015-12-16,137.881603
2015-12-17,135.367292
2015-12-18,133.535991
2015-12-21,134.129931
2015-12-22,136.535353
2015-12-23,137.139186
2015-12-24,136.852125
2015-12-28,136.218597
2015-12-29,138.366653
2015-12-30,137.9311
2015-12-31,136.22849
2016-01-04,134.575378
2016-01-05,134.476398
2016-01-06,133.803266
2016-01-07,131.516625
2016-01-08,130.299066
2016-01-11,131.882879
2016-01-12,131.556214
2016-01-13,129.84371
2016-01-14,131.566122
2016-01-15,128.715238
2016-01-19,126.814653
2016-01-20,120.627848
2016-01-21,121.667234
2016-01-22,121.261376
2016-01-25,120.845625
2016-01-26,121.350463
2016-01-27,119.736947
2016-01-28,120.984209
2016-01-29,123.528223
2016-02-01,123.567819
2016-02-02,121.69693
2016-02-03,123.458931
2016-02-04,126.359305
2016-02-05,127.270009
2016-02-08,126.980003
2016-02-09,124.07
2016-02-10,120.190002
2016-02-11,117.849998
2016-02-12,121.040001
2016-02-16,122.739998
2016-02-17,126.099998
2016-02-18,132.449997
2016-02-19,133.080002
2016-02-22,133.770004
2016-02-23,132.399994
2016-02-24,132.800003
2016-02-25,134.5
2016-02-26,132.029999
2016-02-29,131.029999
"""


In [ ]:
SBUX_CSV = """Date,Close
2007-01-03,16.149666
2007-01-04,16.167992
2007-01-05,16.099269
2007-01-08,16.03971
2007-01-09,15.970989
2007-01-10,15.920592
2007-01-11,16.406228
2007-01-12,16.447462
2007-01-16,16.594068
2007-01-17,16.626138
2007-01-18,16.387902
2007-01-19,16.282529
2007-01-22,16.167992
2007-01-23,16.012222
2007-01-24,15.938919
2007-01-25,15.563239
2007-01-26,15.522005
2007-01-29,15.833545
2007-01-30,15.833545
2007-01-31,16.00764
2007-02-01,15.764822
2007-02-02,15.700682
2007-02-05,15.586146
2007-02-06,15.425793
2007-02-07,15.425793
2007-02-08,15.311257
2007-02-09,15.10051
2007-02-12,14.967647
2007-02-13,14.963066
2007-02-14,15.274606
2007-02-15,15.219629
2007-02-16,15.123417
2007-02-20,15.036369
2007-02-21,14.843949
2007-02-22,15.123417
2007-02-23,15.004299
2007-02-26,14.66527
2007-02-27,14.088006
2007-02-28,14.156728
2007-03-01,13.923073
2007-03-02,13.689418
2007-03-05,13.47409
2007-03-06,13.744396
2007-03-07,13.959725
2007-03-08,14.065099
2007-03-09,13.886421
2007-03-12,13.776466
2007-03-13,13.451183
2007-03-14,13.432856
2007-03-15,13.556556
2007-03-16,14.010121
2007-03-19,14.220869
2007-03-20,14.376638
2007-03-21,14.784389
2007-03-22,14.495756
2007-03-23,14.394964
2007-03-26,14.637782
2007-03-27,14.532408
2007-03-28,14.330824
2007-03-29,14.353731
2007-03-30,14.367476
2007-04-02,14.317079
2007-04-03,14.427035
2007-04-04,14.358313
2007-04-05,14.385801
2007-04-09,14.266683
2007-04-10,14.275846
2007-04-11,14.078843
2007-04-12,14.051354
2007-04-13,14.092588
2007-04-16,14.22545
2007-04-17,14.220869
2007-04-18,14.19338
2007-04-19,14.101751
2007-04-20,14.504919
2007-04-23,14.463686
2007-04-24,14.358313
2007-04-25,14.587386
2007-04-26,14.578223
2007-04-27,14.431616
2007-04-30,14.211706
2007-05-01,14.207124
2007-05-02,14.317079
2007-05-03,14.486594
2007-05-04,14.065099
2007-05-07,13.904748
2007-05-08,13.826863
2007-05-09,13.762723
2007-05-10,13.533649
2007-05-11,13.565719
2007-05-14,13.240435
2007-05-15,13.00678
2007-05-16,12.919733
2007-05-17,13.043432
2007-05-18,13.263343
2007-05-21,13.414531
2007-05-22,13.290831
2007-05-23,13.235853
2007-05-24,12.970128
2007-05-25,13.107573
2007-05-29,13.089247
2007-05-30,13.148806
2007-05-31,13.199202
2007-06-01,13.345808
2007-06-04,13.208365
2007-06-05,13.09841
2007-06-06,12.896825
2007-06-07,12.571541
2007-06-08,12.676915
2007-06-11,12.617356
2007-06-12,12.708985
2007-06-13,12.72273
2007-06-14,12.649426
2007-06-15,12.72273
2007-06-18,12.672333
2007-06-19,12.640263
2007-06-20,12.516563
2007-06-21,12.030928
2007-06-22,11.701063
2007-06-25,11.75604
2007-06-26,11.806437
2007-06-27,11.971369
2007-06-28,12.117976
2007-06-29,12.021765
2007-07-02,11.934717
2007-07-03,12.076743
2007-07-05,12.06758
2007-07-06,12.16379
2007-07-09,12.06758
2007-07-10,11.962206
2007-07-11,11.907229
2007-07-12,11.893484
2007-07-13,11.94388
2007-07-16,11.948462
2007-07-17,11.852251
2007-07-18,12.140883
2007-07-19,12.69524
2007-07-20,12.69066
2007-07-23,12.905988
2007-07-24,12.960966
2007-07-25,12.809777
2007-07-26,12.56696
2007-07-27,12.337887
2007-07-30,12.360794
2007-07-31,12.22335
2007-08-01,12.461586
2007-08-02,12.333305
2007-08-03,12.053835
2007-08-06,12.31956
2007-08-07,12.424935
2007-08-08,12.699822
2007-08-09,12.85101
2007-08-10,12.846429
2007-08-13,12.741055
2007-08-14,12.438678
2007-08-15,12.172953
2007-08-16,12.19128
2007-08-17,12.232513
2007-08-20,12.328723
2007-08-21,12.530308
2007-08-22,12.617356
2007-08-23,12.608193
2007-08-24,12.640263
2007-08-27,12.539471
2007-08-28,12.314979
2007-08-29,12.589867
2007-08-30,12.530308
2007-08-31,12.621937
2007-09-04,12.699822
2007-09-05,12.571541
2007-09-06,12.635682
2007-09-07,12.44326
2007-09-10,12.388283
2007-09-11,12.521145
2007-09-12,12.521145
2007-09-13,12.589867
2007-09-14,12.66317
2007-09-17,12.5074
2007-09-18,12.750218
2007-09-19,12.727311
2007-09-20,12.594448
2007-09-21,12.585285
2007-09-24,12.438678
2007-09-25,12.41119
2007-09-26,12.686078
2007-09-27,12.356212
2007-09-28,12.00344
2007-10-01,12.058417
2007-10-02,12.186698
2007-10-03,12.15921
2007-10-04,12.150047
2007-10-05,12.296653
2007-10-08,12.214187
2007-10-09,12.264583
2007-10-10,12.16379
2007-10-11,12.058417
2007-10-12,12.06758
2007-10-15,11.962206
2007-10-16,11.94388
2007-10-17,12.095069
2007-10-18,12.214187
2007-10-19,11.957625
2007-10-22,11.998858
2007-10-23,11.97595
2007-10-24,11.94388
2007-10-25,11.989695
2007-10-26,11.989695
2007-10-29,12.017184
2007-10-30,11.980532
2007-10-31,12.22335
2007-11-01,11.833925
2007-11-02,11.696482
2007-11-05,11.435337
2007-11-06,11.430757
2007-11-07,11.045913
2007-11-08,10.720629
2007-11-09,10.340367
2007-11-12,10.601511
2007-11-13,10.990935
2007-11-14,11.110054
2007-11-15,11.041332
2007-11-16,10.615255
2007-11-19,10.477812
2007-11-20,10.583185
2007-11-21,10.450322
2007-11-23,10.569441
2007-11-26,10.16169
2007-11-27,10.358694
2007-11-28,10.519044
2007-11-29,10.555697
2007-11-30,10.716047
2007-12-03,10.450322
2007-12-04,10.234994
2007-12-05,10.377019
2007-12-06,10.468649
2007-12-07,10.363275
2007-12-10,10.413671
2007-12-11,10.033409
2007-12-12,10.024246
2007-12-13,9.850151
2007-12-14,9.735614
2007-12-17,9.231653
2007-12-18,9.437819
2007-12-19,9.176676
2007-12-20,9.405749
2007-12-21,9.648566
2007-12-24,9.721869
2007-12-26,9.538611
2007-12-27,9.373678
2007-12-28,9.22249
2007-12-31,9.378259
2008-01-02,8.846809
2008-01-03,8.567341
2008-01-04,8.297034
2008-01-07,8.420733
2008-01-08,9.098791
2008-01-09,8.915531
2008-01-10,9.304956
2008-01-11,9.06672
2008-01-14,8.956765
2008-01-15,8.631481
2008-01-16,8.842228
2008-01-17,8.727691
2008-01-18,8.549014
2008-01-22,8.553596
2008-01-23,9.149186
2008-01-24,9.369097
2008-01-25,9.007161
2008-01-28,9.007161
2008-01-29,9.149186
2008-01-30,8.805576
2008-01-31,8.663551
2008-02-01,8.805576
2008-02-04,8.791832
2008-02-05,8.480293
2008-02-06,8.31536
2008-02-07,8.489456
2008-02-08,8.365756
2008-02-11,8.484874
2008-02-12,8.571921
2008-02-13,8.636062
2008-02-14,8.329104
2008-02-15,8.379501
2008-02-19,8.292453
2008-02-20,8.365756
2008-02-21,8.168753
2008-02-22,8.361174
2008-02-25,8.475711
2008-02-26,8.732273
2008-02-27,8.723111
2008-02-28,8.507781
2008-02-29,8.237475
2008-03-03,8.177916
2008-03-04,8.200823
2008-03-05,8.310778
2008-03-06,8.063379
2008-03-07,7.834306
2008-03-10,7.696862
2008-03-11,8.12752
2008-03-12,8.067961
2008-03-13,8.077123
2008-03-14,7.967168
2008-03-17,7.907609
2008-03-18,8.356593
2008-03-19,8.017564
2008-03-20,8.031309
2008-03-24,8.196241
2008-03-25,8.242056
2008-03-26,8.090868
2008-03-27,8.072542
2008-03-28,7.811398
2008-03-31,8.017564
2008-04-01,8.475711
2008-04-02,8.521526
2008-04-03,8.402408
2008-04-04,8.475711
2008-04-07,8.388663
2008-04-08,8.237475
2008-04-09,8.00382
2008-04-10,8.040471
2008-04-11,7.907609
2008-04-14,7.774747
2008-04-15,7.935098
2008-04-16,8.100031
2008-04-17,8.090868
2008-04-18,8.374919
2008-04-21,8.269545
2008-04-22,8.109194
2008-04-23,8.177916
2008-04-24,7.325763
2008-04-25,7.266204
2008-04-28,7.174575
2008-04-29,7.421974
2008-04-30,7.435718
2008-05-01,7.62814
2008-05-02,7.541092
2008-05-05,7.486115
2008-05-06,7.495278
2008-05-07,7.307437
2008-05-08,7.261623
2008-05-09,7.266204
2008-05-12,7.348671
2008-05-13,7.307437
2008-05-14,7.293693
2008-05-15,7.362415
2008-05-16,7.811398
2008-05-19,7.820561
2008-05-20,7.715188
2008-05-21,7.641884
2008-05-22,7.829724
2008-05-23,7.765584
2008-05-27,8.022146
2008-05-28,8.168753
2008-05-29,8.397826
2008-05-30,8.333686
2008-06-02,8.214568
2008-06-03,8.132101
2008-06-04,8.301616
2008-06-05,8.484874
2008-06-06,8.095449
2008-06-09,8.026728
2008-06-10,8.173334
2008-06-11,8.049635
2008-06-12,8.164171
2008-06-13,8.324523
2008-06-16,8.406989
2008-06-17,8.301616
2008-06-18,8.141264
2008-06-19,8.242056
2008-06-20,7.893865
2008-06-23,7.467788
2008-06-24,7.591488
2008-06-25,7.838888
2008-06-26,7.458626
2008-06-27,7.490696
2008-06-30,7.211226
2008-07-01,7.156249
2008-07-02,7.183738
2008-07-03,7.12876
2008-07-07,6.849291
2008-07-08,7.027968
2008-07-09,6.743917
2008-07-10,6.583566
2008-07-11,6.441541
2008-07-14,6.414052
2008-07-15,6.22163
2008-07-16,6.569821
2008-07-17,6.592729
2008-07-18,6.569821
2008-07-21,6.455285
2008-07-22,6.931757
2008-07-23,7.06462
2008-07-24,6.661451
2008-07-25,6.606473
2008-07-28,6.519425
2008-07-29,6.867617
2008-07-30,6.72101
2008-07-31,6.730173
2008-08-01,6.606473
2008-08-04,6.441541
2008-08-05,6.652288
2008-08-06,6.844709
2008-08-07,6.652288
2008-08-08,6.927176
2008-08-11,7.467788
2008-08-12,7.495278
2008-08-13,7.467788
2008-08-14,7.75184
2008-08-15,7.646466
2008-08-18,7.564
2008-08-19,7.307437
2008-08-20,7.224971
2008-08-21,7.183738
2008-08-22,7.348671
2008-08-25,7.110435
2008-08-26,7.105853
2008-08-27,7.142505
2008-08-28,7.321182
2008-08-29,7.12876
2008-09-02,7.211226
2008-09-03,7.298275
2008-09-04,6.90885
2008-09-05,6.954665
2008-09-08,7.119597
2008-09-09,6.890524
2008-09-10,6.899687
2008-09-11,7.160831
2008-09-12,7.023386
2008-09-15,6.90885
2008-09-16,7.344089
2008-09-17,7.137923
2008-09-18,7.344089
2008-09-19,7.394485
2008-09-22,6.991316
2008-09-23,6.899687
2008-09-24,6.821802
2008-09-25,6.840128
2008-09-26,6.853872
2008-09-29,6.491937
2008-09-30,6.812639
2008-10-01,6.780569
2008-10-02,6.491937
2008-10-03,6.258282
2008-10-06,5.942161
2008-10-07,5.626039
2008-10-08,5.28243
2008-10-09,5.044194
2008-10-10,5.076264
2008-10-13,5.37864
2008-10-14,5.177056
2008-10-15,4.636443
2008-10-16,4.838028
2008-10-17,4.787631
2008-10-20,5.099171
2008-10-21,4.81512
2008-10-22,4.576884
2008-10-23,4.700584
2008-10-24,4.434859
2008-10-27,4.393625
2008-10-28,4.975471
2008-10-29,5.1908
2008-10-30,5.781809
2008-10-31,6.015464
2008-11-03,5.740576
2008-11-04,5.699343
2008-11-05,5.34657
2008-11-06,5.080845
2008-11-07,4.833446
2008-11-10,4.673095
2008-11-11,4.576884
2008-11-12,4.274507
2008-11-13,4.265345
2008-11-14,3.944642
2008-11-17,3.958386
2008-11-18,3.839268
2008-11-19,3.651428
2008-11-20,3.284911
2008-11-21,3.587287
2008-11-24,3.871338
2008-11-25,3.761383
2008-11-26,4.022527
2008-11-28,4.091249
2008-12-01,3.642265
2008-12-02,3.90799
2008-12-03,3.958386
2008-12-04,3.944642
2008-12-05,4.178296
2008-12-08,4.398207
2008-12-09,4.247019
2008-12-10,4.366136
2008-12-11,4.109575
2008-12-12,4.279089
2008-12-15,4.123319
2008-12-16,4.453185
2008-12-17,4.50358
2008-12-18,4.393625
2008-12-19,4.526488
2008-12-22,4.288251
2008-12-23,4.210367
2008-12-24,4.279089
2008-12-26,4.28367
2008-12-29,4.137063
2008-12-30,4.288251
2008-12-31,4.334066
2009-01-02,4.508162
2009-01-05,4.544814
2009-01-06,4.682258
2009-01-07,4.576884
2009-01-08,4.645606
2009-01-09,4.476092
2009-01-12,4.393625
2009-01-13,4.292833
2009-01-14,4.141645
2009-01-15,4.2516
2009-01-16,4.334066
2009-01-20,4.072923
2009-01-21,4.146226
2009-01-22,4.178296
2009-01-23,4.159971
2009-01-26,4.123319
2009-01-27,4.192041
2009-01-28,4.421114
2009-01-29,4.421114
2009-01-30,4.324903
2009-02-02,4.324903
2009-02-03,4.50358
2009-02-04,4.476092
2009-02-05,4.641025
2009-02-06,4.828865
2009-02-09,4.911331
2009-02-10,4.535651
2009-02-11,4.576884
2009-02-12,4.663932
2009-02-13,4.641025
2009-02-17,4.421114
2009-02-18,4.421114
2009-02-19,4.361555
2009-02-20,4.389044
2009-02-23,4.196622
2009-02-24,4.366136
2009-02-25,4.31574
2009-02-26,4.132482
2009-02-27,4.192041
2009-03-02,4.017945
2009-03-03,3.930897
2009-03-04,4.045434
2009-03-05,3.921735
2009-03-06,3.830105
2009-03-09,3.788872
2009-03-10,4.182878
2009-03-11,4.224111
2009-03-12,4.613536
2009-03-13,4.838028
2009-03-16,4.93882
2009-03-17,5.103753
2009-03-18,5.268685
2009-03-19,5.309918
2009-03-20,5.112915
2009-03-23,5.525247
2009-03-24,5.186219
2009-03-25,5.112915
2009-03-26,5.676436
2009-03-27,5.392385
2009-03-30,5.15873
2009-03-31,5.090008
2009-04-01,5.117497
2009-04-02,5.415292
2009-04-03,5.355733
2009-04-06,5.232034
2009-04-07,5.12666
2009-04-08,5.264104
2009-04-09,5.497759
2009-04-13,5.50234
2009-04-14,5.438199
2009-04-15,5.341989
2009-04-16,5.319081
2009-04-17,5.525247
2009-04-20,5.209126
2009-04-21,5.410711
2009-04-22,5.758902
2009-04-23,6.120838
2009-04-24,6.18956
2009-04-27,6.047534
2009-04-28,6.184978
2009-04-29,6.272026
2009-04-30,6.624799
2009-05-01,6.322422
2009-05-04,6.524007
2009-05-05,6.514844
2009-05-06,6.423215
2009-05-07,6.418633
2009-05-08,6.258282
2009-05-11,6.162071
2009-05-12,6.038372
2009-05-13,5.841368
2009-05-14,5.832205
2009-05-15,5.928416
2009-05-18,6.134582
2009-05-19,6.148327
2009-05-20,6.207886
2009-05-21,6.152908
2009-05-22,5.955905
2009-05-26,6.184978
2009-05-27,6.139163
2009-05-28,6.281189
2009-05-29,6.592729
2009-06-01,6.835546
2009-06-02,6.789732
2009-06-03,7.009642
2009-06-04,6.950083
2009-06-05,6.913431
2009-06-08,6.858454
2009-06-09,6.963827
2009-06-10,6.826383
2009-06-11,6.53317
2009-06-12,6.666032
2009-06-15,6.469029
2009-06-16,6.414052
2009-06-17,6.551496
2009-06-18,6.464448
2009-06-19,6.524007
2009-06-22,6.281189
2009-06-23,6.487355
2009-06-24,6.510262
2009-06-25,6.798895
2009-06-26,6.656869
2009-06-29,6.707266
2009-06-30,6.363656
2009-07-01,6.414052
2009-07-02,6.134582
2009-07-06,6.139163
2009-07-07,5.942161
2009-07-08,5.974231
2009-07-09,6.249119
2009-07-10,6.162071
2009-07-13,6.436959
2009-07-14,6.464448
2009-07-15,6.606473
2009-07-16,6.601892
2009-07-17,6.615636
2009-07-20,6.835546
2009-07-21,6.730173
2009-07-22,7.967168
2009-07-23,7.907609
2009-07-24,7.889283
2009-07-27,7.925935
2009-07-28,7.953424
2009-07-29,7.861795
2009-07-30,8.077123
2009-07-31,8.109194
2009-08-03,8.370338
2009-08-04,8.567341
2009-08-05,8.475711
2009-08-06,8.384081
2009-08-07,8.718529
2009-08-10,8.796414
2009-08-11,8.640644
2009-08-12,8.810158
2009-08-13,8.997998
2009-08-14,8.759762
2009-08-17,8.530689
2009-08-18,8.709366
2009-08-19,8.759762
2009-08-20,8.805576
2009-08-21,9.030068
2009-08-24,8.814739
2009-08-25,8.933858
2009-08-26,8.865136
2009-08-27,8.906369
2009-08-28,8.855973
2009-08-31,8.700203
2009-09-01,8.5032
2009-09-02,8.5032
2009-09-03,8.562759
2009-09-04,8.713947
2009-09-08,8.796414
2009-09-09,9.204164
2009-09-10,9.149186
2009-09-11,9.112534
2009-09-14,9.199583
2009-09-15,9.06672
2009-09-16,9.094209
2009-09-17,9.199583
2009-09-18,9.511122
2009-09-21,9.469889
2009-09-22,9.378259
2009-09-23,9.020906
2009-09-24,8.782669
2009-09-25,9.085046
2009-09-28,9.446982
2009-09-29,9.337026
2009-09-30,9.460726
2009-10-01,9.149186
2009-10-02,9.043813
2009-10-05,9.190419
2009-10-06,9.405749
2009-10-07,9.346189
2009-10-08,9.378259
2009-10-09,9.272886
2009-10-12,9.327864
2009-10-13,9.249979
2009-10-14,9.41033
2009-10-15,9.492796
2009-10-16,9.456144
2009-10-19,9.593589
2009-10-20,9.405749
2009-10-21,9.318701
2009-10-22,9.465307
2009-10-23,9.286631
2009-10-26,9.185838
2009-10-27,8.746018
2009-10-28,8.585666
2009-10-29,8.943021
2009-10-30,8.695621
2009-11-02,8.87888
2009-11-03,8.897206
2009-11-04,8.800995
2009-11-05,9.025487
2009-11-06,9.676055
2009-11-09,9.666892
2009-11-10,9.808917
2009-11-11,9.946361
2009-11-12,9.932617
2009-11-13,9.978432
2009-11-16,10.106712
2009-11-17,10.074642
2009-11-18,9.950943
2009-11-19,9.863895
2009-11-20,9.808917
2009-11-23,9.900547
2009-11-24,9.772266
2009-11-25,9.996757
2009-11-27,9.818081
2009-11-30,10.033409
2009-12-01,9.955524
2009-12-02,9.932617
2009-12-03,9.676055
2009-12-04,9.895965
2009-12-07,9.804336
2009-12-08,9.721869
2009-12-09,9.758521
2009-12-10,10.221249
2009-12-11,10.262482
2009-12-14,10.464067
2009-12-15,10.413671
2009-12-16,10.280809
2009-12-17,10.193761
2009-12-18,10.84891
2009-12-21,10.601511
2009-12-22,10.867236
2009-12-23,10.876399
2009-12-24,10.839747
2009-12-28,10.908469
2009-12-29,10.771025
2009-12-30,10.679396
2009-12-31,10.564859
2010-01-04,10.560277
2010-01-05,10.807677
2010-01-06,10.729792
2010-01-07,10.702304
2010-01-08,10.665652
2010-01-11,10.633581
2010-01-12,10.454904
2010-01-13,10.711466
2010-01-14,10.789351
2010-01-15,10.66107
2010-01-19,10.803095
2010-01-20,10.670233
2010-01-21,10.84891
2010-01-22,10.496137
2010-01-25,10.262482
2010-01-26,10.326623
2010-01-27,10.267064
2010-01-28,10.115876
2010-01-29,9.983014
2010-02-01,10.184597
2010-02-02,10.28539
2010-02-03,10.276227
2010-02-04,9.992176
2010-02-05,9.94178
2010-02-08,10.037991
2010-02-09,10.166272
2010-02-10,10.248739
2010-02-11,10.335786
2010-02-12,10.354112
2010-02-16,10.482393
2010-02-17,10.606092
2010-02-18,10.656489
2010-02-19,10.702304
2010-02-22,10.491556
2010-02-23,10.390764
2010-02-24,10.574022
2010-02-25,10.491556
2010-02-26,10.496137
2010-03-01,10.670233
2010-03-02,10.688559
2010-03-03,10.564859
2010-03-04,10.500719
2010-03-05,10.706885
2010-03-08,10.683977
2010-03-09,10.821422
2010-03-10,11.100891
2010-03-11,11.119217
2010-03-12,11.123798
2010-03-15,11.187939
2010-03-16,11.586526
2010-03-17,11.710225
2010-03-18,11.462827
2010-03-19,11.439919
2010-03-22,11.563619
2010-03-23,11.641504
2010-03-24,11.586526
2010-03-25,11.091727
2010-03-26,11.265824
2010-03-29,11.274987
2010-03-30,11.252079
2010-03-31,11.119217
2010-04-01,11.105472
2010-04-05,11.321693
2010-04-06,11.317093
2010-04-07,11.459706
2010-04-08,11.422903
2010-04-09,11.372298
2010-04-12,11.266488
2010-04-13,11.376898
2010-04-14,11.427503
2010-04-15,11.560916
2010-04-16,11.482708
2010-04-19,11.455106
2010-04-20,11.620722
2010-04-21,11.680527
2010-04-22,12.53621
2010-04-23,12.540811
2010-04-26,12.600616
2010-04-27,12.204979
2010-04-28,12.062364
2010-04-29,12.237182
2010-04-30,11.951954
2010-05-03,12.504007
2010-05-04,11.974956
2010-05-05,12.062364
2010-05-06,11.781738
2010-05-07,11.708131
2010-05-10,12.439601
2010-05-11,12.283186
2010-05-12,12.812237
2010-05-13,12.623619
2010-05-14,12.195778
2010-05-17,12.379795
2010-05-18,12.22798
2010-05-19,12.048563
2010-05-20,11.547115
2010-05-21,11.634524
2010-05-24,11.533313
2010-05-25,11.464307
2010-05-26,11.367697
2010-05-27,11.970356
2010-05-28,11.91055
2010-06-01,11.823142
2010-06-02,12.22798
2010-06-03,12.356793
2010-06-04,12.030161
2010-06-07,11.749535
2010-06-08,11.892148
2010-06-09,12.103768
2010-06-10,12.411998
2010-06-11,12.490206
2010-06-14,12.632819
2010-06-15,12.849041
2010-06-16,12.876643
2010-06-17,12.872042
2010-06-18,12.922648
2010-06-21,12.890445
2010-06-22,12.527009
2010-06-23,12.568413
2010-06-24,12.269385
2010-06-25,12.33379
2010-06-28,12.140572
2010-06-29,11.505711
2010-06-30,11.179079
2010-07-01,11.344695
2010-07-02,11.202082
2010-07-06,10.861649
2010-07-07,11.225084
2010-07-08,11.427503
2010-07-09,11.639123
2010-07-12,11.625323
2010-07-13,11.933552
2010-07-14,11.961155
2010-07-15,12.02096
2010-07-16,11.662126
2010-07-19,11.726532
2010-07-20,11.855345
2010-07-21,11.579318
2010-07-22,11.570117
2010-07-23,11.675927
2010-07-26,11.680527
2010-07-27,11.583918
2010-07-28,11.49651
2010-07-29,11.418302
2010-07-30,11.432104
2010-08-02,11.413605
2010-08-03,11.432103
2010-08-04,11.644837
2010-08-05,11.644837
2010-08-06,11.714206
2010-08-09,11.866819
2010-08-10,11.746579
2010-08-11,11.404356
2010-08-12,11.311863
2010-08-13,11.094505
2010-08-16,11.015886
2010-08-17,11.237868
2010-08-18,11.330362
2010-08-19,11.117629
2010-08-20,11.122253
2010-08-23,10.951141
2010-08-24,10.553422
2010-08-25,10.803153
2010-08-26,10.761531
2010-08-27,10.854024
2010-08-30,10.835525
2010-08-31,10.627416
2010-09-01,10.951141
2010-09-02,11.404356
2010-09-03,11.593966
2010-09-07,11.501474
2010-09-08,11.459852
2010-09-09,11.482975
2010-09-10,11.746579
2010-09-13,11.908441
2010-09-14,11.922315
2010-09-15,11.908441
2010-09-16,11.899192
2010-09-17,11.8067
2010-09-20,12.153547
2010-09-21,12.098051
2010-09-22,11.991685
2010-09-23,11.769702
2010-09-24,12.084177
2010-09-27,12.093427
2010-09-28,12.088802
2010-09-29,11.931564
2010-09-30,11.815948
2010-10-01,11.99631
2010-10-04,11.880694
2010-10-05,12.139673
2010-10-06,12.070304
2010-10-07,12.056429
2010-10-08,12.056429
2010-10-11,12.019432
2010-10-12,12.551266
2010-10-13,12.606762
2010-10-14,12.68538
2010-10-15,12.736252
2010-10-18,12.648383
2010-10-19,12.523518
2010-10-20,12.69463
2010-10-21,12.620636
2010-10-22,13.175592
2010-10-25,13.129345
2010-10-26,13.194091
2010-10-27,13.069225
2010-10-28,13.055351
2010-10-29,13.207964
2010-11-01,13.32358
2010-11-02,13.355952
2010-11-03,13.457695
2010-11-04,13.758296
2010-11-05,14.276256
2010-11-08,14.165264
2010-11-09,13.952531
2010-11-10,14.09127
2010-11-11,14.216135
2010-11-12,13.96178
2010-11-15,14.239259
2010-11-16,13.85843
2010-11-17,13.928094
2010-11-18,14.202105
2010-11-19,14.285701
2010-11-22,14.336788
2010-11-23,14.118508
2010-11-24,14.620087
2010-11-26,14.462182
2010-11-29,14.299634
2010-11-30,14.211393
2010-12-01,14.72226
2010-12-02,15.214549
2010-12-03,15.195974
2010-12-06,15.195974
2010-12-07,15.223838
2010-12-08,15.130955
2010-12-09,15.047358
2010-12-10,15.135598
2010-12-13,14.856943
2010-12-14,14.912675
2010-12-15,14.801213
2010-12-16,15.135598
2010-12-17,15.228483
2010-12-20,15.293503
2010-12-21,15.260993
2010-12-22,15.293503
2010-12-23,15.154176
2010-12-27,15.079868
2010-12-28,15.042713
2010-12-29,15.098443
2010-12-30,15.052002
2010-12-31,14.921963
2011-01-03,15.442118
2011-01-04,15.084511
2011-01-05,15.024135
2011-01-06,14.84301
2011-01-07,15.223838
2011-01-10,15.219195
2011-01-11,14.982337
2011-01-12,14.954473
2011-01-13,15.052002
2011-01-14,15.186685
2011-01-18,15.326012
2011-01-19,15.330656
2011-01-20,15.409609
2011-01-21,15.418897
2011-01-24,15.553581
2011-01-25,15.627889
2011-01-26,15.358522
2011-01-27,15.339944
2011-01-28,14.736193
2011-01-31,14.643308
2011-02-01,14.977694
2011-02-02,14.954473
2011-02-03,15.028781
2011-02-04,15.130955
2011-02-07,15.084325
2011-02-08,15.443364
2011-02-09,15.382749
2011-02-10,15.462016
2011-02-11,15.55061
2011-02-14,15.657857
2011-02-15,15.471342
2011-02-16,15.653193
2011-02-17,15.620553
2011-02-18,15.853696
2011-02-22,15.280165
2011-02-23,14.87916
2011-02-24,14.87916
2011-02-25,15.154268
2011-02-28,15.378085
2011-03-01,15.102976
2011-03-02,15.023709
2011-03-03,15.392073
2011-03-04,15.443364
2011-03-07,15.667181
2011-03-08,15.858358
2011-03-09,16.10549
2011-03-10,17.704848
2011-03-11,17.047386
2011-03-14,16.660369
2011-03-15,16.655707
2011-03-16,16.319981
2011-03-17,16.361947
2011-03-18,16.301329
2011-03-21,16.46453
2011-03-22,16.296667
2011-03-23,17.108002
2011-03-24,17.522998
2011-03-25,17.205923
2011-03-28,17.196598
2011-03-29,17.187272
2011-03-30,17.121992
2011-03-31,17.229237
2011-04-01,17.369123
2011-04-04,17.126654
2011-04-05,16.972781
2011-04-06,16.968117
2011-04-07,16.739638
2011-04-08,16.679021
2011-04-11,16.548461
2011-04-12,16.679021
2011-04-13,16.697673
2011-04-14,16.772278
2011-04-15,16.963455
2011-04-18,16.697673
2011-04-19,16.860872
2011-04-20,17.20126
2011-04-21,17.252551
2011-04-25,17.191934
2011-04-26,17.066036
2011-04-27,17.341145
2011-04-28,17.196598
2011-04-29,16.879523
2011-05-02,17.10334
2011-05-03,16.926151
2011-05-04,17.052048
2011-05-05,17.019409
2011-05-06,17.033397
2011-05-09,16.90237
2011-05-10,16.953845
2011-05-11,16.747948
2011-05-12,17.056795
2011-05-13,16.91641
2011-05-16,16.453138
2011-05-17,16.518651
2011-05-18,17.131667
2011-05-19,17.37968
2011-05-20,17.131667
2011-05-23,17.052114
2011-05-24,16.981923
2011-05-25,16.90237
2011-05-26,17.052114
2011-05-27,17.061473
2011-05-31,17.215898
2011-06-01,16.836857
2011-06-02,16.80878
2011-06-03,16.453138
2011-06-06,16.72455
2011-06-07,16.81346
2011-06-08,16.72455
2011-06-09,16.626279
2011-06-10,16.43442
2011-06-13,16.331472
2011-06-14,16.504613
2011-06-15,16.312753
2011-06-16,16.443779
2011-06-17,16.584164
2011-06-20,16.766666
2011-06-21,17.18782
2011-06-22,17.445193
2011-06-23,17.655771
2011-06-24,17.477949
2011-06-27,17.777439
2011-06-28,18.367056
2011-06-29,18.451287
2011-06-30,18.479365
2011-07-01,18.806928
2011-07-05,19.162572
2011-07-06,18.914557
2011-07-07,18.867763
2011-07-08,18.8818
2011-07-11,18.596352
2011-07-12,18.5168
2011-07-13,18.52148
2011-07-14,18.310902
2011-07-15,18.624428
2011-07-18,18.437249
2011-07-19,18.867763
2011-07-20,18.638468
2011-07-21,18.89584
2011-07-22,18.8818
2011-07-25,18.905199
2011-07-26,18.80225
2011-07-27,18.23603
2011-07-28,18.708659
2011-07-29,18.760134
2011-08-01,18.624428
2011-08-02,18.067568
2011-08-03,18.381094
2011-08-04,17.267373
2011-08-05,17.183142
2011-08-08,15.990324
2011-08-09,16.924855
2011-08-10,16.323749
2011-08-11,17.42734
2011-08-12,17.544744
2011-08-15,18.042532
2011-08-16,18.263251
2011-08-17,18.174025
2011-08-18,16.319053
2011-08-19,16.483417
2011-08-22,16.384799
2011-08-23,17.248887
2011-08-24,17.549439
2011-08-25,17.103306
2011-08-26,17.558831
2011-08-29,17.868776
2011-08-30,18.080102
2011-08-31,18.136455
2011-09-01,17.934522
2011-09-02,17.605794
2011-09-06,17.727892
2011-09-07,18.399439
2011-09-08,18.277339
2011-09-09,17.521263
2011-09-12,17.680932
2011-09-13,17.934522
2011-09-14,18.103583
2011-09-15,18.347782
2011-09-16,18.408832
2011-09-19,19.329273
2011-09-20,19.291705
2011-09-21,18.737561
2011-09-22,18.13176
2011-09-23,18.178721
2011-09-26,18.54502
2011-09-27,18.714079
2011-09-28,18.516843
2011-09-29,17.925129
2011-09-30,17.511871
2011-10-03,16.999993
2011-10-04,17.493086
2011-10-05,17.868776
2011-10-06,18.080102
2011-10-07,18.437007
2011-10-10,19.216565
2011-10-11,19.409108
2011-10-12,19.493637
2011-10-13,19.2964
2011-10-14,19.827063
2011-10-17,19.329273
2011-10-18,19.935074
2011-10-19,19.216565
2011-10-20,19.24944
2011-10-21,19.766013
2011-10-24,19.977339
2011-10-25,19.385626
2011-10-26,19.441981
2011-10-27,20.226234
2011-10-28,19.958554
2011-10-31,19.892809
2011-11-01,19.362146
2011-11-02,19.338665
2011-11-03,19.441981
2011-11-04,20.7522
2011-11-07,20.770985
2011-11-08,20.832035
2011-11-09,20.169881
2011-11-10,20.43756
2011-11-11,20.822643
2011-11-14,20.493913
2011-11-15,20.757925
2011-11-16,20.281761
2011-11-17,19.815025
2011-11-18,19.810312
2011-11-21,19.650019
2011-11-22,19.932888
2011-11-23,19.447295
2011-11-25,19.254001
2011-11-28,19.730164
2011-11-29,19.899886
2011-11-30,20.498628
2011-12-01,20.550487
2011-12-02,20.701351
2011-12-05,20.838072
2011-12-06,20.61649
2011-12-07,20.687208
2011-12-08,20.206329
2011-12-09,20.724923
2011-12-12,20.743782
2011-12-13,20.559917
2011-12-14,20.314763
2011-12-15,20.460913
2011-12-16,20.498628
2011-12-19,20.56463
2011-12-20,21.243519
2011-12-21,21.32838
2011-12-22,21.215231
2011-12-23,21.427384
2011-12-27,21.658393
2011-12-28,21.582961
2011-12-29,21.898834
2011-12-30,21.691395
2012-01-03,21.351952
2012-01-04,21.766826
2012-01-05,21.856403
2012-01-06,22.026125
2012-01-09,21.964836
2012-01-10,22.073269
2012-01-11,22.214704
2012-01-12,22.440999
2012-01-13,22.327853
2012-01-17,22.492859
2012-01-18,22.648438
2012-01-19,22.639009
2012-01-20,22.700298
2012-01-23,22.318423
2012-01-24,22.464573
2012-01-25,22.521147
2012-01-26,22.789873
2012-01-27,22.558862
2012-01-30,22.855876
2012-01-31,22.591863
2012-02-01,22.742729
2012-02-02,22.469287
2012-02-03,22.780444
2012-02-06,22.84668
2012-02-07,22.903454
2012-02-08,23.05012
2012-02-09,23.277214
2012-02-10,23.09743
2012-02-13,23.30087
2012-02-14,23.239364
2012-02-15,22.936572
2012-02-16,22.955496
2012-02-17,22.922379
2012-02-21,22.832486
2012-02-22,22.865605
2012-02-23,22.927109
2012-02-24,22.851411
2012-02-27,22.818293
2012-02-28,23.140011
2012-02-29,22.974421
2012-03-01,23.045388
2012-03-02,23.130548
2012-03-05,23.201516
2012-03-06,22.865605
2012-03-07,23.414418
2012-03-08,23.830757
2012-03-09,24.526235
2012-03-12,24.180862
2012-03-13,24.762792
2012-03-14,24.923651
2012-03-15,25.108165
2012-03-16,25.174401
2012-03-19,25.33526
2012-03-20,25.420421
2012-03-21,25.458271
2012-03-22,26.120629
2012-03-23,26.186865
2012-03-26,26.451809
2012-03-27,26.617399
2012-03-28,26.508583
2012-03-29,26.366649
2012-03-30,26.442347
2012-04-02,26.811375
2012-04-03,26.924924
2012-04-04,26.948579
2012-04-05,27.525779
2012-04-09,27.166211
2012-04-10,26.863418
2012-04-11,28.050936
2012-04-12,28.684909
2012-04-13,29.176946
2012-04-16,28.221257
2012-04-17,27.752873
2012-04-18,28.542975
2012-04-19,27.823841
2012-04-20,27.98943
2012-04-23,27.705563
2012-04-24,27.464273
2012-04-25,28.150289
2012-04-26,28.699101
2012-04-27,27.170943
2012-04-30,27.142556
2012-05-01,27.279759
2012-05-02,27.402768
2012-05-03,26.801915
2012-05-04,26.28622
2012-05-07,26.32893
2012-05-08,25.71674
2012-05-09,25.674029
2012-05-10,26.029953
2012-05-11,26.105884
2012-05-14,25.469965
2012-05-15,25.313359
2012-05-16,25.161498
2012-05-17,24.520833
2012-05-18,24.454394
2012-05-21,25.484203
2012-05-22,25.327596
2012-05-23,26.238764
2012-05-24,25.987242
2012-05-25,25.89233
2012-05-29,26.376388
2012-05-30,25.973006
2012-05-31,26.048936
2012-06-01,24.748626
2012-06-04,25.579117
2012-06-05,24.872012
2012-06-06,25.375053
2012-06-07,25.360815
2012-06-08,25.432001
2012-06-11,25.071331
2012-06-12,25.17099
2012-06-13,24.331007
2012-06-14,24.834048
2012-06-15,24.933707
2012-06-18,25.711995
2012-06-19,26.158086
2012-06-20,26.404861
2012-06-21,25.659792
2012-06-22,25.930295
2012-06-25,25.289631
2012-06-26,25.602845
2012-06-27,25.033365
2012-06-28,24.720151
2012-06-29,25.303868
2012-07-02,25.052348
2012-07-03,24.648966
2012-07-05,24.867268
2012-07-06,24.663204
2012-07-09,24.862521
2012-07-10,24.838793
2012-07-11,24.753371
2012-07-12,25.000146
2012-07-13,25.441492
2012-07-16,25.104551
2012-07-17,25.49844
2012-07-18,25.294376
2012-07-19,25.721486
2012-07-20,24.658457
2012-07-23,24.013047
2012-07-24,23.9561
2012-07-25,23.92288
2012-07-26,24.867268
2012-07-27,22.527656
2012-07-30,22.247662
2012-07-31,21.488355
2012-08-01,20.776506
2012-08-02,20.482275
2012-08-03,20.8382
2012-08-06,20.714333
2012-08-07,21.586165
2012-08-08,21.576636
2012-08-09,21.490882
2012-08-10,21.710031
2012-08-13,21.967293
2012-08-14,22.119744
2012-08-15,22.915349
2012-08-16,23.058274
2012-08-17,22.97252
2012-08-20,22.891529
2012-08-21,22.910586
2012-08-22,22.924877
2012-08-23,22.820068
2012-08-24,23.201196
2012-08-27,23.410816
2012-08-28,23.515627
2012-08-29,23.425108
2012-08-30,23.68237
2012-08-31,23.63473
2012-09-04,23.587088
2012-09-05,23.725247
2012-09-06,24.220715
2012-09-07,24.377929
2012-09-10,24.215952
2012-09-11,24.168309
2012-09-12,24.377929
2012-09-13,24.639957
2012-09-14,24.039678
2012-09-17,23.65855
2012-09-18,23.463222
2012-09-19,23.872935
2012-09-20,24.387458
2012-09-21,24.330289
2012-09-24,24.373166
2012-09-25,24.073027
2012-09-26,23.86817
2012-09-27,24.315997
2012-09-28,24.158781
2012-10-01,23.891992
2012-10-02,23.487042
2012-10-03,23.577561
2012-10-04,23.39176
2012-10-05,23.220253
2012-10-08,23.22978
2012-10-09,22.558041
2012-10-10,22.367478
2012-10-11,22.467524
2012-10-12,22.477052
2012-10-15,22.710492
2012-10-16,23.325062
2012-10-17,23.053509
2012-10-18,22.586626
2012-10-19,21.7672
2012-10-22,21.5814
2012-10-23,21.424185
2012-10-24,21.562343
2012-10-25,22.03399
2012-10-26,21.852954
2012-10-31,21.867247
2012-11-01,22.210262
2012-11-02,24.220715
2012-11-05,24.301704
2012-11-06,24.635192
2012-11-07,24.682834
2012-11-08,24.230243
2012-11-09,24.277884
2012-11-12,24.144489
2012-11-13,24.067947
2012-11-14,23.364709
2012-11-15,23.168568
2012-11-16,23.326437
2012-11-19,23.795264
2012-11-20,23.948349
2012-11-21,24.163625
2012-11-23,24.488932
2012-11-26,24.345414
2012-11-27,24.020107
2012-11-28,24.575043
2012-11-29,24.780752
2012-11-30,24.814239
2012-12-03,24.775969
2012-12-04,24.455444
2012-12-05,24.297576
2012-12-06,25.689699
2012-12-07,25.660995
2012-12-10,25.335687
2012-12-11,25.445718
2012-12-12,25.641859
2012-12-13,25.440935
2012-12-14,25.527045
2012-12-17,26.110685
2012-12-18,25.99587
2012-12-19,25.962383
2012-12-20,25.933679
2012-12-21,25.641859
2012-12-24,25.684914
2012-12-26,25.417015
2012-12-27,25.469639
2012-12-28,25.182602
2012-12-31,25.656212
2013-01-02,26.311609
2013-01-03,26.488614
2013-01-04,26.6417
2013-01-07,26.656053
2013-01-08,26.608213
2013-01-09,26.134605
2013-01-10,26.091549
2013-01-11,26.316393
2013-01-14,26.153739
2013-01-15,26.062845
2013-01-16,26.000653
2013-01-17,26.072413
2013-01-18,26.220715
2013-01-22,26.306826
2013-01-23,26.053277
2013-01-24,26.1059
2013-01-25,27.177501
2013-01-28,26.79957
2013-01-29,26.598644
2013-01-30,26.790002
2013-01-31,26.847409
2013-02-01,27.201421
2013-02-04,26.833058
2013-02-05,26.986719
2013-02-06,26.91469
2013-02-07,26.823454
2013-02-08,27.068351
2013-02-11,26.957907
2013-02-12,27.015529
2013-02-13,26.809049
2013-02-14,26.674594
2013-02-15,26.093564
2013-02-19,26.146385
2013-02-20,25.598968
2013-02-21,25.618174
2013-02-22,26.011931
2013-02-25,25.550948
2013-02-26,25.584561
2013-02-27,26.204008
2013-02-28,26.33846
2013-03-01,26.348065
2013-03-04,26.746624
2013-03-05,27.135577
2013-03-06,27.41889
2013-03-07,27.97111
2013-03-08,28.172789
2013-03-11,28.143979
2013-03-12,27.985515
2013-03-13,28.134375
2013-03-14,27.697401
2013-03-15,27.687797
2013-03-18,27.337258
2013-03-19,27.28924
2013-03-20,27.577353
2013-03-21,27.39488
2013-03-22,27.553344
2013-03-25,27.222011
2013-03-26,27.39488
2013-03-27,27.327654
2013-03-28,27.346862
2013-04-01,27.308446
2013-04-02,27.975911
2013-04-03,27.707005
2013-04-04,27.903883
2013-04-05,27.755023
2013-04-08,27.908684
2013-04-09,27.567749
2013-04-10,27.755023
2013-04-11,28.129573
2013-04-12,28.37447
2013-04-15,27.711806
2013-04-16,28.119969
2013-04-17,27.932694
2013-04-18,27.716609
2013-04-19,28.043139
2013-04-22,28.158384
2013-04-23,28.518527
2013-04-24,28.729812
2013-04-25,29.051539
2013-04-26,28.811443
2013-04-29,29.089955
2013-04-30,29.214804
2013-05-01,28.87867
2013-05-02,28.993916
2013-05-03,29.7094
2013-05-06,30.002316
2013-05-07,29.963771
2013-05-08,30.06977
2013-05-09,30.04568
2013-05-10,30.402218
2013-05-13,30.267312
2013-05-14,30.604579
2013-05-15,30.869574
2013-05-16,30.619033
2013-05-17,30.898482
2013-05-20,30.753941
2013-05-21,30.9563
2013-05-22,30.90812
2013-05-23,30.604579
2013-05-24,30.52749
2013-05-28,30.96112
2013-05-29,30.657579
2013-05-30,30.580489
2013-05-31,30.421491
2013-06-03,30.57567
2013-06-04,30.513036
2013-06-05,30.036043
2013-06-06,30.363674
2013-06-07,31.336931
2013-06-10,31.852468
2013-06-11,31.520017
2013-06-12,31.052661
2013-06-13,31.785014
2013-06-14,31.606743
2013-06-17,31.813922
2013-06-18,32.329458
2013-06-19,31.997012
2013-06-20,31.423657
2013-06-21,31.168298
2013-06-24,30.840667
2013-06-25,31.192386
2013-06-26,31.703108
2013-06-27,31.650108
2013-06-28,31.563382
2013-07-01,31.915102
2013-07-02,32.180099
2013-07-03,32.425823
2013-07-05,32.628182
2013-07-08,32.902814
2013-07-09,32.825722
2013-07-10,32.811269
2013-07-11,33.408712
2013-07-12,33.591802
2013-07-15,33.562895
2013-07-16,33.495438
2013-07-17,32.859449
2013-07-18,32.994359
2013-07-19,33.143719
2013-07-22,33.254533
2013-07-23,32.599275
2013-07-24,32.093373
2013-07-25,32.844995
2013-07-26,35.345591
2013-07-29,34.907142
2013-07-30,34.656602
2013-07-31,34.348244
2013-08-01,35.403409
2013-08-02,35.764767
2013-08-05,35.639495
2013-08-06,35.175641
2013-08-07,34.8809
2013-08-08,35.243286
2013-08-09,35.175641
2013-08-12,35.238453
2013-08-13,35.09833
2013-08-14,34.716616
2013-08-15,34.180284
2013-08-16,34.165789
2013-08-19,33.962854
2013-08-20,34.141632
2013-08-21,34.165789
2013-08-22,34.740777
2013-08-23,34.774599
2013-08-26,34.731111
2013-08-27,33.900041
2013-08-28,34.286585
2013-08-29,34.392885
2013-08-30,34.073983
2013-09-03,34.595821
2013-09-04,34.856739
2013-09-05,34.813255
2013-09-06,34.581326
2013-09-09,35.001695
2013-09-10,35.861759
2013-09-11,36.42708
2013-09-12,36.562371
2013-09-13,36.514053
2013-09-16,36.354602
2013-09-17,36.741149
2013-09-18,37.364454
2013-09-19,36.890935
2013-09-20,36.779805
2013-09-23,36.412586
2013-09-24,36.987573
2013-09-25,36.886102
2013-09-26,37.291976
2013-09-27,37.364454
2013-09-30,37.190509
2013-10-01,37.282314
2013-10-02,37.296809
2013-10-03,37.142191
2013-10-04,37.398277
2013-10-07,37.137358
2013-10-08,36.494726
2013-10-09,36.364268
2013-10-10,37.262987
2013-10-11,37.601213
2013-10-14,37.736503
2013-10-15,37.06488
2013-10-16,37.707513
2013-10-17,38.045739
2013-10-18,38.321152
2013-10-21,38.39363
2013-10-22,39.089414
2013-10-23,38.67871
2013-10-24,38.195528
2013-10-25,38.635221
2013-10-28,38.031244
2013-10-29,38.470941
2013-10-30,39.055591
2013-10-31,39.161892
2013-11-01,38.833328
2013-11-04,38.833328
2013-11-05,39.61608
2013-11-06,39.200543
2013-11-07,38.20519
2013-11-08,39.234366
2013-11-11,39.132898
2013-11-12,39.07473
2013-11-13,39.486756
2013-11-14,39.33164
2013-11-15,39.355879
2013-11-18,39.040798
2013-11-19,38.740259
2013-11-20,38.628772
2013-11-21,39.520688
2013-11-22,39.433435
2013-11-25,39.123203
2013-11-26,39.510995
2013-11-27,39.564316
2013-11-29,39.486756
2013-12-02,39.297709
2013-12-03,39.045647
2013-12-04,38.53667
2013-12-05,38.643313
2013-12-06,38.749956
2013-12-09,38.648162
2013-12-10,37.509025
2013-12-11,37.033983
2013-12-12,37.072763
2013-12-13,37.009745
2013-12-16,37.063066
2013-12-17,36.883712
2013-12-18,37.654447
2013-12-19,37.397537
2013-12-20,37.644754
2013-12-23,37.964679
2013-12-24,38.085864
2013-12-26,38.236132
2013-12-27,38.085864
2013-12-30,38.076171
2013-12-31,37.998611
2014-01-02,37.40723
2014-01-03,37.300587
2014-01-06,36.922492
2014-01-07,37.42662
2014-01-08,37.824105
2014-01-09,37.615667
2014-01-10,37.649599
2014-01-13,36.413519
2014-01-14,36.578328
2014-01-15,36.932189
2014-01-16,36.495924
2014-01-17,36.306876
2014-01-21,35.700954
2014-01-22,35.676715
2014-01-23,35.574921
2014-01-24,36.345656
2014-01-27,35.972406
2014-01-28,35.81729
2014-01-29,34.68785
2014-01-30,34.857511
2014-01-31,34.474567
2014-02-03,33.43238
2014-02-04,34.376331
2014-02-05,34.298478
2014-02-06,35.208369
2014-02-07,36.025811
2014-02-10,36.395607
2014-02-11,36.249634
2014-02-12,35.962558
2014-02-13,36.342084
2014-02-14,36.507516
2014-02-18,35.991751
2014-02-19,35.675478
2014-02-20,35.787392
2014-02-21,35.305682
2014-02-24,35.305682
2014-02-25,34.327675
2014-02-26,34.926157
2014-02-27,35.125653
2014-02-28,34.527168
2014-03-03,34.288748
2014-03-04,34.86777
2014-03-05,34.692604
2014-03-06,35.368939
2014-03-07,35.553835
2014-03-10,35.792255
2014-03-11,36.507516
2014-03-12,36.799459
2014-03-13,36.215574
2014-03-14,36.137721
2014-03-17,36.093931
2014-03-18,36.29829
2014-03-19,36.935702
2014-03-20,37.446601
2014-03-21,37.324958
2014-03-24,36.867579
2014-03-25,36.303157
2014-03-26,35.782525
2014-03-27,35.714405
2014-03-28,35.860374
2014-03-31,35.704672
2014-04-01,36.011214
2014-04-02,35.845778
2014-04-03,35.563565
2014-04-04,34.814247
2014-04-07,34.269285
2014-04-08,34.780187
2014-04-09,35.266759
2014-04-10,34.167105
2014-04-11,33.442113
2014-04-14,33.724323
2014-04-15,33.519963
2014-04-16,34.444451
2014-04-17,34.133045
2014-04-21,34.293615
2014-04-22,34.619617
2014-04-23,34.249821
2014-04-24,34.590421
2014-04-25,34.765587
2014-04-28,34.512571
2014-04-29,34.371465
2014-04-30,34.361735
2014-05-01,34.605021
2014-05-02,34.352001
2014-05-05,34.507704
2014-05-06,33.980273
2014-05-07,34.058409
2014-05-08,33.980273
2014-05-09,34.32701
2014-05-12,34.747003
2014-05-13,34.751887
2014-05-14,34.268406
2014-05-15,34.11213
2014-05-16,34.644447
2014-05-19,34.683513
2014-05-20,34.29771
2014-05-21,34.380731
2014-05-22,34.869093
2014-05-23,35.152344
2014-05-27,35.972794
2014-05-28,35.782329
2014-05-29,35.704193
2014-05-30,35.767679
2014-06-02,36.06558
2014-06-03,36.226741
2014-06-04,36.466037
2014-06-05,36.490457
2014-06-06,36.788359
2014-06-09,36.715103
2014-06-10,36.431852
2014-06-11,36.529527
2014-06-12,36.1193
2014-06-13,36.475807
2014-06-16,36.671149
2014-06-17,36.778589
2014-06-18,36.90068
2014-06-19,37.716248
2014-06-20,37.408577
2014-06-23,37.472067
2014-06-24,37.813919
2014-06-25,38.150891
2014-06-26,38.121586
2014-06-27,38.062985
2014-06-30,37.7895
2014-07-01,38.131356
2014-07-02,38.185076
2014-07-03,38.609949
2014-07-07,38.429257
2014-07-08,38.365768
2014-07-09,38.80041
2014-07-10,38.507393
2014-07-11,38.385303
2014-07-14,38.365768
2014-07-15,38.526928
2014-07-16,38.448792
2014-07-17,37.721129
2014-07-18,38.062985
2014-07-21,37.901825
2014-07-22,38.453673
2014-07-23,38.649019
2014-07-24,39.288773
2014-07-25,38.453673
2014-07-28,38.268097
2014-07-29,38.409722
2014-07-30,38.531813
2014-07-31,37.93601
2014-08-01,37.594158
2014-08-04,37.862755
2014-08-05,37.627554
2014-08-06,37.794155
2014-08-07,37.588353
2014-08-08,38.03426
2014-08-11,38.156762
2014-08-12,38.13226
2014-08-13,37.848055
2014-08-14,37.544254
2014-08-15,37.686356
2014-08-18,38.019557
2014-08-19,38.279263
2014-08-20,38.235161
2014-08-21,37.960758
2014-08-22,37.867656
2014-08-25,38.205761
2014-08-26,38.11756
2014-08-27,38.18126
2014-08-28,38.127359
2014-08-29,38.127359
2014-09-02,37.96566
2014-09-03,37.627554
2014-09-04,37.808858
2014-09-05,38.195959
2014-09-08,38.058758
2014-09-09,37.789257
2014-09-10,37.833356
2014-09-11,37.299251
2014-09-12,36.980747
2014-09-15,36.711242
2014-09-16,36.794542
2014-09-17,36.917043
2014-09-18,37.108149
2014-09-19,37.274749
2014-09-22,36.55444
2014-09-23,36.240837
2014-09-24,36.907245
2014-09-25,36.319239
2014-09-26,36.833743
2014-09-29,36.882743
2014-09-30,36.975846
2014-10-01,36.559341
2014-10-02,36.480939
2014-10-03,37.186548
2014-10-06,36.823945
2014-10-07,36.284939
2014-10-08,36.877846
2014-10-09,36.495642
2014-10-10,36.48584
2014-10-13,35.373528
2014-10-14,35.643029
2014-10-15,35.466626
2014-10-16,35.594029
2014-10-17,36.035035
2014-10-20,36.60344
2014-10-21,36.43684
2014-10-22,36.55444
2014-10-23,36.672041
2014-10-24,37.147347
2014-10-27,37.22575
2014-10-28,37.754957
2014-10-29,37.505053
2014-10-30,37.887257
2014-10-31,37.024845
2014-11-03,37.289449
2014-11-04,37.588353
2014-11-05,37.563855
2014-11-06,37.950956
2014-11-07,38.11756
2014-11-10,38.206125
2014-11-11,38.245489
2014-11-12,38.30453
2014-11-13,38.324211
2014-11-14,38.43738
2014-11-17,38.294691
2014-11-18,38.166762
2014-11-19,38.289769
2014-11-20,38.476739
2014-11-21,39.244309
2014-11-24,39.618249
2014-11-25,39.465721
2014-11-26,39.214784
2014-11-28,39.957751
2014-12-01,39.78062
2014-12-02,39.544447
2014-12-03,39.593649
2014-12-04,40.006953
2014-12-05,41.118942
2014-12-08,41.23211
2014-12-09,40.853245
2014-12-10,40.671196
2014-12-11,40.89753
2014-12-12,40.961492
2014-12-15,39.800301
2014-12-16,38.934327
2014-12-17,39.578889
2014-12-18,39.377155
2014-12-19,39.086859
2014-12-22,39.628091
2014-12-23,40.065998
2014-12-24,39.987271
2014-12-26,40.26281
2014-12-29,40.533425
2014-12-30,40.243129
2014-12-31,40.371058
2015-01-02,40.070919
2015-01-05,39.30335
2015-01-06,38.983533
2015-01-07,39.94299
2015-01-08,40.587548
2015-01-09,39.259069
2015-01-12,39.475563
2015-01-13,39.790462
2015-01-14,39.569046
2015-01-15,39.155743
2015-01-16,39.662533
2015-01-20,39.962672
2015-01-21,39.997114
2015-01-22,40.710556
2015-01-23,43.406881
2015-01-26,43.357679
2015-01-27,43.465923
2015-01-28,43.087061
2015-01-29,43.815267
2015-01-30,43.06738
2015-02-02,43.293713
2015-02-03,43.698651
2015-02-04,43.802354
2015-02-05,44.266552
2015-02-06,43.950504
2015-02-09,43.861615
2015-02-10,45.027044
2015-02-11,44.834452
2015-02-12,45.348032
2015-02-13,45.224575
2015-02-17,45.446795
2015-02-18,45.925807
2015-02-19,46.009757
2015-02-20,46.177659
2015-02-23,46.212227
2015-02-24,46.148027
2015-02-25,46.548029
2015-02-26,46.691239
2015-02-27,46.167781
2015-03-02,46.533215
2015-03-03,46.419633
2015-03-04,45.955435
2015-03-05,46.236916
2015-03-06,45.540624
2015-03-09,45.945561
2015-03-10,45.52087
2015-03-11,45.145561
2015-03-12,46.11346
2015-03-13,46.069017
2015-03-16,46.444326
2015-03-17,46.612225
2015-03-18,47.328271
2015-03-19,48.276419
2015-03-20,48.12827
2015-03-23,48.083828
2015-03-24,48.35543
2015-03-25,47.293703
2015-03-26,46.952966
2015-03-27,46.948027
2015-03-30,47.39741
2015-03-31,46.76531
2015-04-01,45.935682
2015-04-02,46.612225
2015-04-06,46.676421
2015-04-07,46.454201
2015-04-08,47.02704
2015-04-09,47.367778
2015-04-10,47.575184
2015-04-13,47.901111
2015-04-14,47.70358
2015-04-15,47.545555
2015-04-16,47.654198
2015-04-17,47.031976
2015-04-20,47.377656
2015-04-21,47.772715
2015-04-22,47.743086
2015-04-23,48.819627
2015-04-24,51.199868
2015-04-27,50.241844
2015-04-28,49.985056
2015-04-29,50.024563
2015-04-30,48.967776
2015-05-01,49.669008
2015-05-04,49.827032
2015-05-05,48.955132
2015-05-06,48.479551
2015-05-07,48.895683
2015-05-08,49.321724
2015-05-11,49.044303
2015-05-12,49.252369
2015-05-13,49.133475
2015-05-14,50.094546
2015-05-15,50.332335
2015-05-18,50.708837
2015-05-19,50.946626
2015-05-20,50.560217
2015-05-21,50.857458
2015-05-22,51.006075
2015-05-26,50.371967
2015-05-27,51.115063
2015-05-28,51.333039
2015-05-29,51.481655
2015-06-01,51.739264
2015-06-02,51.253773
2015-06-03,51.640182
2015-06-04,51.243867
2015-06-05,51.709538
2015-06-08,51.055614
2015-06-09,51.065524
2015-06-10,52.204935
2015-06-11,52.006779
2015-06-12,52.145489
2015-06-15,51.788803
2015-06-16,52.48236
2015-06-17,52.749874
2015-06-18,53.611864
2015-06-19,53.433521
2015-06-22,53.403798
2015-06-23,53.62177
2015-06-24,53.215545
2015-06-25,53.572231
2015-06-26,54.117167
2015-06-29,53.057018
2015-06-30,53.126373
2015-07-01,53.393888
2015-07-02,53.740668
2015-07-06,53.810024
2015-07-07,53.879379
2015-07-08,52.898491
2015-07-09,53.552415
2015-07-10,54.067628
2015-07-13,55.187227
2015-07-14,55.236766
2015-07-15,54.83054
2015-07-16,55.226859
2015-07-17,55.177317
2015-07-20,55.69253
2015-07-21,55.682624
2015-07-22,56.168111
2015-07-23,56.03931
2015-07-24,56.762589
2015-07-27,56.455442
2015-07-28,56.613969
2015-07-29,56.980561
2015-07-30,57.525501
2015-07-31,57.396697
2015-08-03,57.654302
2015-08-04,58.319965
2015-08-05,58.627956
2015-08-06,56.859481
2015-08-07,56.829677
2015-08-10,55.905697
2015-08-11,55.985177
2015-08-12,56.014986
2015-08-13,56.48194
2015-08-14,56.730322
2015-08-17,57.366182
2015-08-18,57.455599
2015-08-19,57.217151
2015-08-20,55.448676
2015-08-21,52.497904
2015-08-24,50.014089
2015-08-25,50.759233
2015-08-26,53.610651
2015-08-27,55.587769
2015-08-28,55.269841
2015-08-31,54.355796
2015-09-01,53.15363
2015-09-02,54.902234
2015-09-03,54.335925
2015-09-04,53.928579
2015-09-08,54.852559
2015-09-09,54.335925
2015-09-10,55.011523
2015-09-11,56.164012
2015-09-14,55.925568
2015-09-15,56.541553
2015-09-16,56.889286
2015-09-17,56.909157
2015-09-18,56.472007
2015-09-21,57.167476
2015-09-22,56.750193
2015-09-23,57.415857
2015-09-24,57.9921
2015-09-25,57.614563
2015-09-28,55.408934
2015-09-29,55.359259
2015-09-30,56.472007
2015-10-01,57.107863
2015-10-02,57.70398
2015-10-05,58.657764
2015-10-06,58.310028
2015-10-07,58.399445
2015-10-08,59.075043
2015-10-09,59.681095
2015-10-12,60.148053
2015-10-13,59.770512
2015-10-14,58.439187
2015-10-15,59.303554
2015-10-16,59.542002
2015-10-19,60.575269
2015-10-20,60.485852
2015-10-21,60.138116
2015-10-22,61.091903
2015-10-23,62.204651
2015-10-26,63.019342
2015-10-27,62.304002
2015-10-28,63.098822
2015-10-29,62.095363
2015-10-30,62.164909
2015-11-02,61.837048
2015-11-03,62.39342
2015-11-04,61.558858
2015-11-05,61.876786
2015-11-06,61.568795
2015-11-09,61.140195
2015-11-10,61.977459
2015-11-11,61.668468
2015-11-12,60.871074
2015-11-13,59.545409
2015-11-16,60.482345
2015-11-17,60.352768
2015-11-18,61.598696
2015-11-19,61.259803
2015-11-20,61.78808
2015-11-23,62.43596
2015-11-24,61.758175
2015-11-25,61.987425
2015-11-27,61.977459
2015-11-30,61.190032
2015-12-01,61.170096
2015-12-02,61.020587
2015-12-03,59.356025
2015-12-04,61.54886
2015-12-07,61.688403
2015-12-08,61.957524
2015-12-09,60.980717
2015-12-10,61.668468
2015-12-11,59.625146
2015-12-14,59.724819
2015-12-15,59.784625
2015-12-16,60.153418
2015-12-17,59.326124
2015-12-18,58.429054
2015-12-21,59.346059
2015-12-22,59.794594
2015-12-23,60.143453
2015-12-24,60.123517
2015-12-28,59.99394
2015-12-29,60.93088
2015-12-30,60.621889
2015-12-31,59.834461
2016-01-04,58.070226
2016-01-05,58.458959
2016-01-06,57.940652
2016-01-07,56.50534
2016-01-08,56.445538
2016-01-11,57.631661
2016-01-12,59.266318
2016-01-13,57.681497
2016-01-14,58.787882
2016-01-15,57.811075
2016-01-19,58.359282
2016-01-20,56.734591
2016-01-21,58.837718
2016-01-22,58.977262
2016-01-25,57.522018
2016-01-26,58.419088
2016-01-27,57.442281
2016-01-28,59.096874
2016-01-29,60.572052
2016-02-01,61.200001
2016-02-02,60.700001
2016-02-03,59.529999
2016-02-04,58.290001
2016-02-05,54.490002
2016-02-08,54.139999
2016-02-09,54.419998
2016-02-10,55.139999
2016-02-11,54.919998
2016-02-12,55.860001
2016-02-16,56.41
2016-02-17,57.630001
2016-02-18,56.959999
2016-02-19,57.669998
2016-02-22,58.869999
2016-02-23,58.459999
2016-02-24,58.110001
2016-02-25,58.75
2016-02-26,58.34
2016-02-29,58.209999
"""


In [ ]:
STOCKS = ['AAPL', 'MSFT', 'IBM', 'SBUX']
raw_data = {
    'AAPL': pd.read_csv(io.StringIO(AAPL_CSV)),
    'MSFT': pd.read_csv(io.StringIO(MSFT_CSV)),
    'IBM': pd.read_csv(io.StringIO(IBM_CSV)),
    'SBUX': pd.read_csv(io.StringIO(SBUX_CSV)),
}

for s in STOCKS:
    print(s, raw_data[s].shape, raw_data[s]['Date'].min(), '->', raw_data[s]['Date'].max())


In [ ]:
# Quick look at the raw price data
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, s in zip(axes.flatten(), STOCKS):
    ax.plot(pd.to_datetime(raw_data[s]['Date']), raw_data[s]['Close'])
    ax.set_title(f'{s} Closing Price')
    ax.set_ylabel('Price ($)')
plt.tight_layout()
plt.show()


## 3. Feature Engineering: Technical Indicators

Instead of feeding the agent raw prices, we compute standard **technical indicators**
used by real traders. This gives the agent much richer signals to learn from:

| Feature | Meaning |
|---|---|
| `returns` | Daily percentage price change |
| `sma5` / `sma10` / `sma20` | Price vs its 5/10/20-day moving average (trend signal) |
| `volatility10` | Standard deviation of returns over last 10 days (risk signal) |
| `momentum5` | Price change over the last 5 days (momentum signal) |
| `rsi14` | Relative Strength Index (overbought/oversold signal, normalized 0-1) |


In [ ]:
def add_technical_indicators(prices):
    prices = pd.Series(prices)
    df = pd.DataFrame({'Close': prices})
    df['returns'] = df['Close'].pct_change()
    df['sma5'] = df['Close'].rolling(5).mean() / df['Close'] - 1
    df['sma10'] = df['Close'].rolling(10).mean() / df['Close'] - 1
    df['sma20'] = df['Close'].rolling(20).mean() / df['Close'] - 1
    df['volatility10'] = df['returns'].rolling(10).std()
    df['momentum5'] = df['Close'] / df['Close'].shift(5) - 1
    delta = df['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df['rsi14'] = (100 - (100 / (1 + rs))) / 100
    df = df.fillna(0)
    return df

# preview features for AAPL
add_technical_indicators(raw_data['AAPL']['Close'].values).tail()


## 4. Custom Trading Environment

This defines the "world" the RL agent interacts with:
- **State**: 7 technical indicators + position (holding or not) + cash ratio + net worth ratio = 10 numbers
- **Actions**: `0 = HOLD`, `1 = BUY` (invest all cash), `2 = SELL` (liquidate all shares)
- **Reward**: change in total portfolio value (cash + shares) each day
- **Transaction cost**: 0.1% per trade, simulating real broker fees


In [ ]:
class AdvancedTradingEnv:
    FEATURE_COLS = [
        'returns', 'sma5', 'sma10', 'sma20',
        'volatility10', 'momentum5', 'rsi14'
    ]

    def __init__(
        self,
        prices,
        initial_balance=10000.0,
        transaction_cost=0.001,
        start_offset=20
    ):
        self.prices = np.asarray(prices, dtype=np.float32)
        if self.prices.ndim != 1 or len(self.prices) < start_offset + 2:
            raise ValueError("Not enough price data for the selected start_offset.")
        if np.any(~np.isfinite(self.prices)) or np.any(self.prices <= 0):
            raise ValueError("Prices must be finite and strictly positive.")
        if initial_balance <= 0:
            raise ValueError("initial_balance must be positive.")
        if not 0 <= transaction_cost < 1:
            raise ValueError("transaction_cost must be in [0, 1).")

        self.feat_df = add_technical_indicators(self.prices)
        self.initial_balance = float(initial_balance)
        self.transaction_cost = float(transaction_cost)
        self.start_offset = int(start_offset)
        self.n_steps = len(self.prices)

        self.action_space_n = 3
        self.observation_space_n = len(self.FEATURE_COLS) + 3
        self.reset()

    def reset(self):
        self.current_step = self.start_offset
        self.balance = self.initial_balance
        self.shares_held = 0.0
        self.net_worth = self.initial_balance
        self.prev_net_worth = self.initial_balance
        self.trade_history = []
        return self._get_observation()

    def _get_observation(self):
        row = self.feat_df.iloc[self.current_step]
        feats = row[self.FEATURE_COLS].to_numpy(dtype=np.float32)

        position = 1.0 if self.shares_held > 0 else 0.0
        cash_ratio = self.balance / self.initial_balance
        networth_ratio = self.net_worth / self.initial_balance

        return np.concatenate(
            [feats, np.array([position, cash_ratio, networth_ratio], dtype=np.float32)]
        ).astype(np.float32)

    def step(self, action):
        if action not in (0, 1, 2):
            raise ValueError("Action must be 0=HOLD, 1=BUY, or 2=SELL.")
        if self.current_step >= self.n_steps - 1:
            raise RuntimeError("Cannot call step() after the episode is done.")

        current_price = float(self.prices[self.current_step])

        # Execute today's action.
        if action == 1 and self.balance > 0 and self.shares_held == 0:
            invested = self.balance * (1.0 - self.transaction_cost)
            self.shares_held = invested / current_price
            self.balance = 0.0
            self.trade_history.append((self.current_step, 'BUY', current_price))

        elif action == 2 and self.shares_held > 0:
            proceeds = self.shares_held * current_price * (1.0 - self.transaction_cost)
            self.balance += proceeds
            self.shares_held = 0.0
            self.trade_history.append((self.current_step, 'SELL', current_price))

        # Mark the portfolio at the next trading day's close.
        self.current_step += 1
        next_price = float(self.prices[self.current_step])
        self.net_worth = self.balance + self.shares_held * next_price

        reward = self.net_worth - self.prev_net_worth
        self.prev_net_worth = self.net_worth

        done = self.current_step >= self.n_steps - 1
        if done:
            obs = np.zeros(self.observation_space_n, dtype=np.float32)
        else:
            obs = self._get_observation()

        info = {
            'net_worth': float(self.net_worth),
            'balance': float(self.balance),
            'shares_held': float(self.shares_held),
        }
        return obs, float(reward), done, info

print(
    "AdvancedTradingEnv ready. State size:",
    AdvancedTradingEnv(raw_data['AAPL']['Close'].values).observation_space_n
)


## 5. The RL Agent: Double DQN

We use **Double DQN**, an improvement over vanilla DQN that reduces overestimation of
Q-values. It uses the online network to *select* the best next action, and the target
network to *evaluate* how good that action is — this makes training more stable and accurate.

Components:
- **Q-Network**: neural network that estimates the value of each action
- **Target Network**: a slowly-updated copy used for stable learning targets
- **Experience Replay**: stores past experiences and samples randomly for training
- **Epsilon-Greedy**: starts by exploring randomly, gradually shifts to using learned knowledge


In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size)
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=20000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(args)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            np.asarray(s, dtype=np.float32),
            np.asarray(a, dtype=np.int64),
            np.asarray(r, dtype=np.float32),
            np.asarray(ns, dtype=np.float32),
            np.asarray(d, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DoubleDQNAgent:
    def __init__(
        self,
        state_size,
        action_size,
        lr=1e-3,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay=0.97,
        buffer_size=20000,
        batch_size=64,
        target_update_freq=5
    ):
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.device = DEVICE

        self.q_network = QNetwork(state_size, action_size).to(self.device)
        self.target_network = QNetwork(state_size, action_size).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size)
        self.loss_fn = nn.MSELoss()

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_size)

        state_t = torch.as_tensor(
            state, dtype=torch.float32, device=self.device
        ).unsqueeze(0)

        with torch.no_grad():
            q_values = self.q_network(state_t)

        return int(torch.argmax(q_values, dim=1).item())

    def remember(self, s, a, r, ns, d):
        self.memory.push(s, a, r, ns, d)

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return None

        s, a, r, ns, d = self.memory.sample(self.batch_size)

        s = torch.as_tensor(s, dtype=torch.float32, device=self.device)
        a = torch.as_tensor(a, dtype=torch.long, device=self.device).unsqueeze(1)
        r = torch.as_tensor(r, dtype=torch.float32, device=self.device).unsqueeze(1)
        ns = torch.as_tensor(ns, dtype=torch.float32, device=self.device)
        d = torch.as_tensor(d, dtype=torch.float32, device=self.device).unsqueeze(1)

        q_values = self.q_network(s).gather(1, a)

        # Double DQN:
        # online network selects the action, target network evaluates it.
        with torch.no_grad():
            next_actions = self.q_network(ns).argmax(dim=1, keepdim=True)
            next_q = self.target_network(ns).gather(1, next_actions)
            target_q = r + self.gamma * next_q * (1.0 - d)

        loss = self.loss_fn(q_values, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=1.0)
        self.optimizer.step()

        return float(loss.item())

    def update_target_network(self):
        self.target_network.load_state_dict(self.q_network.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(
            self.epsilon_end,
            self.epsilon * self.epsilon_decay
        )

print("DoubleDQNAgent ready.")


## 6. Train an Agent for Each Stock

We train a **separate DQN agent for each of the 4 stocks**, using the first 80% of each
stock's history (2007–2014). The last 20% (2014–2016) is kept completely unseen, for testing.

This will take a few minutes per stock (training progress is printed every 10 episodes).


In [ ]:
EPISODES = 40
WINDOW_SPLIT = 0.8

trained_agents = {}
train_curves = {}
train_test_prices = {}

for stock in STOCKS:
    prices = raw_data[stock]['Close'].to_numpy(dtype=np.float32)
    split_idx = int(len(prices) * WINDOW_SPLIT)

    train_prices = prices[:split_idx]
    test_prices = prices[split_idx:]
    train_test_prices[stock] = (train_prices, test_prices)

    env = AdvancedTradingEnv(train_prices)
    agent = DoubleDQNAgent(
        env.observation_space_n,
        env.action_space_n
    )

    print(f"\n=== Training on {stock} ===")
    ep_networths = []

    for ep in range(EPISODES):
        state = env.reset()
        done = False
        info = {'net_worth': env.initial_balance}

        while not done:
            action = agent.select_action(state, training=True)
            next_state, reward, done, info = env.step(action)

            agent.remember(state, action, reward, next_state, done)
            agent.train_step()

            state = next_state

        agent.decay_epsilon()

        if (ep + 1) % agent.target_update_freq == 0:
            agent.update_target_network()

        ep_networths.append(info['net_worth'])

        if (ep + 1) % 10 == 0:
            print(
                f"  Episode {ep + 1}/{EPISODES} | "
                f"Net Worth: ${info['net_worth']:.2f} | "
                f"Epsilon: {agent.epsilon:.3f}"
            )

    trained_agents[stock] = agent
    train_curves[stock] = ep_networths

    print(
        f"{stock} training complete. "
        f"Final training net worth: ${ep_networths[-1]:.2f}"
    )


In [ ]:
# Plot training progress for all 4 stocks
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, stock in zip(axes.flatten(), STOCKS):
    ax.plot(train_curves[stock])
    ax.axhline(y=10000, linestyle='--', label='Start ($10,000)')
    ax.set_title(f'{stock} - Training Progress')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Net Worth ($)')
    ax.legend()

plt.tight_layout()
plt.savefig('training_progress_all_stocks.png', dpi=150)
plt.show()


## 7. Evaluate Each Agent on Unseen Test Data

Now we test each trained agent on data it has **never seen** (the last 20% of each stock's
history), and compare it against:
- **Buy & Hold**: buy on day 1, never sell
- **Random**: random buy/sell/hold decisions (sanity-check baseline)


In [ ]:
def evaluate_agent(agent, test_prices, initial_balance=10000.0):
    env = AdvancedTradingEnv(
        test_prices,
        initial_balance=initial_balance
    )
    state = env.reset()
    done = False

    curve = [initial_balance]
    actions = []
    info = {'net_worth': initial_balance}

    while not done:
        action = agent.select_action(state, training=False)
        state, reward, done, info = env.step(action)
        curve.append(info['net_worth'])
        actions.append(action)

    return curve, actions, float(info['net_worth'])


def evaluate_buy_hold(
    test_prices,
    initial_balance=10000.0,
    start_offset=20,
    transaction_cost=0.001
):
    if len(test_prices) <= start_offset:
        raise ValueError("Test set is too short for the selected start_offset.")

    start_price = float(test_prices[start_offset])

    # Apply the same initial transaction cost as the trading environment.
    shares = (initial_balance * (1.0 - transaction_cost)) / start_price

    prices = np.asarray(test_prices[start_offset:], dtype=np.float32)
    curve = [initial_balance] + [float(shares * p) for p in prices[1:]]

    return curve, float(curve[-1])


def evaluate_random(
    test_prices,
    initial_balance=10000.0,
    seed=42
):
    rng = np.random.default_rng(seed)
    env = AdvancedTradingEnv(
        test_prices,
        initial_balance=initial_balance
    )
    state = env.reset()
    done = False

    curve = [initial_balance]
    info = {'net_worth': initial_balance}

    while not done:
        action = int(rng.integers(0, 3))
        state, reward, done, info = env.step(action)
        curve.append(info['net_worth'])

    return curve, float(info['net_worth'])


results = []
eval_curves = {}

for stock in STOCKS:
    train_prices, test_prices = train_test_prices[stock]
    agent = trained_agents[stock]

    dqn_curve, dqn_actions, dqn_final = evaluate_agent(agent, test_prices)
    bh_curve, bh_final = evaluate_buy_hold(test_prices)
    rand_curve, rand_final = evaluate_random(test_prices)

    eval_curves[stock] = {
        'dqn': dqn_curve,
        'bh': bh_curve,
        'random': rand_curve,
        'actions': dqn_actions
    }

    results.append({
        'Stock': stock,
        'DQN Return %': round((dqn_final - 10000) / 10000 * 100, 2),
        'Buy&Hold Return %': round((bh_final - 10000) / 10000 * 100, 2),
        'Random Return %': round((rand_final - 10000) / 10000 * 100, 2),
        'DQN Final $': round(dqn_final, 2),
        'Buy&Hold Final $': round(bh_final, 2),
        'Random Final $': round(rand_final, 2),
    })

results_df = pd.DataFrame(results)
results_df


In [ ]:
# Plot comparison for each stock
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, stock in zip(axes.flatten(), STOCKS):
    c = eval_curves[stock]
    ax.plot(c['dqn'], label='DQN Agent', linewidth=2)
    ax.plot(c['bh'], label='Buy & Hold', linewidth=2)
    ax.plot(c['random'], label='Random', linestyle='--', alpha=0.6)
    ax.axhline(y=10000, linestyle=':')
    ax.set_title(f'{stock} - Test Period Performance')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Portfolio Value ($)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_comparison_all_stocks.png', dpi=150)
plt.show()


In [ ]:
# Buy/Sell decision markers for one example stock (AAPL)
example_stock = 'AAPL'
train_prices, test_prices = train_test_prices[example_stock]
actions = eval_curves[example_stock]['actions']

start_offset = 20
plot_prices = test_prices[start_offset:start_offset + len(actions)]

if len(plot_prices) != len(actions):
    raise RuntimeError(
        "Price/action lengths do not match. "
        f"prices={len(plot_prices)}, actions={len(actions)}"
    )

plt.figure(figsize=(13, 6))
plt.plot(
    plot_prices,
    alpha=0.6,
    linewidth=1,
    label=f'{example_stock} Price'
)

buy_steps = [i for i, a in enumerate(actions) if a == 1]
sell_steps = [i for i, a in enumerate(actions) if a == 2]

plt.scatter(
    buy_steps,
    [plot_prices[i] for i in buy_steps],
    marker='^',
    s=70,
    label='BUY',
    zorder=5
)
plt.scatter(
    sell_steps,
    [plot_prices[i] for i in sell_steps],
    marker='v',
    s=70,
    label='SELL',
    zorder=5
)

plt.title(f'{example_stock} - DQN Agent Buy/Sell Decisions (Test Period)')
plt.xlabel('Trading Day')
plt.ylabel('Price ($)')
plt.legend()
plt.tight_layout()
plt.savefig('agent_decisions_example.png', dpi=150)
plt.show()


## 8. Final Results Summary

The table below is the key result for your report — showing how the RL agent performed
on each stock compared to Buy & Hold and Random trading, on completely unseen test data.


In [ ]:
print("="*80)
print("FINAL RESULTS: DQN Agent vs Buy & Hold vs Random (Unseen Test Data)")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

avg_dqn = results_df['DQN Return %'].mean()
avg_bh = results_df['Buy&Hold Return %'].mean()
avg_rand = results_df['Random Return %'].mean()
print(f"\nAverage across all 4 stocks:")
print(f"  DQN Agent:   {avg_dqn:.2f}%")
print(f"  Buy & Hold:  {avg_bh:.2f}%")
print(f"  Random:      {avg_rand:.2f}%")

results_df.to_csv('final_results.csv', index=False)
print("\nSaved final_results.csv")


## 9. Conclusion

This project built a complete Reinforcement Learning pipeline for stock trading:

1. **Data**: Historical daily closing prices for 4 stocks (AAPL, MSFT, IBM, SBUX), 2007–2016
2. **Feature Engineering**: 7 technical indicators (returns, moving averages, volatility, momentum, RSI)
3. **Environment**: A custom Buy/Sell/Hold trading simulation with transaction costs
4. **Algorithm**: Double DQN implemented in PyTorch
5. **Evaluation**: Tested on unseen data across 4 stocks and compared against Buy & Hold and Random baselines

**Important:** A positive DQN result does not prove that a strategy will make money in live markets.
The test results should be interpreted as an experiment on historical data, and performance can
change substantially with different periods, transaction costs, random seeds, and market conditions.

### Possible Extensions

- Add trading volume as a feature (if available)
- Try PPO or A2C and compare against DQN
- Build a portfolio-allocation agent that trades all 4 stocks simultaneously
- Add Sharpe Ratio and Maximum Drawdown as risk-adjusted performance metrics
- Run multiple random seeds and report mean ± standard deviation
